### Data loading and Preprocessing

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('Result 2025-09-23 05-01-51.csv')
df

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,stories,fcc_asr_number,faa_study_number,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason
0,942614,Phoenix Tower International,NaN,Phoenix Tower International,2019-08-22 20:53:42.284115,2023-01-13 15:25:12.493704,33.94619,-118.28200,CA12409,160407,...,NaN,NaN,NaN,NaN,NaN,"FIGUEROA ST WL 75F S OF 99TH ST S/N , August F...",NaN,No,Unconfirmed,Focus Asset
1,942614,Phoenix Tower International,943927.0,Phoenix Tower International,2019-08-22 21:36:14.93064,2023-06-26 12:05:50.800659,33.94574,-118.28200,MAIN ST EL 185F N OF 110TH ST SF,161773,...,NaN,NaN,NaN,NaN,NaN,"MAIN ST EL 185F N OF 110TH ST SF , , CA 90003,...",NaN,No,Unconfirmed,Proximity Audit (48m)
2,942664,Phoenix Tower International,NaN,Phoenix Tower International,2019-08-22 20:55:21.34445,2023-01-13 11:49:35.586868,34.03929,-118.19900,CA12465,160463,...,NaN,NaN,NaN,NaN,NaN,"1ST ST SL 2F E OF DACOTAH ST E/W , Los Angeles...",NaN,No,Unconfirmed,Focus Asset
3,942664,Phoenix Tower International,942203.0,Phoenix Tower International,2019-08-22 20:40:05.047291,2023-01-13 13:04:52.922687,34.03910,-118.19900,CA11987,159985,...,NaN,NaN,NaN,NaN,NaN,"1ST ST SL 53F W OF FRESNO ST WF , Los Angeles,...",NaN,No,Unconfirmed,Proximity Audit (21m)
4,942761,Phoenix Tower International,NaN,Phoenix Tower International,2019-08-22 20:58:33.921034,2023-01-13 12:41:13.613007,33.98899,-118.31900,CA12570,160568,...,NaN,NaN,NaN,NaN,NaN,"SLAUSON AVE NL 42F E OF 3RD AVE EF , Los Angel...",NaN,No,Unconfirmed,Focus Asset
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64688,1161828,ASR,264885.0,Crown Castle,2015-04-20 20:26:56.595238,2017-07-28 23:55:23.576292,39.00210,-95.73923,SKYLINE PARK/BURNETT'S MOUND,877829,...,NaN,NaN,NaN,NaN,NaN,"3511 SW SKYLINE PARKWAY , TOPEKA, KS 66614, UN...",2007-01-12,No,Active,Proximity Audit (7m)
64689,1161829,ASR,NaN,ASR,2025-04-01 15:38:18.798044,2025-04-09 11:29:48.458842,32.81861,-86.61714,NaN,NaN,...,NaN,1330244.0,2024-ASO-23463-OE,NaN,NaN,"Off Logan Road , Clanton, AL 35045, UNITED STATES",2024-11-26,No,Active,Focus Asset
64690,1161829,ASR,1136163.0,ASR,2024-01-23 15:24:06.649527,2024-01-23 09:31:32.771277,32.81861,-86.61714,NaN,NaN,...,NaN,1327054.0,2023-ASO-27414-OE,NaN,NaN,"Off Logan Road , Clanton, AL 35046, UNITED STATES",2024-11-26,No,Active,Proximity Audit (0m)
64691,1162574,ASR,NaN,ASR,2025-07-24 21:02:34.966713,2025-07-28 09:49:51.734224,42.34739,-91.47869,NaN,NaN,...,NaN,1330947.0,2024-ACE-6738-OE,NaN,NaN,"3048 Hwy 13 IA-5254 , Ryan, IA 52330, UNITED S...",NaN,No,ASR-Granted,Focus Asset


In [3]:
df.dtypes

focus_asset_id           int64
focus_asset             object
associated_asset_id    float64
source                  object
created_at              object
updated_at              object
latitude               float64
longitude              float64
name                    object
operator_site_id        object
type                    object
description             object
operator_name           object
manager_name            object
fcc_owner_name          object
agl                    float64
amsl                   float64
ground_elevation       float64
haat                   float64
shelter                 object
power                   object
stories                float64
fcc_asr_number         float64
faa_study_number        object
cdbs_facility_id       float64
region                  object
address                 object
construction_date       object
stealth                 object
asset_status            object
audit_reason            object
dtype: object

In [4]:
df['focus_asset_id'] = df['focus_asset_id'].astype('Int64')
df['focus_asset'] = df['focus_asset'].astype('string')
df['associated_asset_id'] = df['associated_asset_id'].astype('Int64')
df['source'] = df['source'].astype('string')
df['name'] = df['name'].astype('string')
df['operator_site_id'] = df['operator_site_id'].astype('string')
df['type'] = df['type'].astype('string')
df['description'] = df['description'].astype('string')
df['operator_name'] = df['operator_name'].astype('string')
df['manager_name'] = df['manager_name'].astype('string')
df['fcc_owner_name'] = df['fcc_owner_name'].astype('string')
df['shelter'] = df['shelter'].astype('string')
df['power'] = df['power'].astype('string')
df['fcc_asr_number'] = df['fcc_asr_number'].astype('string')
df['faa_study_number'] = df['faa_study_number'].astype('string')
df['cdbs_facility_id'] = df['cdbs_facility_id'].astype('string')
df['region'] = df['region'].astype('string')
df['address'] = df['address'].astype('string')
df['stealth'] = df['stealth'].astype('string')
df['asset_status'] = df['asset_status'].astype('string')
df['audit_reason'] = df['audit_reason'].astype('string')

In [5]:
from geopy.distance import geodesic
import numpy as np

def calculate_distances_to_reference(df):
    df_copy = df.copy()
    df_copy['distance_to_reference'] = np.nan

    for group_name, group_df in df_copy.groupby('focus_asset_id'):
        reference_record = group_df[group_df['associated_asset_id'].isna()]

        if not reference_record.empty:
            ref_lat = reference_record['latitude'].iloc[0]
            ref_lon = reference_record['longitude'].iloc[0]
            reference_coords = (ref_lat, ref_lon)

            for index, row in group_df.iterrows():
                if pd.notna(row['associated_asset_id']):
                    record_coords = (row['latitude'], row['longitude'])
                    distance = geodesic(reference_coords, record_coords).meters
                    df_copy.loc[index, 'distance_to_reference'] = distance
    return df_copy

df_with_distances = calculate_distances_to_reference(df)
df_with_distances

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,fcc_asr_number,faa_study_number,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference
0,942614,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:53:42.284115,2023-01-13 15:25:12.493704,33.94619,-118.28200,CA12409,160407,...,<NA>,<NA>,<NA>,<NA>,"FIGUEROA ST WL 75F S OF 99TH ST S/N , August F...",NaN,No,Unconfirmed,Focus Asset,NaN
1,942614,Phoenix Tower International,943927,Phoenix Tower International,2019-08-22 21:36:14.93064,2023-06-26 12:05:50.800659,33.94574,-118.28200,MAIN ST EL 185F N OF 110TH ST SF,161773,...,<NA>,<NA>,<NA>,<NA>,"MAIN ST EL 185F N OF 110TH ST SF , , CA 90003,...",NaN,No,Unconfirmed,Proximity Audit (48m),49.914635
2,942664,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:55:21.34445,2023-01-13 11:49:35.586868,34.03929,-118.19900,CA12465,160463,...,<NA>,<NA>,<NA>,<NA>,"1ST ST SL 2F E OF DACOTAH ST E/W , Los Angeles...",NaN,No,Unconfirmed,Focus Asset,NaN
3,942664,Phoenix Tower International,942203,Phoenix Tower International,2019-08-22 20:40:05.047291,2023-01-13 13:04:52.922687,34.03910,-118.19900,CA11987,159985,...,<NA>,<NA>,<NA>,<NA>,"1ST ST SL 53F W OF FRESNO ST WF , Los Angeles,...",NaN,No,Unconfirmed,Proximity Audit (21m),21.075388
4,942761,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:58:33.921034,2023-01-13 12:41:13.613007,33.98899,-118.31900,CA12570,160568,...,<NA>,<NA>,<NA>,<NA>,"SLAUSON AVE NL 42F E OF 3RD AVE EF , Los Angel...",NaN,No,Unconfirmed,Focus Asset,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64688,1161828,ASR,264885,Crown Castle,2015-04-20 20:26:56.595238,2017-07-28 23:55:23.576292,39.00210,-95.73923,SKYLINE PARK/BURNETT'S MOUND,877829,...,<NA>,<NA>,<NA>,<NA>,"3511 SW SKYLINE PARKWAY , TOPEKA, KS 66614, UN...",2007-01-12,No,Active,Proximity Audit (7m),7.276909
64689,1161829,ASR,<NA>,ASR,2025-04-01 15:38:18.798044,2025-04-09 11:29:48.458842,32.81861,-86.61714,<NA>,<NA>,...,1330244.0,2024-ASO-23463-OE,<NA>,<NA>,"Off Logan Road , Clanton, AL 35045, UNITED STATES",2024-11-26,No,Active,Focus Asset,NaN
64690,1161829,ASR,1136163,ASR,2024-01-23 15:24:06.649527,2024-01-23 09:31:32.771277,32.81861,-86.61714,<NA>,<NA>,...,1327054.0,2023-ASO-27414-OE,<NA>,<NA>,"Off Logan Road , Clanton, AL 35046, UNITED STATES",2024-11-26,No,Active,Proximity Audit (0m),0.000000
64691,1162574,ASR,<NA>,ASR,2025-07-24 21:02:34.966713,2025-07-28 09:49:51.734224,42.34739,-91.47869,<NA>,<NA>,...,1330947.0,2024-ACE-6738-OE,<NA>,<NA>,"3048 Hwy 13 IA-5254 , Ryan, IA 52330, UNITED S...",NaN,No,ASR-Granted,Focus Asset,NaN


In [6]:
def calculate_agldiff_to_reference(df):
    df_copy = df_with_distances.copy()
    df_copy['agldiff_to_reference'] = np.nan

    for group_name, group_df in df_copy.groupby('focus_asset_id'):
        reference_record = group_df[group_df['associated_asset_id'].isna()]

        if not reference_record.empty:
            ref_agl = reference_record['agl'].iloc[0]

            for index, row in group_df.iterrows():
                if pd.notna(row['associated_asset_id']):
                    record_agl = row['agl']
                    distance = ref_agl - record_agl
                    df_copy.loc[index, 'agldiff_to_reference'] = distance
    return df_copy

df_with_differences = calculate_agldiff_to_reference(df_with_distances)
df_with_differences

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,faa_study_number,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference
0,942614,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:53:42.284115,2023-01-13 15:25:12.493704,33.94619,-118.28200,CA12409,160407,...,<NA>,<NA>,<NA>,"FIGUEROA ST WL 75F S OF 99TH ST S/N , August F...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN
1,942614,Phoenix Tower International,943927,Phoenix Tower International,2019-08-22 21:36:14.93064,2023-06-26 12:05:50.800659,33.94574,-118.28200,MAIN ST EL 185F N OF 110TH ST SF,161773,...,<NA>,<NA>,<NA>,"MAIN ST EL 185F N OF 110TH ST SF , , CA 90003,...",NaN,No,Unconfirmed,Proximity Audit (48m),49.914635,0.0
2,942664,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:55:21.34445,2023-01-13 11:49:35.586868,34.03929,-118.19900,CA12465,160463,...,<NA>,<NA>,<NA>,"1ST ST SL 2F E OF DACOTAH ST E/W , Los Angeles...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN
3,942664,Phoenix Tower International,942203,Phoenix Tower International,2019-08-22 20:40:05.047291,2023-01-13 13:04:52.922687,34.03910,-118.19900,CA11987,159985,...,<NA>,<NA>,<NA>,"1ST ST SL 53F W OF FRESNO ST WF , Los Angeles,...",NaN,No,Unconfirmed,Proximity Audit (21m),21.075388,0.0
4,942761,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:58:33.921034,2023-01-13 12:41:13.613007,33.98899,-118.31900,CA12570,160568,...,<NA>,<NA>,<NA>,"SLAUSON AVE NL 42F E OF 3RD AVE EF , Los Angel...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64688,1161828,ASR,264885,Crown Castle,2015-04-20 20:26:56.595238,2017-07-28 23:55:23.576292,39.00210,-95.73923,SKYLINE PARK/BURNETT'S MOUND,877829,...,<NA>,<NA>,<NA>,"3511 SW SKYLINE PARKWAY , TOPEKA, KS 66614, UN...",2007-01-12,No,Active,Proximity Audit (7m),7.276909,4.5
64689,1161829,ASR,<NA>,ASR,2025-04-01 15:38:18.798044,2025-04-09 11:29:48.458842,32.81861,-86.61714,<NA>,<NA>,...,2024-ASO-23463-OE,<NA>,<NA>,"Off Logan Road , Clanton, AL 35045, UNITED STATES",2024-11-26,No,Active,Focus Asset,NaN,NaN
64690,1161829,ASR,1136163,ASR,2024-01-23 15:24:06.649527,2024-01-23 09:31:32.771277,32.81861,-86.61714,<NA>,<NA>,...,2023-ASO-27414-OE,<NA>,<NA>,"Off Logan Road , Clanton, AL 35046, UNITED STATES",2024-11-26,No,Active,Proximity Audit (0m),0.000000,-20.4
64691,1162574,ASR,<NA>,ASR,2025-07-24 21:02:34.966713,2025-07-28 09:49:51.734224,42.34739,-91.47869,<NA>,<NA>,...,2024-ACE-6738-OE,<NA>,<NA>,"3048 Hwy 13 IA-5254 , Ryan, IA 52330, UNITED S...",NaN,No,ASR-Granted,Focus Asset,NaN,NaN


In [7]:
prox_audits_table = df_with_differences
prox_audits_table

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,faa_study_number,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference
0,942614,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:53:42.284115,2023-01-13 15:25:12.493704,33.94619,-118.28200,CA12409,160407,...,<NA>,<NA>,<NA>,"FIGUEROA ST WL 75F S OF 99TH ST S/N , August F...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN
1,942614,Phoenix Tower International,943927,Phoenix Tower International,2019-08-22 21:36:14.93064,2023-06-26 12:05:50.800659,33.94574,-118.28200,MAIN ST EL 185F N OF 110TH ST SF,161773,...,<NA>,<NA>,<NA>,"MAIN ST EL 185F N OF 110TH ST SF , , CA 90003,...",NaN,No,Unconfirmed,Proximity Audit (48m),49.914635,0.0
2,942664,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:55:21.34445,2023-01-13 11:49:35.586868,34.03929,-118.19900,CA12465,160463,...,<NA>,<NA>,<NA>,"1ST ST SL 2F E OF DACOTAH ST E/W , Los Angeles...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN
3,942664,Phoenix Tower International,942203,Phoenix Tower International,2019-08-22 20:40:05.047291,2023-01-13 13:04:52.922687,34.03910,-118.19900,CA11987,159985,...,<NA>,<NA>,<NA>,"1ST ST SL 53F W OF FRESNO ST WF , Los Angeles,...",NaN,No,Unconfirmed,Proximity Audit (21m),21.075388,0.0
4,942761,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:58:33.921034,2023-01-13 12:41:13.613007,33.98899,-118.31900,CA12570,160568,...,<NA>,<NA>,<NA>,"SLAUSON AVE NL 42F E OF 3RD AVE EF , Los Angel...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64688,1161828,ASR,264885,Crown Castle,2015-04-20 20:26:56.595238,2017-07-28 23:55:23.576292,39.00210,-95.73923,SKYLINE PARK/BURNETT'S MOUND,877829,...,<NA>,<NA>,<NA>,"3511 SW SKYLINE PARKWAY , TOPEKA, KS 66614, UN...",2007-01-12,No,Active,Proximity Audit (7m),7.276909,4.5
64689,1161829,ASR,<NA>,ASR,2025-04-01 15:38:18.798044,2025-04-09 11:29:48.458842,32.81861,-86.61714,<NA>,<NA>,...,2024-ASO-23463-OE,<NA>,<NA>,"Off Logan Road , Clanton, AL 35045, UNITED STATES",2024-11-26,No,Active,Focus Asset,NaN,NaN
64690,1161829,ASR,1136163,ASR,2024-01-23 15:24:06.649527,2024-01-23 09:31:32.771277,32.81861,-86.61714,<NA>,<NA>,...,2023-ASO-27414-OE,<NA>,<NA>,"Off Logan Road , Clanton, AL 35046, UNITED STATES",2024-11-26,No,Active,Proximity Audit (0m),0.000000,-20.4
64691,1162574,ASR,<NA>,ASR,2025-07-24 21:02:34.966713,2025-07-28 09:49:51.734224,42.34739,-91.47869,<NA>,<NA>,...,2024-ACE-6738-OE,<NA>,<NA>,"3048 Hwy 13 IA-5254 , Ryan, IA 52330, UNITED S...",NaN,No,ASR-Granted,Focus Asset,NaN,NaN


In [8]:
column_names_list = list(prox_audits_table.columns)

for i in column_names_list:
    print(i)

focus_asset_id
focus_asset
associated_asset_id
source
created_at
updated_at
latitude
longitude
name
operator_site_id
type
description
operator_name
manager_name
fcc_owner_name
agl
amsl
ground_elevation
haat
shelter
power
stories
fcc_asr_number
faa_study_number
cdbs_facility_id
region
address
construction_date
stealth
asset_status
audit_reason
distance_to_reference
agldiff_to_reference


---

In [9]:
import pandas as pd
from typing import Tuple, List
from datetime import datetime
import numpy as np
import requests
import json
import re
import time 
import warnings

# Suppress all FutureWarning messages
warnings.simplefilter(action='ignore', category=FutureWarning)

# Imports for API robustness and progress bars
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm # For progress bars

# Global Caches for API Calls
asn_cache = {}
asr_cache = {}

# API Configuration
api_endpoint_asn = "https://oeaaa.faa.gov/oeaaa/tools-api/namedOperation.do"
api_endpoint_asr = "https://oeaaa.faa.gov/oeaaa/oe3a/external/portal-api/caseFiling/dynamicCaseDataByAsn.do"

api_payload_template_asn = {
    "areaType": "id", "timeSpan": "120", "formLat": "", "formLon": "",
    "radiusNM": 25, "structureType": "ANY", "allStatusSelected": True,
    "criteria": {}, "fcc": "", "opName": "GET_CASE_BY_FCC",
    "placement": "OFF_AIRPORT", "status": {}, "structureTypes": ["ANY"]
}
api_payload_template_asr = {
    "areaType": "id", "timeSpan": "120", "formLat": "", "formLon": "",
    "radiusNM": 25, "structureType": "ANY", "allStatusSelected": True,
    "criteria": {}, "asnRegion": "", "asnYear": 0, "asnSequence": "",
    "asnCaseType": "", "placement": "OFF_AIRPORT", "status": {},
    "structureTypes": ["ANY"]
}
headers = {
    "Content-Type": "application/json",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36"
}

asn_key_in_response = "ASN"
date_key_in_response_asn = "SUBMITTED_DATE"
fcc_asr_key_in_response = "fccAsr"

# Robust Request Session (Retries, Timeout)
def create_robust_session():
    session = requests.Session()
    session.headers.update(headers)
    retry_strategy = Retry(
        total=5, 
        backoff_factor=1, 
        status_forcelist=[429, 500, 502, 503, 504],
    )
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    session.timeout = 60 
    return session

api_session = create_robust_session()

def parse_asn(asn_string):
    match = re.match(r"(\d{4})-([A-Z]{3})-([\d\w]+)-([A-Z]{2})", str(asn_string))
    if match:
        year, region, sequence, casetype = match.groups()
        return {
            "asnYear": int(year),
            "asnRegion": region,
            "asnSequence": sequence,
            "asnCaseType": casetype
        }
    return None

# API function with Caching
def get_asn_via_api(asr_number):
    asr_number_str = str(asr_number)
    if asr_number_str in asn_cache:
        return asn_cache[asr_number_str]

    payload = api_payload_template_asn.copy()
    payload["fcc"] = asr_number_str
    
    try:
        response = api_session.post(api_endpoint_asn, data=json.dumps(payload))
        if response.status_code == 200:
            data = response.json()
            if data and isinstance(data, list) and len(data) > 0:
                def parse_date(record):
                    date_str = record.get(date_key_in_response_asn)
                    if date_str and date_str != "N/A":
                        try:
                            return datetime.strptime(date_str, "%m/%d/%Y")
                        except ValueError:
                            return datetime.min
                    return datetime.min
                latest_record = max(data, key=parse_date)
                if asn_key_in_response in latest_record:
                    result = latest_record[asn_key_in_response]
                    asn_cache[asr_number_str] = result 
                    return result
    except Exception:
        pass
    asn_cache[asr_number_str] = None 
    return None

# API function with Caching
def get_asr_via_api(asn_number):
    asn_number_str = str(asn_number)
    if asn_number_str in asr_cache:
        return asr_cache[asn_number_str]

    asn_parts = parse_asn(asn_number_str)
    if not asn_parts:
        asr_cache[asn_number_str] = None
        return None

    payload = api_payload_template_asr.copy()
    payload.update(asn_parts)

    try:
        response = api_session.post(api_endpoint_asr, data=json.dumps(payload))
        if response.status_code == 200:
            data = response.json()
            result = None
            if isinstance(data, dict) and fcc_asr_key_in_response in data:
                result = str(data[fcc_asr_key_in_response])
            elif isinstance(data, list) and len(data) > 0 and fcc_asr_key_in_response in data[0]:
                result = str(data[0][fcc_asr_key_in_response])
            if result:
                asr_cache[asn_number_str] = result 
                return result
    except Exception:
        pass
    asr_cache[asn_number_str] = None
    return None

# Helper functions for pre-caching
def _cache_asn(asr):
    """Worker function to cache ASR->ASN. Result is stored in global cache."""
    get_asn_via_api(asr)

def _cache_asr(asn):
    """Worker function to cache ASN->ASR. Result is stored in global cache."""
    get_asr_via_api(asn)

# Pre-caching function
def pre_populate_api_caches(prox_audits_table: pd.DataFrame, max_workers: int = 3):
    """
    Scans the entire table for unique ASRs and ASNs and scrapes them
    to populate the global caches, using a safe number of workers.
    """
    start_time = time.time()
    print(f"Starting API pre-caching with max_workers={max_workers}...")
    
    # Get unique, non-null ASRs
    unique_asrs = prox_audits_table['fcc_asr_number'].dropna().unique()
    
    # Get unique, non-null, and validly-formatted ASNs
    unique_asns_raw = prox_audits_table['faa_study_number'].dropna().unique()
    unique_asns = [asn for asn in unique_asns_raw if parse_asn(asn)]
    
    # Cache ASR -> ASN
    print(f"Caching {len(unique_asrs)} unique ASR numbers...")
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Use tqdm for a progress bar
        list(tqdm(executor.map(_cache_asn, unique_asrs), total=len(unique_asrs), desc="Caching ASRs"))
            
    # Cache ASN -> ASR
    print(f"Caching {len(unique_asns)} unique ASN numbers...")
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        list(tqdm(executor.map(_cache_asr, unique_asns), total=len(unique_asns), desc="Caching ASNs"))

    end_time = time.time()
    duration_minutes = (end_time - start_time) / 60
    print(f"--- API Pre-caching completed in: {duration_minutes:.2f} minutes ---")
    print(f"Total items cached: {len(asn_cache) + len(asr_cache)}")


# Scraping Logic Functions
def determine_faa_study_number(ref_fcc, ref_faa, assoc_faa):
    new_faa = get_asn_via_api(ref_fcc)
    if pd.notnull(new_faa) and new_faa != 'N/A':
        return str(new_faa)
    faa_ids_to_check = set()
    if pd.notnull(ref_faa) and parse_asn(str(ref_faa)): faa_ids_to_check.add(str(ref_faa))
    if pd.notnull(assoc_faa) and parse_asn(str(assoc_faa)): faa_ids_to_check.add(str(assoc_faa))
    for faa_id in faa_ids_to_check:
        found_asr = get_asr_via_api(faa_id)
        if pd.notnull(found_asr) and str(found_asr) == str(ref_fcc):
            return faa_id
    return None
    
def get_case_1_final_faa(ref_fcc, ref_faa, assoc_faa):
    new_faa = get_asn_via_api(ref_fcc)
    if pd.notnull(new_faa) and new_faa != 'N/A':
        return str(new_faa)
    else:
        return ref_faa

# ASR Cleaning Logic
def clean_asr_in_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    if 'fcc_asr_number' in df.columns:
        # Preserve original strings if they can't be numeric
        original_strings = df['fcc_asr_number'].copy()
        numeric_fcc = pd.to_numeric(df['fcc_asr_number'], errors='coerce')
        
        def to_clean_str(x, original_val):
            if pd.notnull(x) and x == int(x):
                return str(int(x))
            # If it's not a clean int, return the original string value
            if pd.notnull(original_val):
                return str(original_val)
            return np.nan
        
        df['fcc_asr_number'] = [to_clean_str(num, orig) for num, orig in zip(numeric_fcc, original_strings)]
    return df

# `split_case_1_audits` (with ASR Cleaning)
def split_case_1_audits(prox_audits_table: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    working_table = prox_audits_table.copy()
    working_table = clean_asr_in_dataframe(working_table)
    
    candidates_indices = set()
    grouped = working_table.groupby('focus_asset_id')
    for focus_asset_id, group in grouped:
        reference_record_df = group[group['associated_asset_id'].isnull()]
        associated_records_df = group[group['associated_asset_id'].notnull()]
        ref_index = reference_record_df.index[0] if len(reference_record_df) == 1 else None

        if len(reference_record_df) == 1 and ref_index is not None:
            reference_record = reference_record_df.iloc[0]
            ref_fcc = reference_record['fcc_asr_number']
            ref_faa = reference_record['faa_study_number']
            if pd.notnull(ref_fcc) and pd.notnull(ref_faa):
                matching_mask = (associated_records_df['fcc_asr_number'] == ref_fcc) & \
                                (associated_records_df['faa_study_number'] == ref_faa)
                matching_associated_records = associated_records_df[matching_mask]
                if not matching_associated_records.empty:
                    candidates_indices.add(ref_index)
                    candidates_indices.update(matching_associated_records.index)

    case1_auto_merge_candidates = working_table.loc[list(candidates_indices)].copy()
    all_original_indices = set(working_table.index)
    final_post_merge_indices = all_original_indices.difference(candidates_indices)
    case1_prox_audits_post_auto_merge_table = working_table.loc[list(final_post_merge_indices)].copy()

    cols = working_table.columns
    if case1_auto_merge_candidates.empty:
         case1_auto_merge_candidates = pd.DataFrame(columns=cols)
    if case1_prox_audits_post_auto_merge_table.empty:
         case1_prox_audits_post_auto_merge_table = pd.DataFrame(columns=cols)
    return case1_auto_merge_candidates, case1_prox_audits_post_auto_merge_table

# `merge_records` (with ASR Cleaning)
def merge_records(reference_record, associated_record, merge_timestamp, faa_study_number=None):
    merged_record = reference_record.copy()
    ref_op = reference_record['operator_name']
    assoc_op = associated_record['operator_name']
    
    if pd.notnull(ref_op) and ref_op in ["Unassigned", "Unkown"] and \
       pd.notnull(assoc_op) and assoc_op not in ["Unassigned", "Unkown"]:
        merged_record['operator_name'] = assoc_op
    
    merged_record['associated_asset_id'] = reference_record['associated_asset_id']
    merged_record['source'] = f"Auto-Merged {merge_timestamp.strftime('%m/%Y')}"
    merged_record['created_at'] = reference_record['created_at']
    merged_record['updated_at'] = merge_timestamp.strftime('%Y-%m-%d %H:%M:%S')

    fields_to_check = [
        "latitude", "longitude", "name", "operator_site_id", "type", "description", 
        "manager_name", "fcc_owner_name", "agl", "amsl", "ground_elevation", "haat", 
        "shelter", "power", "stories", "fcc_asr_number", "cdbs_facility_id", "region", 
        "address", "construction_date", "stealth", "asset_status"
    ]
    for field in fields_to_check:
        if pd.isnull(merged_record[field]) and pd.notnull(associated_record[field]):
            merged_record[field] = associated_record[field]
    if faa_study_number is not None:
        merged_record['faa_study_number'] = faa_study_number
        
    if pd.notnull(merged_record['fcc_asr_number']):
        try:
            numeric_val = pd.to_numeric(merged_record['fcc_asr_number'], errors='coerce')
            if pd.notnull(numeric_val) and numeric_val == int(numeric_val):
                merged_record['fcc_asr_number'] = str(int(numeric_val))
            else:
                merged_record['fcc_asr_number'] = str(merged_record['fcc_asr_number'])
        except Exception:
            pass 
            
    if pd.notnull(merged_record['construction_date']):
        try:
            pd.to_datetime(merged_record['construction_date'])
            merged_record['asset_status'] = "Active"
        except:
            pass
    return merged_record


# `apply_case_1_full_processing` (List-Append, Set-Filter, NO ThreadPool)
def apply_case_1_full_processing(
    candidates_table: pd.DataFrame,
    initial_prox_audits_table: pd.DataFrame
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:

    start_time = time.time()
    print("Starting Case 1 processing...")
    
    merging_timestamp = datetime.now()
    cols = candidates_table.columns
    
    case1_auto_merge_further_filter_list = []
    case1_raw_post_auto_merge_list = []
    case1_post_auto_merge_list = []
    new_prox_audit_records_list = [] 
    failed_prox_audit_records_list = [] 
    
    case1_prox_audits_post_auto_merge_table = initial_prox_audits_table.copy()
    
    processed_focus_ids = set()
    processed_assoc_ids = set()
    
    if not pd.api.types.is_datetime64_any_dtype(candidates_table['created_at']):
        candidates_table['created_at'] = pd.to_datetime(candidates_table['created_at'], errors='coerce')
    candidates_table['created_at_month'] = candidates_table['created_at'].dt.to_period('M')
    
    grouped = candidates_table.groupby('focus_asset_id')
    
    for focus_asset_id, group in tqdm(grouped, desc="Processing Case 1"):
        reference_record_df = group[group['associated_asset_id'].isnull()]
        associated_records_df = group[group['associated_asset_id'].notnull()].copy()
        
        if len(reference_record_df) != 1:
            continue
            
        reference_record = reference_record_df.iloc[0]
        ref_fcc = reference_record['fcc_asr_number']
        ref_faa = reference_record['faa_study_number']
        ref_source = reference_record['source']
        ref_year_month = reference_record['created_at_month']
        
        remaining_associated_records_df = associated_records_df.copy()
        indices_to_remove = set()
        
        if processed_focus_ids:
            focus_id_match_mask = remaining_associated_records_df['associated_asset_id'].isin(processed_focus_ids)
            indices_to_remove.update(remaining_associated_records_df[focus_id_match_mask].index)
        if processed_assoc_ids:
            assoc_id_match_mask = remaining_associated_records_df['associated_asset_id'].isin(processed_assoc_ids)
            indices_to_remove.update(remaining_associated_records_df[assoc_id_match_mask].index)

        remaining_associated_records_df['created_at_month'] = remaining_associated_records_df['created_at'].dt.to_period('M')
        date_source_match_mask = (remaining_associated_records_df['source'] == ref_source) & \
                                 (remaining_associated_records_df['created_at_month'] == ref_year_month)
        indices_to_remove.update(remaining_associated_records_df[date_source_match_mask].index)
        
        removed_associated_records_df = remaining_associated_records_df.loc[list(indices_to_remove)].copy()
        remaining_associated_records_df = remaining_associated_records_df.loc[
            ~remaining_associated_records_df.index.isin(indices_to_remove)
        ]
        
        temp_cols_to_drop = ['created_at_month']
        for df in [removed_associated_records_df, remaining_associated_records_df]:
            if 'created_at_month' in df.columns: df.drop(columns=['created_at_month'], inplace=True)
                
        ref_record_cleaned = reference_record_df.copy()
        if 'created_at_month' in ref_record_cleaned.columns: ref_record_cleaned.drop(columns=['created_at_month'], inplace=True)
        
        if not removed_associated_records_df.empty:
            failed_prox_audit_records_list.append(removed_associated_records_df)
            
        if len(remaining_associated_records_df) > 0:
            current_group_to_merge = pd.concat([ref_record_cleaned, remaining_associated_records_df], ignore_index=False)
            case1_auto_merge_further_filter_list.append(current_group_to_merge)

            # SIMPLE LOOP (No ThreadPool)
            for index, assoc_record in remaining_associated_records_df.iterrows():
                # This call is now an instant cache hit
                final_faa = get_case_1_final_faa(
                    ref_fcc, 
                    ref_faa,
                    assoc_record['faa_study_number']
                )

                merged_record_series = merge_records(reference_record, assoc_record, merging_timestamp, faa_study_number=final_faa)
                merged_record_df = pd.DataFrame([merged_record_series], columns=cols)
                
                new_prox_audit_records_list.append(merged_record_df)
                case1_post_auto_merge_list.append(merged_record_df)
                case1_raw_post_auto_merge_list.append(merged_record_df)

                assoc_record_df = pd.DataFrame([assoc_record], columns=cols)
                case1_raw_post_auto_merge_list.append(assoc_record_df)
                
                ref_record_df_cleaned_raw = ref_record_cleaned.copy()
                case1_raw_post_auto_merge_list.append(ref_record_df_cleaned_raw)
                
                if pd.notnull(merged_record_series['focus_asset_id']):
                    processed_focus_ids.add(merged_record_series['focus_asset_id'])
                if pd.notnull(assoc_record['associated_asset_id']):
                    processed_assoc_ids.add(assoc_record['associated_asset_id'])
        else:
            failed_prox_audit_records_list.append(ref_record_cleaned)
            
    # Concatenate lists ONCE at the end
    if new_prox_audit_records_list:
        new_records_df = pd.concat(new_prox_audit_records_list, ignore_index=True)
        case1_prox_audits_post_auto_merge_table = pd.concat(
            [case1_prox_audits_post_auto_merge_table, new_records_df], ignore_index=True
        )
    if failed_prox_audit_records_list:
        failed_records_df = pd.concat(failed_prox_audit_records_list, ignore_index=True)
        case1_prox_audits_post_auto_merge_table = pd.concat(
            [case1_prox_audits_post_auto_merge_table, failed_records_df], ignore_index=True
        )
    if case1_auto_merge_further_filter_list:
        case1_auto_merge_further_filter = pd.concat(case1_auto_merge_further_filter_list, ignore_index=False)
    else:
        case1_auto_merge_further_filter = pd.DataFrame(columns=cols)
    if case1_post_auto_merge_list:
        case1_post_auto_merge_table = pd.concat(case1_post_auto_merge_list, ignore_index=True)
    else:
        case1_post_auto_merge_table = pd.DataFrame(columns=cols)
    if case1_raw_post_auto_merge_list:
        case1_raw_post_auto_merge_table = pd.concat(case1_raw_post_auto_merge_list, ignore_index=True)
    else:
        case1_raw_post_auto_merge_table = pd.DataFrame(columns=cols)

    end_time = time.time()
    duration_minutes = (end_time - start_time) / 60
    print(f"--- Case 1 Processing completed in: {duration_minutes:.2f} minutes ---")
    
    return (
        case1_auto_merge_further_filter, 
        case1_prox_audits_post_auto_merge_table.drop_duplicates(ignore_index=True),
        case1_post_auto_merge_table.drop_duplicates(ignore_index=True), 
        case1_raw_post_auto_merge_table.drop_duplicates(ignore_index=True)
    )

# `apply_case_1_maintenance_logic` (No changes needed)
def apply_case_1_maintenance_logic(
    prox_audits_table: pd.DataFrame,
    post_auto_merge_table: pd.DataFrame, 
    post_merge_table: pd.DataFrame, 
    raw_post_merge_table: pd.DataFrame
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if post_auto_merge_table.empty:
        empty_df = pd.DataFrame(columns=prox_audits_table.columns)
        return empty_df, empty_df
    working_post_merge = post_auto_merge_table.reset_index(drop=True)
    final_asset_table_list = []
    associated_records_mask = working_post_merge['associated_asset_id'].notnull()
    if not post_merge_table.empty:
        post_merge_lookup = post_merge_table.set_index('focus_asset_id')
        all_cols = working_post_merge.columns.tolist()
        cols_to_exclude = ["audit_reason", "distance_to_reference", "agldiff_to_reference", "associated_asset_id", "index"]
        cols_to_update = [col for col in all_cols if col not in cols_to_exclude]
        for idx, assoc_record in working_post_merge[associated_records_mask].iterrows():
            assoc_asset_id = assoc_record['associated_asset_id']
            if assoc_asset_id in post_merge_lookup.index:
                matching_merged_record = post_merge_lookup.loc[assoc_asset_id]
                if isinstance(matching_merged_record, pd.DataFrame):
                    matching_merged_record = matching_merged_record.iloc[0]
                for col in cols_to_update:
                    if col in matching_merged_record.index:
                        if col in working_post_merge.columns:
                            working_post_merge.loc[idx, col] = matching_merged_record[col]
    if not raw_post_merge_table.empty:
        raw_assoc_ids = set(raw_post_merge_table['associated_asset_id'].dropna())
        removal_mask = (working_post_merge['associated_asset_id'].notnull()) & \
                       (working_post_merge['associated_asset_id'].isin(raw_assoc_ids))
        working_post_merge = working_post_merge[~removal_mask]
    valid_groups = working_post_merge['focus_asset_id'].dropna()
    if not valid_groups.empty:
        group_sizes = working_post_merge.groupby('focus_asset_id').size()
        single_record_groups = group_sizes[group_sizes == 1].index
        final_asset_table_df = working_post_merge[
            working_post_merge['focus_asset_id'].isin(single_record_groups)
        ].copy()
        final_asset_table_list.append(final_asset_table_df)
        working_post_merge = working_post_merge[
            ~working_post_merge['focus_asset_id'].isin(single_record_groups)
        ]
    sorted_post_merge_table = working_post_merge.sort_values(
        by=['focus_asset_id', 'associated_asset_id'], 
        ascending=[True, True],
        na_position='first'
    ).reset_index(drop=True)
    final_asset_table = pd.concat(final_asset_table_list, ignore_index=True)
    if final_asset_table.empty:
        final_asset_table = pd.DataFrame(columns=prox_audits_table.columns) 
    return final_asset_table, sorted_post_merge_table


# `split_case_2_audits` (with ASR Cleaning)
def split_case_2_audits(case1_sorted_post_merge_table: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    working_table = case1_sorted_post_merge_table.copy()
    working_table = clean_asr_in_dataframe(working_table)
    
    candidates_indices = set()
    grouped = working_table.groupby('focus_asset_id')
    for focus_asset_id, group in grouped:
        reference_record_df = group[group['associated_asset_id'].isnull()]
        associated_records_df = group[group['associated_asset_id'].notnull()]
        ref_index = reference_record_df.index[0] if len(reference_record_df) == 1 else None
        if len(reference_record_df) == 1 and ref_index is not None:
            reference_record = reference_record_df.iloc[0]
            ref_fcc = reference_record['fcc_asr_number']
            ref_faa = reference_record['faa_study_number']
            if pd.notnull(ref_fcc) and pd.notnull(ref_faa):
                matching_mask = (associated_records_df['fcc_asr_number'] == ref_fcc) & \
                                (associated_records_df['faa_study_number'] != ref_faa) & \
                                (associated_records_df['fcc_asr_number'].notnull()) & \
                                (associated_records_df['faa_study_number'].notnull())
                matching_associated_records = associated_records_df[matching_mask]
                if not matching_associated_records.empty:
                    candidates_indices.add(ref_index)
                    candidates_indices.update(matching_associated_records.index)
    case2_auto_merge_candidates = working_table.loc[list(candidates_indices)].copy()
    all_original_indices = set(working_table.index) 
    final_post_merge_indices = all_original_indices.difference(candidates_indices)
    case2_prox_audits_post_auto_merge_table = working_table.loc[list(final_post_merge_indices)].copy()
    cols = working_table.columns
    if case2_auto_merge_candidates.empty:
         case2_auto_merge_candidates = pd.DataFrame(columns=cols)
    if case2_prox_audits_post_auto_merge_table.empty:
         case2_prox_audits_post_auto_merge_table = pd.DataFrame(columns=cols)
    return case2_auto_merge_candidates, case2_prox_audits_post_auto_merge_table


# `apply_case_2_full_processing` (List-Append, Set-Filter, NO ThreadPool)
def apply_case_2_full_processing(
    candidates_table: pd.DataFrame,
    initial_prox_audits_table: pd.DataFrame
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:

    start_time = time.time()
    print("Starting Case 2 processing...")
    
    merging_timestamp = datetime.now()
    cols = candidates_table.columns
    merge_source_check = f"Auto-Merged {merging_timestamp.strftime('%m/%Y')}"
    
    case2_auto_merge_further_filter_list = []
    case2_raw_post_auto_merge_list = []
    case2_post_auto_merge_list = []
    new_prox_audit_records_list = []
    failed_prox_audit_records_list = []
    
    case2_prox_audits_post_auto_merge_table = initial_prox_audits_table.copy()
    
    processed_focus_ids = set()
    processed_assoc_ids = set()
    
    if not pd.api.types.is_datetime64_any_dtype(candidates_table['created_at']):
        candidates_table['created_at'] = pd.to_datetime(candidates_table['created_at'], errors='coerce')
    candidates_table['created_at_month'] = candidates_table['created_at'].dt.to_period('M')
    
    grouped = candidates_table.groupby('focus_asset_id')
    
    for focus_asset_id, group in tqdm(grouped, desc="Processing Case 2"):
        reference_record_df = group[group['associated_asset_id'].isnull()]
        associated_records_df = group[group['associated_asset_id'].notnull()].copy()
        
        if len(reference_record_df) != 1:
            continue
            
        reference_record = reference_record_df.iloc[0]
        ref_fcc = reference_record['fcc_asr_number']
        ref_faa = reference_record['faa_study_number']
        ref_source = reference_record['source']
        ref_year_month = reference_record['created_at_month']
        
        remaining_associated_records_df = associated_records_df.copy()
        indices_to_remove = set()
        
        if processed_focus_ids:
            focus_id_match_mask = remaining_associated_records_df['associated_asset_id'].isin(processed_focus_ids)
            indices_to_remove.update(remaining_associated_records_df[focus_id_match_mask].index)
        if processed_assoc_ids:
            assoc_id_match_mask = remaining_associated_records_df['associated_asset_id'].isin(processed_assoc_ids)
            indices_to_remove.update(remaining_associated_records_df[assoc_id_match_mask].index)
        
        source_match_mask = (remaining_associated_records_df['source'] == merge_source_check)
        indices_to_remove.update(remaining_associated_records_df[source_match_mask].index)
        
        ref_source_match = reference_record['source'] == merge_source_check
        
        ref_record_cleaned = reference_record_df.copy()
        if 'created_at_month' in ref_record_cleaned.columns: ref_record_cleaned.drop(columns=['created_at_month'], inplace=True)
        
        if ref_source_match:
            group_associated_cleaned = associated_records_df.copy()
            if 'created_at_month' in group_associated_cleaned.columns: group_associated_cleaned.drop(columns=['created_at_month'], inplace=True)
            failed_prox_audit_records_list.append(ref_record_cleaned)
            if not group_associated_cleaned.empty:
                failed_prox_audit_records_list.append(group_associated_cleaned)
            continue 

        remaining_associated_records_df['created_at_month'] = remaining_associated_records_df['created_at'].dt.to_period('M')
        date_source_match_mask = (remaining_associated_records_df['source'] == ref_source) & \
                                 (remaining_associated_records_df['created_at_month'] == ref_year_month)
        indices_to_remove.update(remaining_associated_records_df[date_source_match_mask].index)

        removed_associated_records_df = remaining_associated_records_df.loc[list(indices_to_remove)].copy()
        remaining_associated_records_df = remaining_associated_records_df.loc[
            ~remaining_associated_records_df.index.isin(indices_to_remove)
        ]

        if 'created_at_month' in removed_associated_records_df.columns: removed_associated_records_df.drop(columns=['created_at_month'], inplace=True)
        if 'created_at_month' in remaining_associated_records_df.columns: remaining_associated_records_df.drop(columns=['created_at_month'], inplace=True)

        if not removed_associated_records_df.empty:
            failed_prox_audit_records_list.append(removed_associated_records_df)
        
        if len(remaining_associated_records_df) > 0:
            current_group_to_merge = pd.concat([ref_record_cleaned, remaining_associated_records_df], ignore_index=False)
            case2_auto_merge_further_filter_list.append(current_group_to_merge)

            # SIMPLE LOOP (No ThreadPool)
            for index, assoc_record in remaining_associated_records_df.iterrows():
                # This call is now an instant cache hit
                final_faa = determine_faa_study_number(
                    ref_fcc, 
                    ref_faa,
                    assoc_record['faa_study_number']
                )

                if pd.notnull(final_faa):
                    merged_record_series = merge_records(reference_record, assoc_record, merging_timestamp, faa_study_number=final_faa)
                    merged_record_df = pd.DataFrame([merged_record_series], columns=cols)
                    
                    new_prox_audit_records_list.append(merged_record_df)
                    case2_post_auto_merge_list.append(merged_record_df)
                    case2_raw_post_auto_merge_list.append(merged_record_df)

                    assoc_record_df = pd.DataFrame([assoc_record], columns=cols)
                    case2_raw_post_auto_merge_list.append(assoc_record_df)
                    
                    ref_record_df_cleaned_raw = ref_record_cleaned.copy()
                    case2_raw_post_auto_merge_list.append(ref_record_df_cleaned_raw)
                    
                    if pd.notnull(merged_record_series['focus_asset_id']):
                        processed_focus_ids.add(merged_record_series['focus_asset_id'])
                    if pd.notnull(assoc_record['associated_asset_id']):
                        processed_assoc_ids.add(assoc_record['associated_asset_id'])
                else:
                    assoc_record_df = pd.DataFrame([assoc_record], columns=cols)
                    failed_prox_audit_records_list.append(ref_record_cleaned)
                    failed_prox_audit_records_list.append(assoc_record_df)
        else:
            failed_prox_audit_records_list.append(ref_record_cleaned)
            
    # Concatenate lists ONCE at the end
    if new_prox_audit_records_list:
        new_records_df = pd.concat(new_prox_audit_records_list, ignore_index=True)
        case2_prox_audits_post_auto_merge_table = pd.concat(
            [case2_prox_audits_post_auto_merge_table, new_records_df], ignore_index=True
        )
    if failed_prox_audit_records_list:
        failed_records_df = pd.concat(failed_prox_audit_records_list, ignore_index=True)
        case2_prox_audits_post_auto_merge_table = pd.concat(
            [case2_prox_audits_post_auto_merge_table, failed_records_df], ignore_index=True
        )
    if case2_auto_merge_further_filter_list:
        case2_auto_merge_further_filter = pd.concat(case2_auto_merge_further_filter_list, ignore_index=False)
    else:
        case2_auto_merge_further_filter = pd.DataFrame(columns=cols)
    if case2_post_auto_merge_list:
        case2_post_auto_merge_table = pd.concat(case2_post_auto_merge_list, ignore_index=True)
    else:
        case2_post_auto_merge_table = pd.DataFrame(columns=cols)
    if case2_raw_post_auto_merge_list:
        case2_raw_post_auto_merge_table = pd.concat(case2_raw_post_auto_merge_list, ignore_index=True)
    else:
        case2_raw_post_auto_merge_table = pd.DataFrame(columns=cols)

    end_time = time.time()
    duration_minutes = (end_time - start_time) / 60
    print(f"--- Case 2 Processing completed in: {duration_minutes:.2f} minutes ---")
    
    return (
        case2_auto_merge_further_filter, 
        case2_prox_audits_post_auto_merge_table.drop_duplicates(ignore_index=True),
        case2_post_auto_merge_table.drop_duplicates(ignore_index=True), 
        case2_raw_post_auto_merge_table.drop_duplicates(ignore_index=True)
    )

# `apply_case_2_maintenance_logic` (No changes needed)
def apply_case_2_maintenance_logic(
    prox_audits_table: pd.DataFrame,
    post_auto_merge_table: pd.DataFrame, 
    post_merge_table: pd.DataFrame, 
    raw_post_merge_table: pd.DataFrame,
    running_final_asset_table: pd.DataFrame
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if post_auto_merge_table.empty:
        empty_df = pd.DataFrame(columns=prox_audits_table.columns)
        return running_final_asset_table, empty_df
    working_post_merge = post_auto_merge_table.reset_index(drop=True)
    final_asset_table_list = []
    associated_records_mask = working_post_merge['associated_asset_id'].notnull()
    if not post_merge_table.empty:
        post_merge_lookup = post_merge_table.set_index('focus_asset_id')
        all_cols = working_post_merge.columns.tolist()
        cols_to_exclude = ["audit_reason", "distance_to_reference", "agldiff_to_reference", "associated_asset_id", "index"]
        cols_to_update = [col for col in all_cols if col not in cols_to_exclude]
        for idx, assoc_record in working_post_merge[associated_records_mask].iterrows():
            assoc_asset_id = assoc_record['associated_asset_id']
            if assoc_asset_id in post_merge_lookup.index:
                matching_merged_record = post_merge_lookup.loc[assoc_asset_id]
                if isinstance(matching_merged_record, pd.DataFrame):
                    matching_merged_record = matching_merged_record.iloc[0]
                for col in cols_to_update:
                    if col in matching_merged_record.index:
                        if col in working_post_merge.columns:
                            working_post_merge.loc[idx, col] = matching_merged_record[col]
    if not raw_post_merge_table.empty:
        raw_assoc_ids = set(raw_post_merge_table['associated_asset_id'].dropna())
        removal_mask = (working_post_merge['associated_asset_id'].notnull()) & \
                       (working_post_merge['associated_asset_id'].isin(raw_assoc_ids))
        working_post_merge = working_post_merge[~removal_mask]
    valid_groups = working_post_merge['focus_asset_id'].dropna()
    if not valid_groups.empty:
        group_sizes = working_post_merge.groupby('focus_asset_id').size()
        single_record_groups = group_sizes[group_sizes == 1].index
        final_asset_table_df = working_post_merge[
            working_post_merge['focus_asset_id'].isin(single_record_groups)
        ].copy()
        final_asset_table_list.append(final_asset_table_df)
        working_post_merge = working_post_merge[
            ~working_post_merge['focus_asset_id'].isin(single_record_groups)
        ]
    sorted_post_merge_table = working_post_merge.sort_values(
        by=['focus_asset_id', 'associated_asset_id'], 
        ascending=[True, True],
        na_position='first'
    ).reset_index(drop=True)
    case2_final_assets = pd.concat(final_asset_table_list, ignore_index=True)
    if case2_final_assets.empty:
        case2_final_assets = pd.DataFrame(columns=prox_audits_table.columns) 
    aggregated_final_asset_table = pd.concat([running_final_asset_table, case2_final_assets], ignore_index=True)
    return aggregated_final_asset_table, sorted_post_merge_table


# `split_case_3_audits` (with ASR Cleaning)
def split_case_3_audits(case2_sorted_post_merge_table: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    working_table = case2_sorted_post_merge_table.copy()
    working_table = clean_asr_in_dataframe(working_table)
    candidates_indices = set()
    grouped = working_table.groupby('focus_asset_id')
    for focus_asset_id, group in grouped:
        reference_record_df = group[group['associated_asset_id'].isnull()]
        associated_records_df = group[group['associated_asset_id'].notnull()]
        ref_index = reference_record_df.index[0] if len(reference_record_df) == 1 else None
        if len(reference_record_df) == 1 and ref_index is not None:
            reference_record = reference_record_df.iloc[0]
            ref_fcc = reference_record['fcc_asr_number']
            ref_faa = reference_record['faa_study_number']
            if pd.notnull(ref_fcc):
                mask_ref_null = pd.isnull(ref_faa)
                mask_assoc_not_null = associated_records_df['faa_study_number'].notnull()
                mask_ref_not_null = pd.notnull(ref_faa)
                mask_assoc_null = associated_records_df['faa_study_number'].isnull()
                matching_mask = (associated_records_df['fcc_asr_number'] == ref_fcc) & \
                                (associated_records_df['fcc_asr_number'].notnull()) & \
                                (
                                    (mask_ref_null & mask_assoc_not_null) |
                                    (mask_ref_not_null & mask_assoc_null)
                                )
                matching_associated_records = associated_records_df[matching_mask]
                if not matching_associated_records.empty:
                    candidates_indices.add(ref_index)
                    candidates_indices.update(matching_associated_records.index)
    case3_auto_merge_candidates = working_table.loc[list(candidates_indices)].copy()
    all_original_indices = set(working_table.index) 
    final_post_merge_indices = all_original_indices.difference(candidates_indices)
    case3_prox_audits_post_auto_merge_table = working_table.loc[list(final_post_merge_indices)].copy()
    cols = working_table.columns
    if case3_auto_merge_candidates.empty:
         case3_auto_merge_candidates = pd.DataFrame(columns=cols)
    if case3_prox_audits_post_auto_merge_table.empty:
         case3_prox_audits_post_auto_merge_table = pd.DataFrame(columns=cols)
    return case3_auto_merge_candidates, case3_prox_audits_post_auto_merge_table


# REFACTORED: `apply_case_3_full_processing` (List-Append, Set-Filter, NO ThreadPool)
def apply_case_3_full_processing(
    candidates_table: pd.DataFrame,
    initial_prox_audits_table: pd.DataFrame
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    
    start_time = time.time()
    print("Starting Case 3 processing...")
    
    merging_timestamp = datetime.now()
    cols = candidates_table.columns
    merge_source_check = f"Auto-Merged {merging_timestamp.strftime('%m/%Y')}"
    
    case3_auto_merge_further_filter_list = []
    case3_raw_post_auto_merge_list = []
    case3_post_auto_merge_list = []
    new_prox_audit_records_list = []
    failed_prox_audit_records_list = []
    
    case3_prox_audits_post_auto_merge_table = initial_prox_audits_table.copy()
    
    processed_focus_ids = set()
    processed_assoc_ids = set()
    
    if not pd.api.types.is_datetime64_any_dtype(candidates_table['created_at']):
        candidates_table['created_at'] = pd.to_datetime(candidates_table['created_at'], errors='coerce')
    candidates_table['created_at_month'] = candidates_table['created_at'].dt.to_period('M')
    
    grouped = candidates_table.groupby('focus_asset_id')
    
    for focus_asset_id, group in tqdm(grouped, desc="Processing Case 3"):
        reference_record_df = group[group['associated_asset_id'].isnull()]
        associated_records_df = group[group['associated_asset_id'].notnull()].copy()
        
        if len(reference_record_df) != 1:
            continue
            
        reference_record = reference_record_df.iloc[0]
        ref_fcc = reference_record['fcc_asr_number']
        ref_faa = reference_record['faa_study_number']
        ref_source = reference_record['source']
        ref_year_month = reference_record['created_at_month']
        
        remaining_associated_records_df = associated_records_df.copy()
        indices_to_remove = set()
        
        if processed_focus_ids:
            focus_id_match_mask = remaining_associated_records_df['associated_asset_id'].isin(processed_focus_ids)
            indices_to_remove.update(remaining_associated_records_df[focus_id_match_mask].index)
        if processed_assoc_ids:
            assoc_id_match_mask = remaining_associated_records_df['associated_asset_id'].isin(processed_assoc_ids)
            indices_to_remove.update(remaining_associated_records_df[assoc_id_match_mask].index)

        source_match_mask = (remaining_associated_records_df['source'] == merge_source_check)
        indices_to_remove.update(remaining_associated_records_df[source_match_mask].index)
        
        ref_source_match = reference_record['source'] == merge_source_check
        
        ref_record_cleaned = reference_record_df.copy()
        if 'created_at_month' in ref_record_cleaned.columns: ref_record_cleaned.drop(columns=['created_at_month'], inplace=True)
        
        if ref_source_match:
            group_associated_cleaned = associated_records_df.copy()
            if 'created_at_month' in group_associated_cleaned.columns: group_associated_cleaned.drop(columns=['created_at_month'], inplace=True)
            failed_prox_audit_records_list.append(ref_record_cleaned)
            if not group_associated_cleaned.empty:
                failed_prox_audit_records_list.append(group_associated_cleaned)
            continue 

        remaining_associated_records_df['created_at_month'] = remaining_associated_records_df['created_at'].dt.to_period('M')
        date_source_match_mask = (remaining_associated_records_df['source'] == ref_source) & \
                                 (remaining_associated_records_df['created_at_month'] == ref_year_month)
        indices_to_remove.update(remaining_associated_records_df[date_source_match_mask].index)

        removed_associated_records_df = remaining_associated_records_df.loc[list(indices_to_remove)].copy()
        remaining_associated_records_df = remaining_associated_records_df.loc[
            ~remaining_associated_records_df.index.isin(indices_to_remove)
        ]

        if 'created_at_month' in removed_associated_records_df.columns: removed_associated_records_df.drop(columns=['created_at_month'], inplace=True)
        if 'created_at_month' in remaining_associated_records_df.columns: remaining_associated_records_df.drop(columns=['created_at_month'], inplace=True)

        if not removed_associated_records_df.empty:
            failed_prox_audit_records_list.append(removed_associated_records_df)

        if len(remaining_associated_records_df) > 0:
            current_group_to_merge = pd.concat([ref_record_cleaned, remaining_associated_records_df], ignore_index=False)
            case3_auto_merge_further_filter_list.append(current_group_to_merge)

            # SIMPLE LOOP (No ThreadPool)
            for index, assoc_record in remaining_associated_records_df.iterrows():
                # This call is now an instant cache hit
                final_faa = determine_faa_study_number(
                    ref_fcc, 
                    ref_faa,
                    assoc_record['faa_study_number']
                )

                if pd.notnull(final_faa):
                    merged_record_series = merge_records(reference_record, assoc_record, merging_timestamp, faa_study_number=final_faa)
                    merged_record_df = pd.DataFrame([merged_record_series], columns=cols)

                    new_prox_audit_records_list.append(merged_record_df)
                    case3_post_auto_merge_list.append(merged_record_df)
                    case3_raw_post_auto_merge_list.append(merged_record_df)

                    assoc_record_df = pd.DataFrame([assoc_record], columns=cols)
                    case3_raw_post_auto_merge_list.append(assoc_record_df)
                    
                    ref_record_df_cleaned_raw = ref_record_cleaned.copy()
                    case3_raw_post_auto_merge_list.append(ref_record_df_cleaned_raw)
                    
                    if pd.notnull(merged_record_series['focus_asset_id']):
                        processed_focus_ids.add(merged_record_series['focus_asset_id'])
                    if pd.notnull(assoc_record['associated_asset_id']):
                        processed_assoc_ids.add(assoc_record['associated_asset_id'])
                else:
                    assoc_record_df = pd.DataFrame([assoc_record], columns=cols)
                    failed_prox_audit_records_list.append(ref_record_cleaned)
                    failed_prox_audit_records_list.append(assoc_record_df)
        else:
            failed_prox_audit_records_list.append(ref_record_cleaned)
            
    # Concatenate lists ONCE at the end
    if new_prox_audit_records_list:
        new_records_df = pd.concat(new_prox_audit_records_list, ignore_index=True)
        case3_prox_audits_post_auto_merge_table = pd.concat(
            [case3_prox_audits_post_auto_merge_table, new_records_df], ignore_index=True
        )
    if failed_prox_audit_records_list:
        failed_records_df = pd.concat(failed_prox_audit_records_list, ignore_index=True)
        case3_prox_audits_post_auto_merge_table = pd.concat(
            [case3_prox_audits_post_auto_merge_table, failed_records_df], ignore_index=True
        )
    if case3_auto_merge_further_filter_list:
        case3_auto_merge_further_filter = pd.concat(case3_auto_merge_further_filter_list, ignore_index=False)
    else:
        case3_auto_merge_further_filter = pd.DataFrame(columns=cols)
    if case3_post_auto_merge_list:
        case3_post_auto_merge_table = pd.concat(case3_post_auto_merge_list, ignore_index=True)
    else:
        case3_post_auto_merge_table = pd.DataFrame(columns=cols)
    if case3_raw_post_auto_merge_list:
        case3_raw_post_auto_merge_table = pd.concat(case3_raw_post_auto_merge_list, ignore_index=True)
    else:
        case3_raw_post_auto_merge_table = pd.DataFrame(columns=cols)

    end_time = time.time()
    duration_minutes = (end_time - start_time) / 60
    print(f"--- Case 3 Processing completed in: {duration_minutes:.2f} minutes ---")
    
    return (
        case3_auto_merge_further_filter, 
        case3_prox_audits_post_auto_merge_table.drop_duplicates(ignore_index=True),
        case3_post_auto_merge_table.drop_duplicates(ignore_index=True), 
        case3_raw_post_auto_merge_table.drop_duplicates(ignore_index=True)
    )

# `apply_case_3_maintenance_logic` (No changes needed)
def apply_case_3_maintenance_logic(
    prox_audits_table: pd.DataFrame,
    post_auto_merge_table: pd.DataFrame, 
    post_merge_table: pd.DataFrame, 
    raw_post_merge_table: pd.DataFrame,
    running_final_asset_table: pd.DataFrame
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if post_auto_merge_table.empty:
        empty_df = pd.DataFrame(columns=prox_audits_table.columns)
        return running_final_asset_table, empty_df
    working_post_merge = post_auto_merge_table.reset_index(drop=True)
    final_asset_table_list = []
    associated_records_mask = working_post_merge['associated_asset_id'].notnull()
    if not post_merge_table.empty:
        post_merge_lookup = post_merge_table.set_index('focus_asset_id')
        all_cols = working_post_merge.columns.tolist()
        cols_to_exclude = ["audit_reason", "distance_to_reference", "agldiff_to_reference", "associated_asset_id", "index"]
        cols_to_update = [col for col in all_cols if col not in cols_to_exclude]
        for idx, assoc_record in working_post_merge[associated_records_mask].iterrows():
            assoc_asset_id = assoc_record['associated_asset_id']
            if assoc_asset_id in post_merge_lookup.index:
                matching_merged_record = post_merge_lookup.loc[assoc_asset_id]
                if isinstance(matching_merged_record, pd.DataFrame):
                    matching_merged_record = matching_merged_record.iloc[0]
                for col in cols_to_update:
                    if col in matching_merged_record.index:
                        if col in working_post_merge.columns:
                            working_post_merge.loc[idx, col] = matching_merged_record[col]
    if not raw_post_merge_table.empty:
        raw_assoc_ids = set(raw_post_merge_table['associated_asset_id'].dropna())
        removal_mask = (working_post_merge['associated_asset_id'].notnull()) & \
                       (working_post_merge['associated_asset_id'].isin(raw_assoc_ids))
        working_post_merge = working_post_merge[~removal_mask]
    valid_groups = working_post_merge['focus_asset_id'].dropna()
    if not valid_groups.empty:
        group_sizes = working_post_merge.groupby('focus_asset_id').size()
        single_record_groups = group_sizes[group_sizes == 1].index
        final_asset_table_df = working_post_merge[
            working_post_merge['focus_asset_id'].isin(single_record_groups)
        ].copy()
        final_asset_table_list.append(final_asset_table_df)
        working_post_merge = working_post_merge[
            ~working_post_merge['focus_asset_id'].isin(single_record_groups)
        ]
    sorted_post_merge_table = working_post_merge.sort_values(
        by=['focus_asset_id', 'associated_asset_id'], 
        ascending=[True, True],
        na_position='first'
    ).reset_index(drop=True)
    case3_final_assets = pd.concat(final_asset_table_list, ignore_index=True)
    if case3_final_assets.empty:
        case3_final_assets = pd.DataFrame(columns=prox_audits_table.columns) 
    aggregated_final_asset_table = pd.concat([running_final_asset_table, case3_final_assets], ignore_index=True)
    return aggregated_final_asset_table, sorted_post_merge_table


# `split_case_4_audits` (with ASR Cleaning)
def split_case_4_audits(case3_sorted_post_merge_table: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    working_table = case3_sorted_post_merge_table.copy()
    working_table = clean_asr_in_dataframe(working_table)
    candidates_indices = set()
    grouped = working_table.groupby('focus_asset_id')
    for focus_asset_id, group in grouped:
        reference_record_df = group[group['associated_asset_id'].isnull()]
        associated_records_df = group[group['associated_asset_id'].notnull()]
        ref_index = reference_record_df.index[0] if len(reference_record_df) == 1 else None
        if len(reference_record_df) == 1 and ref_index is not None:
            reference_record = reference_record_df.iloc[0]
            if pd.isnull(reference_record['fcc_asr_number']) and pd.isnull(reference_record['faa_study_number']):
                ref_op_name = reference_record['operator_name']
                ref_op_site_id = reference_record['operator_site_id']
                ref_asset_status = reference_record['asset_status']
                ref_type = reference_record['type']
                ref_name = reference_record['name']
                mask_assoc_not_null = (associated_records_df['fcc_asr_number'].notnull()) & \
                                      (associated_records_df['faa_study_number'].notnull())
                mask_field_match = (associated_records_df['operator_name'] == ref_op_name) & \
                                   (associated_records_df['operator_site_id'] == ref_op_site_id) & \
                                   (associated_records_df['asset_status'] == ref_asset_status) & \
                                   (associated_records_df['type'] == ref_type) & \
                                   (associated_records_df['name'] == ref_name)
                final_matching_mask = mask_assoc_not_null & mask_field_match
                matching_associated_records = associated_records_df[final_matching_mask]
                if not matching_associated_records.empty:
                    candidates_indices.add(ref_index)
                    candidates_indices.update(matching_associated_records.index)
    case4_auto_merge_candidates = working_table.loc[list(candidates_indices)].copy()
    all_original_indices = set(working_table.index) 
    final_post_merge_indices = all_original_indices.difference(candidates_indices)
    case4_prox_audits_post_auto_merge_table = working_table.loc[list(final_post_merge_indices)].copy()
    cols = working_table.columns
    if case4_auto_merge_candidates.empty:
         case4_auto_merge_candidates = pd.DataFrame(columns=cols)
    if case4_prox_audits_post_auto_merge_table.empty:
         case4_prox_audits_post_auto_merge_table = pd.DataFrame(columns=cols)
    return case4_auto_merge_candidates, case4_prox_audits_post_auto_merge_table


# REFACTORED: `apply_case_4_full_processing` (List-Append, Set-Filter, .copy() Fix, NO ThreadPool)
def apply_case_4_full_processing(
    candidates_table: pd.DataFrame,
    initial_prox_audits_table: pd.DataFrame
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:

    start_time = time.time()
    print("Starting Case 4 processing...")
    
    merging_timestamp = datetime.now()
    cols = candidates_table.columns
    merge_source_check = f"Auto-Merged {merging_timestamp.strftime('%m/%Y')}"
    
    case4_auto_merge_further_filter_list = []
    case4_raw_post_auto_merge_list = []
    case4_post_auto_merge_list = []
    new_prox_audit_records_list = []
    failed_prox_audit_records_list = []
    
    case4_prox_audits_post_auto_merge_table = initial_prox_audits_table.copy()
    
    processed_focus_ids = set()
    processed_assoc_ids = set()
    
    if not pd.api.types.is_datetime64_any_dtype(candidates_table['created_at']):
        candidates_table['created_at'] = pd.to_datetime(candidates_table['created_at'], errors='coerce')
    candidates_table['created_at_month'] = candidates_table['created_at'].dt.to_period('M')
    
    grouped = candidates_table.groupby('focus_asset_id')
    
    for focus_asset_id, group in tqdm(grouped, desc="Processing Case 4"):
        reference_record_df = group[group['associated_asset_id'].isnull()]
        associated_records_df = group[group['associated_asset_id'].notnull()].copy()
        
        if len(reference_record_df) != 1:
            continue
            
        reference_record = reference_record_df.iloc[0]
        ref_source = reference_record['source']
        ref_year_month = reference_record['created_at_month']
        
        remaining_associated_records_df = associated_records_df.copy()
        indices_to_remove = set()
        
        if processed_focus_ids:
            focus_id_match_mask = remaining_associated_records_df['associated_asset_id'].isin(processed_focus_ids)
            indices_to_remove.update(remaining_associated_records_df[focus_id_match_mask].index)
        if processed_assoc_ids:
            assoc_id_match_mask = remaining_associated_records_df['associated_asset_id'].isin(processed_assoc_ids)
            indices_to_remove.update(remaining_associated_records_df[assoc_id_match_mask].index)

        source_match_mask = (remaining_associated_records_df['source'] == merge_source_check)
        indices_to_remove.update(remaining_associated_records_df[source_match_mask].index)
        
        ref_source_match = reference_record['source'] == merge_source_check
        
        ref_record_cleaned = reference_record_df.copy()
        if 'created_at_month' in ref_record_cleaned.columns: ref_record_cleaned.drop(columns=['created_at_month'], inplace=True)
        
        if ref_source_match:
            group_associated_cleaned = associated_records_df.copy()
            if 'created_at_month' in group_associated_cleaned.columns: group_associated_cleaned.drop(columns=['created_at_month'], inplace=True)
            failed_prox_audit_records_list.append(ref_record_cleaned)
            if not group_associated_cleaned.empty:
                failed_prox_audit_records_list.append(group_associated_cleaned)
            continue 

        remaining_associated_records_df['created_at_month'] = remaining_associated_records_df['created_at'].dt.to_period('M')
        date_source_match_mask = (remaining_associated_records_df['source'] == ref_source) & \
                                 (remaining_associated_records_df['created_at_month'] == ref_year_month)
        indices_to_remove.update(remaining_associated_records_df[date_source_match_mask].index)
        
        removed_after_abcde_df = remaining_associated_records_df.loc[list(indices_to_remove)].copy()
        
        # ROBUSTNESS: Add .copy() to prevent SettingWithCopyWarning
        remaining_associated_records_df = remaining_associated_records_df.loc[
            ~remaining_associated_records_df.index.isin(indices_to_remove)
        ].copy() # <--- FIX
        
        # AGL percentage difference
        indices_to_remove_f = set()
        if not remaining_associated_records_df.empty:
            ref_agl = pd.to_numeric(reference_record['agl'], errors='coerce')
            if pd.notnull(ref_agl):
                assoc_agl_series = pd.to_numeric(remaining_associated_records_df['agl'], errors='coerce')
                diff = ((ref_agl - assoc_agl_series).abs() / assoc_agl_series) * 100
                diff_filled = diff.fillna(0).replace([np.inf, -np.inf], 999) 
                agl_diff_mask = diff_filled > 25
                indices_to_remove_f.update(remaining_associated_records_df[agl_diff_mask].index)
        
        removed_after_f_df = remaining_associated_records_df.loc[list(indices_to_remove_f)].copy()
        
        # Add .copy() to prevent SettingWithCopyWarning
        remaining_associated_records_df = remaining_associated_records_df.loc[
            ~remaining_associated_records_df.index.isin(indices_to_remove_f)
        ].copy() # <--- FIX

        # Least distance logic
        removed_after_g_df = pd.DataFrame(columns=remaining_associated_records_df.columns)
        if len(remaining_associated_records_df) > 1:
            remaining_associated_records_df['abs_agldiff'] = pd.to_numeric(
                remaining_associated_records_df['agldiff_to_reference'], errors='coerce'
            ).abs()
            
            if not remaining_associated_records_df['abs_agldiff'].isnull().all():
                closest_record_index = remaining_associated_records_df['abs_agldiff'].idxmin()
                closest_record_df = remaining_associated_records_df.loc[[closest_record_index]]
                removed_after_g_df = remaining_associated_records_df.loc[
                    ~remaining_associated_records_df.index.isin([closest_record_index])
                ]
                remaining_associated_records_df = closest_record_df
            else:
                removed_after_g_df = remaining_associated_records_df.copy()
                remaining_associated_records_df = pd.DataFrame(columns=remaining_associated_records_df.columns)
        
        if 'abs_agldiff' in remaining_associated_records_df.columns:
            remaining_associated_records_df = remaining_associated_records_df.drop(columns=['abs_agldiff'])
        
        all_removed_associated_records_df = pd.concat([
            removed_after_abcde_df, removed_after_f_df, removed_after_g_df
        ], ignore_index=False)
        
        if 'created_at_month' in all_removed_associated_records_df.columns: all_removed_associated_records_df.drop(columns=['created_at_month'], inplace=True)
        if 'created_at_month' in remaining_associated_records_df.columns: remaining_associated_records_df.drop(columns=['created_at_month'], inplace=True)
                
        if not all_removed_associated_records_df.empty:
            failed_prox_audit_records_list.append(all_removed_associated_records_df)
        
        # Final Check and Merge Logic
        if len(remaining_associated_records_df) == 1:
            current_group_to_merge = pd.concat([ref_record_cleaned, remaining_associated_records_df], ignore_index=False)
            case4_auto_merge_further_filter_list.append(current_group_to_merge)
            
            assoc_record = remaining_associated_records_df.iloc[0]
            
            # This call is now an instant cache hit
            final_faa = determine_faa_study_number(
                assoc_record['fcc_asr_number'],    
                reference_record['faa_study_number'], 
                assoc_record['faa_study_number']
            )

            if pd.notnull(final_faa):
                merged_record_series = merge_records(reference_record, assoc_record, merging_timestamp, faa_study_number=final_faa)
                merged_record_df = pd.DataFrame([merged_record_series], columns=cols)
                
                new_prox_audit_records_list.append(merged_record_df)
                case4_post_auto_merge_list.append(merged_record_df)
                case4_raw_post_auto_merge_list.append(merged_record_df)

                assoc_record_df = pd.DataFrame([assoc_record], columns=cols)
                case4_raw_post_auto_merge_list.append(assoc_record_df)
                
                ref_record_df_cleaned_raw = ref_record_cleaned.copy()
                case4_raw_post_auto_merge_list.append(ref_record_df_cleaned_raw)

                if pd.notnull(merged_record_series['focus_asset_id']):
                    processed_focus_ids.add(merged_record_series['focus_asset_id'])
                if pd.notnull(assoc_record['associated_asset_id']):
                    processed_assoc_ids.add(assoc_record['associated_asset_id'])
            else:
                assoc_record_df = pd.DataFrame([assoc_record], columns=cols)
                failed_prox_audit_records_list.append(ref_record_cleaned)
                failed_prox_audit_records_list.append(assoc_record_df)
        else:
            failed_prox_audit_records_list.append(ref_record_cleaned)
            
    # Concatenate lists ONCE at the end
    if new_prox_audit_records_list:
        new_records_df = pd.concat(new_prox_audit_records_list, ignore_index=True)
        case4_prox_audits_post_auto_merge_table = pd.concat(
            [case4_prox_audits_post_auto_merge_table, new_records_df], ignore_index=True
        )
    if failed_prox_audit_records_list:
        failed_records_df = pd.concat(failed_prox_audit_records_list, ignore_index=True)
        case4_prox_audits_post_auto_merge_table = pd.concat(
            [case4_prox_audits_post_auto_merge_table, failed_records_df], ignore_index=True
        )
    if case4_auto_merge_further_filter_list:
        case4_auto_merge_further_filter = pd.concat(case4_auto_merge_further_filter_list, ignore_index=False)
    else:
        case4_auto_merge_further_filter = pd.DataFrame(columns=cols)
    if case4_post_auto_merge_list:
        case4_post_auto_merge_table = pd.concat(case4_post_auto_merge_list, ignore_index=True)
    else:
        case4_post_auto_merge_table = pd.DataFrame(columns=cols)
    if case4_raw_post_auto_merge_list:
        case4_raw_post_auto_merge_table = pd.concat(case4_raw_post_auto_merge_list, ignore_index=True)
    else:
        case4_raw_post_auto_merge_table = pd.DataFrame(columns=cols)

    end_time = time.time()
    duration_minutes = (end_time - start_time) / 60
    print(f"--- Case 4 Processing completed in: {duration_minutes:.2f} minutes ---")
    
    return (
        case4_auto_merge_further_filter, 
        case4_prox_audits_post_auto_merge_table.drop_duplicates(ignore_index=True),
        case4_post_auto_merge_table.drop_duplicates(ignore_index=True), 
        case4_raw_post_auto_merge_table.drop_duplicates(ignore_index=True)
    )

# `apply_case_4_maintenance_logic` (No changes needed)
def apply_case_4_maintenance_logic(
    prox_audits_table: pd.DataFrame,
    post_auto_merge_table: pd.DataFrame, 
    post_merge_table: pd.DataFrame, 
    raw_post_merge_table: pd.DataFrame,
    running_final_asset_table: pd.DataFrame
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if post_auto_merge_table.empty:
        empty_df = pd.DataFrame(columns=prox_audits_table.columns)
        return running_final_asset_table, empty_df
    working_post_merge = post_auto_merge_table.reset_index(drop=True)
    final_asset_table_list = []
    associated_records_mask = working_post_merge['associated_asset_id'].notnull()
    if not post_merge_table.empty:
        post_merge_lookup = post_merge_table.set_index('focus_asset_id')
        all_cols = working_post_merge.columns.tolist()
        cols_to_exclude = ["audit_reason", "distance_to_reference", "agldiff_to_reference", "associated_asset_id", "index"]
        cols_to_update = [col for col in all_cols if col not in cols_to_exclude]
        for idx, assoc_record in working_post_merge[associated_records_mask].iterrows():
            assoc_asset_id = assoc_record['associated_asset_id']
            if assoc_asset_id in post_merge_lookup.index:
                matching_merged_record = post_merge_lookup.loc[assoc_asset_id]
                if isinstance(matching_merged_record, pd.DataFrame):
                    matching_merged_record = matching_merged_record.iloc[0]
                for col in cols_to_update:
                    if col in matching_merged_record.index:
                        if col in working_post_merge.columns:
                            working_post_merge.loc[idx, col] = matching_merged_record[col]
    if not raw_post_merge_table.empty:
        raw_assoc_ids = set(raw_post_merge_table['associated_asset_id'].dropna())
        removal_mask = (working_post_merge['associated_asset_id'].notnull()) & \
                       (working_post_merge['associated_asset_id'].isin(raw_assoc_ids))
        working_post_merge = working_post_merge[~removal_mask]
    valid_groups = working_post_merge['focus_asset_id'].dropna()
    if not valid_groups.empty:
        group_sizes = working_post_merge.groupby('focus_asset_id').size()
        single_record_groups = group_sizes[group_sizes == 1].index
        final_asset_table_df = working_post_merge[
            working_post_merge['focus_asset_id'].isin(single_record_groups)
        ].copy()
        final_asset_table_list.append(final_asset_table_df)
        working_post_merge = working_post_merge[
            ~working_post_merge['focus_asset_id'].isin(single_record_groups)
        ]
    sorted_post_merge_table = working_post_merge.sort_values(
        by=['focus_asset_id', 'associated_asset_id'], 
        ascending=[True, True],
        na_position='first'
    ).reset_index(drop=True)
    case4_final_assets = pd.concat(final_asset_table_list, ignore_index=True)
    if case4_final_assets.empty:
        case4_final_assets = pd.DataFrame(columns=prox_audits_table.columns) 
    aggregated_final_asset_table = pd.concat([running_final_asset_table, case4_final_assets], ignore_index=True)
    return aggregated_final_asset_table, sorted_post_merge_table


def split_case_5_audits(case4_sorted_post_merge_table: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    
    working_table = case4_sorted_post_merge_table.copy()
    
    # ASR cleaning is not strictly necessary as we're checking for NULL,
    # but it's good practice to keep it consistent.
    working_table = clean_asr_in_dataframe(working_table)

    candidates_indices = set()
    grouped = working_table.groupby('focus_asset_id')

    for focus_asset_id, group in grouped:
        reference_record_df = group[group['associated_asset_id'].isnull()]
        associated_records_df = group[group['associated_asset_id'].notnull()]
        
        ref_index = reference_record_df.index[0] if len(reference_record_df) == 1 else None

        if len(reference_record_df) == 1 and ref_index is not None:
            reference_record = reference_record_df.iloc[0]
            
            # Case 5 Split Logic
            # Reference record must have NULL FCC and NULL FAA
            ref_is_null = pd.isnull(reference_record['fcc_asr_number']) and \
                          pd.isnull(reference_record['faa_study_number'])
            
            if ref_is_null:
                # Get fields for 5-way match
                ref_op_name = reference_record['operator_name']
                ref_op_site_id = reference_record['operator_site_id']
                ref_asset_status = reference_record['asset_status']
                ref_type = reference_record['type']
                ref_name = reference_record['name']

                # Associated records must also have NULL FCC and NULL FAA
                mask_assoc_is_null = (associated_records_df['fcc_asr_number'].isnull()) & \
                                     (associated_records_df['faa_study_number'].isnull())
                
                # Associated records must match on the 5 key fields
                mask_field_match = (associated_records_df['operator_name'] == ref_op_name) & \
                                   (associated_records_df['operator_site_id'] == ref_op_site_id) & \
                                   (associated_records_df['asset_status'] == ref_asset_status) & \
                                   (associated_records_df['type'] == ref_type) & \
                                   (associated_records_df['name'] == ref_name)
                
                final_matching_mask = mask_assoc_is_null & mask_field_match
                
                matching_associated_records = associated_records_df[final_matching_mask]
                
                if not matching_associated_records.empty:
                    candidates_indices.add(ref_index)
                    candidates_indices.update(matching_associated_records.index)
            
    case5_auto_merge_candidates = working_table.loc[list(candidates_indices)].copy()
    
    all_original_indices = set(working_table.index) 
    final_post_merge_indices = all_original_indices.difference(candidates_indices)
    
    case5_prox_audits_post_auto_merge_table = working_table.loc[list(final_post_merge_indices)].copy()

    cols = working_table.columns
    if case5_auto_merge_candidates.empty:
         case5_auto_merge_candidates = pd.DataFrame(columns=cols)
         
    if case5_prox_audits_post_auto_merge_table.empty:
         case5_prox_audits_post_auto_merge_table = pd.DataFrame(columns=cols)

    return case5_auto_merge_candidates, case5_prox_audits_post_auto_merge_table


def apply_case_5_full_processing(
    candidates_table: pd.DataFrame,
    initial_prox_audits_table: pd.DataFrame
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:

    # This function is identical to Case 4's processing,
    # but with the web-scraping logic removed.
    
    start_time = time.time()
    print("Starting Case 5 processing...")
    
    merging_timestamp = datetime.now()
    cols = candidates_table.columns
    merge_source_check = f"Auto-Merged {merging_timestamp.strftime('%m/%Y')}"
    
    # PERFORMANCE: Initialize lists
    case5_auto_merge_further_filter_list = []
    case5_raw_post_auto_merge_list = []
    case5_post_auto_merge_list = []
    new_prox_audit_records_list = []
    failed_prox_audit_records_list = []
    
    case5_prox_audits_post_auto_merge_table = initial_prox_audits_table.copy()
    
    # PERFORMANCE: Initialize sets
    processed_focus_ids = set()
    processed_assoc_ids = set()
    
    if not pd.api.types.is_datetime64_any_dtype(candidates_table['created_at']):
        candidates_table['created_at'] = pd.to_datetime(candidates_table['created_at'], errors='coerce')
    candidates_table['created_at_month'] = candidates_table['created_at'].dt.to_period('M')
    
    grouped = candidates_table.groupby('focus_asset_id')
    
    for focus_asset_id, group in tqdm(grouped, desc="Processing Case 5"):
        reference_record_df = group[group['associated_asset_id'].isnull()]
        associated_records_df = group[group['associated_asset_id'].notnull()].copy()
        
        if len(reference_record_df) != 1:
            continue
            
        reference_record = reference_record_df.iloc[0]
        ref_source = reference_record['source']
        ref_year_month = reference_record['created_at_month']
        
        remaining_associated_records_df = associated_records_df.copy()
        indices_to_remove = set()
        

        if processed_focus_ids:
            focus_id_match_mask = remaining_associated_records_df['associated_asset_id'].isin(processed_focus_ids)
            indices_to_remove.update(remaining_associated_records_df[focus_id_match_mask].index)

        if processed_assoc_ids:
            assoc_id_match_mask = remaining_associated_records_df['associated_asset_id'].isin(processed_assoc_ids)
            indices_to_remove.update(remaining_associated_records_df[assoc_id_match_mask].index)

        source_match_mask = (remaining_associated_records_df['source'] == merge_source_check)
        indices_to_remove.update(remaining_associated_records_df[source_match_mask].index)
        

        ref_source_match = reference_record['source'] == merge_source_check
        ref_record_cleaned = reference_record_df.copy()
        if 'created_at_month' in ref_record_cleaned.columns: ref_record_cleaned.drop(columns=['created_at_month'], inplace=True)
        if ref_source_match:
            group_associated_cleaned = associated_records_df.copy()
            if 'created_at_month' in group_associated_cleaned.columns: group_associated_cleaned.drop(columns=['created_at_month'], inplace=True)
            failed_prox_audit_records_list.append(ref_record_cleaned)
            if not group_associated_cleaned.empty:
                failed_prox_audit_records_list.append(group_associated_cleaned)
            continue 


        remaining_associated_records_df['created_at_month'] = remaining_associated_records_df['created_at'].dt.to_period('M')
        date_source_match_mask = (remaining_associated_records_df['source'] == ref_source) & \
                                 (remaining_associated_records_df['created_at_month'] == ref_year_month)
        indices_to_remove.update(remaining_associated_records_df[date_source_match_mask].index)
        
        removed_after_abcde_df = remaining_associated_records_df.loc[list(indices_to_remove)].copy()
        

        remaining_associated_records_df = remaining_associated_records_df.loc[
            ~remaining_associated_records_df.index.isin(indices_to_remove)
        ].copy() # <--- FIX
        
        # AGL percentage difference (Same as Case 4) ---
        indices_to_remove_f = set()
        if not remaining_associated_records_df.empty:
            ref_agl = pd.to_numeric(reference_record['agl'], errors='coerce')
            if pd.notnull(ref_agl):
                assoc_agl_series = pd.to_numeric(remaining_associated_records_df['agl'], errors='coerce')
                # Formula: abs(ref - assoc) / assoc
                # We fillna(0) and replace inf with 999 to handle NaNs and divide-by-zero
                diff = ((ref_agl - assoc_agl_series).abs() / assoc_agl_series) * 100
                diff_filled = diff.fillna(0).replace([np.inf, -np.inf], 999) 
                agl_diff_mask = diff_filled > 25
                indices_to_remove_f.update(remaining_associated_records_df[agl_diff_mask].index)
        
        removed_after_f_df = remaining_associated_records_df.loc[list(indices_to_remove_f)].copy()
        

        remaining_associated_records_df = remaining_associated_records_df.loc[
            ~remaining_associated_records_df.index.isin(indices_to_remove_f)
        ].copy() # <--- FIX

        # Least distance logic (Same as Case 4)
        removed_after_g_df = pd.DataFrame(columns=remaining_associated_records_df.columns)
        if len(remaining_associated_records_df) > 1:
            # This operation is now safe due to .copy() above
            remaining_associated_records_df['abs_agldiff'] = pd.to_numeric(
                remaining_associated_records_df['agldiff_to_reference'], errors='coerce'
            ).abs()
            
            if not remaining_associated_records_df['abs_agldiff'].isnull().all():
                closest_record_index = remaining_associated_records_df['abs_agldiff'].idxmin()
                closest_record_df = remaining_associated_records_df.loc[[closest_record_index]]
                removed_after_g_df = remaining_associated_records_df.loc[
                    ~remaining_associated_records_df.index.isin([closest_record_index])
                ]
                remaining_associated_records_df = closest_record_df
            else:
                # All 'agldiff_to_reference' are NaN, cannot determine closest. Remove all.
                removed_after_g_df = remaining_associated_records_df.copy()
                remaining_associated_records_df = pd.DataFrame(columns=remaining_associated_records_df.columns)
        
        # Drop the temporary column
        if 'abs_agldiff' in remaining_associated_records_df.columns:
            remaining_associated_records_df = remaining_associated_records_df.drop(columns=['abs_agldiff'])
        
        # Collect all removed records
        all_removed_associated_records_df = pd.concat([
            removed_after_abcde_df, 
            removed_after_f_df,
            removed_after_g_df
        ], ignore_index=False)
        
        if 'created_at_month' in all_removed_associated_records_df.columns: all_removed_associated_records_df.drop(columns=['created_at_month'], inplace=True)
        if 'created_at_month' in remaining_associated_records_df.columns: remaining_associated_records_df.drop(columns=['created_at_month'], inplace=True)
                
        if not all_removed_associated_records_df.empty:
            failed_prox_audit_records_list.append(all_removed_associated_records_df)
        
        # Final Check and Merge Logic
        if len(remaining_associated_records_df) == 1:
            # This is the success case: 1 ref + 1 assoc
            current_group_to_merge = pd.concat([ref_record_cleaned, remaining_associated_records_df], ignore_index=False)
            case5_auto_merge_further_filter_list.append(current_group_to_merge)
            
            assoc_record = remaining_associated_records_df.iloc[0]
            
            # Case 5 Merge Logic (Difference 2)
            # No web scraping needed. Proceed directly to merge.
            # We pass faa_study_number=None, which is correct since both are NULL.
            merged_record_series = merge_records(reference_record, assoc_record, merging_timestamp, faa_study_number=None)
            merged_record_df = pd.DataFrame([merged_record_series], columns=cols)
            
            # Add to lists
            new_prox_audit_records_list.append(merged_record_df)
            case5_post_auto_merge_list.append(merged_record_df)
            case5_raw_post_auto_merge_list.append(merged_record_df)

            assoc_record_df = pd.DataFrame([assoc_record], columns=cols)
            case5_raw_post_auto_merge_list.append(assoc_record_df)
            
            ref_record_df_cleaned_raw = ref_record_cleaned.copy()
            case5_raw_post_auto_merge_list.append(ref_record_df_cleaned_raw)

            # Update sets
            if pd.notnull(merged_record_series['focus_asset_id']):
                processed_focus_ids.add(merged_record_series['focus_asset_id'])
            if pd.notnull(assoc_record['associated_asset_id']):
                processed_assoc_ids.add(assoc_record['associated_asset_id'])
            
        else:
            # This group fails (0 associated records remain after filters)
            failed_prox_audit_records_list.append(ref_record_cleaned)
            
    # Concatenate lists ONCE at the end
    if new_prox_audit_records_list:
        new_records_df = pd.concat(new_prox_audit_records_list, ignore_index=True)
        case5_prox_audits_post_auto_merge_table = pd.concat(
            [case5_prox_audits_post_auto_merge_table, new_records_df], ignore_index=True
        )
    if failed_prox_audit_records_list:
        failed_records_df = pd.concat(failed_prox_audit_records_list, ignore_index=True)
        case5_prox_audits_post_auto_merge_table = pd.concat(
            [case5_prox_audits_post_auto_merge_table, failed_records_df], ignore_index=True
        )
    if case5_auto_merge_further_filter_list:
        case5_auto_merge_further_filter = pd.concat(case5_auto_merge_further_filter_list, ignore_index=False)
    else:
        case5_auto_merge_further_filter = pd.DataFrame(columns=cols)
    if case5_post_auto_merge_list:
        case5_post_auto_merge_table = pd.concat(case5_post_auto_merge_list, ignore_index=True)
    else:
        case5_post_auto_merge_table = pd.DataFrame(columns=cols)
    if case5_raw_post_auto_merge_list:
        case5_raw_post_auto_merge_table = pd.concat(case5_raw_post_auto_merge_list, ignore_index=True)
    else:
        case5_raw_post_auto_merge_table = pd.DataFrame(columns=cols)

    end_time = time.time()
    duration_minutes = (end_time - start_time) / 60
    print(f"--- Case 5 Processing completed in: {duration_minutes:.2f} minutes ---")
    
    return (
        case5_auto_merge_further_filter, 
        case5_prox_audits_post_auto_merge_table.drop_duplicates(ignore_index=True),
        case5_post_auto_merge_table.drop_duplicates(ignore_index=True), 
        case5_raw_post_auto_merge_table.drop_duplicates(ignore_index=True)
    )



def apply_case_5_maintenance_logic(
    prox_audits_table: pd.DataFrame,
    post_auto_merge_table: pd.DataFrame, 
    post_merge_table: pd.DataFrame, 
    raw_post_merge_table: pd.DataFrame,
    running_final_asset_table: pd.DataFrame
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    
    # This function is identical to Case 4's maintenance logic
    
    if post_auto_merge_table.empty:
        empty_df = pd.DataFrame(columns=prox_audits_table.columns)
        # For Case 5, the final output is the running_final_asset_table and the empty_df
        return running_final_asset_table, empty_df
        
    working_post_merge = post_auto_merge_table.reset_index(drop=True)
    final_asset_table_list = []
    

    associated_records_mask = working_post_merge['associated_asset_id'].notnull()
    
    if not post_merge_table.empty:
        post_merge_lookup = post_merge_table.set_index('focus_asset_id')
        all_cols = working_post_merge.columns.tolist()
        cols_to_exclude = ["audit_reason", "distance_to_reference", "agldiff_to_reference", "associated_asset_id", "index"]
        cols_to_update = [col for col in all_cols if col not in cols_to_exclude]
        
        for idx, assoc_record in working_post_merge[associated_records_mask].iterrows():
            assoc_asset_id = assoc_record['associated_asset_id']
            if assoc_asset_id in post_merge_lookup.index:
                matching_merged_record = post_merge_lookup.loc[assoc_asset_id]
                if isinstance(matching_merged_record, pd.DataFrame):
                    matching_merged_record = matching_merged_record.iloc[0]
                for col in cols_to_update:
                    if col in matching_merged_record.index:
                        if col in working_post_merge.columns:
                            working_post_merge.loc[idx, col] = matching_merged_record[col]


    if not raw_post_merge_table.empty:

        raw_assoc_ids = set(raw_post_merge_table['associated_asset_id'].dropna())
        removal_mask = (working_post_merge['associated_asset_id'].notnull()) & \
                       (working_post_merge['associated_asset_id'].isin(raw_assoc_ids))
        working_post_merge = working_post_merge[~removal_mask]


    valid_groups = working_post_merge['focus_asset_id'].dropna()
    if not valid_groups.empty:
        group_sizes = working_post_merge.groupby('focus_asset_id').size()
        single_record_groups = group_sizes[group_sizes == 1].index
        final_asset_table_df = working_post_merge[
            working_post_merge['focus_asset_id'].isin(single_record_groups)
        ].copy()
        final_asset_table_list.append(final_asset_table_df)
        working_post_merge = working_post_merge[
            ~working_post_merge['focus_asset_id'].isin(single_record_groups)
        ]


    sorted_post_merge_table = working_post_merge.sort_values(
        by=['focus_asset_id', 'associated_asset_id'], 
        ascending=[True, True],
        na_position='first'
    ).reset_index(drop=True)

    case5_final_assets = pd.concat(final_asset_table_list, ignore_index=True)
    if case5_final_assets.empty:
        case5_final_assets = pd.DataFrame(columns=prox_audits_table.columns) 
        
    aggregated_final_asset_table = pd.concat([running_final_asset_table, case5_final_assets], ignore_index=True)

    # The final_sorted_table is the last output of the entire 5-case process
    final_sorted_table = sorted_post_merge_table
    
    return aggregated_final_asset_table, final_sorted_table

## Case 1

In [10]:
case1_auto_merge_candidates, initial_case1_prox_audits_post_auto_merge_table = split_case_1_audits(prox_audits_table)

In [11]:
case1_auto_merge_further_filter, updated_case1_prox_audits_post_auto_merge_table, case1_post_auto_merge_table, case1_raw_post_auto_merge_table = apply_case_1_full_processing(case1_auto_merge_candidates, initial_case1_prox_audits_post_auto_merge_table)

Starting Case 1 processing...


Processing Case 1: 100%|████████████████████████| 82/82 [00:30<00:00,  2.72it/s]

--- Case 1 Processing completed in: 0.51 minutes ---


In [12]:
case1_aggregated_final_asset_table, final_case1_prox_audits_post_auto_merge_table = apply_case_1_maintenance_logic(prox_audits_table, updated_case1_prox_audits_post_auto_merge_table, case1_post_auto_merge_table, case1_raw_post_auto_merge_table)

In [13]:
prox_audits_table

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,faa_study_number,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference
0,942614,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:53:42.284115,2023-01-13 15:25:12.493704,33.94619,-118.28200,CA12409,160407,...,<NA>,<NA>,<NA>,"FIGUEROA ST WL 75F S OF 99TH ST S/N , August F...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN
1,942614,Phoenix Tower International,943927,Phoenix Tower International,2019-08-22 21:36:14.93064,2023-06-26 12:05:50.800659,33.94574,-118.28200,MAIN ST EL 185F N OF 110TH ST SF,161773,...,<NA>,<NA>,<NA>,"MAIN ST EL 185F N OF 110TH ST SF , , CA 90003,...",NaN,No,Unconfirmed,Proximity Audit (48m),49.914635,0.0
2,942664,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:55:21.34445,2023-01-13 11:49:35.586868,34.03929,-118.19900,CA12465,160463,...,<NA>,<NA>,<NA>,"1ST ST SL 2F E OF DACOTAH ST E/W , Los Angeles...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN
3,942664,Phoenix Tower International,942203,Phoenix Tower International,2019-08-22 20:40:05.047291,2023-01-13 13:04:52.922687,34.03910,-118.19900,CA11987,159985,...,<NA>,<NA>,<NA>,"1ST ST SL 53F W OF FRESNO ST WF , Los Angeles,...",NaN,No,Unconfirmed,Proximity Audit (21m),21.075388,0.0
4,942761,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:58:33.921034,2023-01-13 12:41:13.613007,33.98899,-118.31900,CA12570,160568,...,<NA>,<NA>,<NA>,"SLAUSON AVE NL 42F E OF 3RD AVE EF , Los Angel...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64688,1161828,ASR,264885,Crown Castle,2015-04-20 20:26:56.595238,2017-07-28 23:55:23.576292,39.00210,-95.73923,SKYLINE PARK/BURNETT'S MOUND,877829,...,<NA>,<NA>,<NA>,"3511 SW SKYLINE PARKWAY , TOPEKA, KS 66614, UN...",2007-01-12,No,Active,Proximity Audit (7m),7.276909,4.5
64689,1161829,ASR,<NA>,ASR,2025-04-01 15:38:18.798044,2025-04-09 11:29:48.458842,32.81861,-86.61714,<NA>,<NA>,...,2024-ASO-23463-OE,<NA>,<NA>,"Off Logan Road , Clanton, AL 35045, UNITED STATES",2024-11-26,No,Active,Focus Asset,NaN,NaN
64690,1161829,ASR,1136163,ASR,2024-01-23 15:24:06.649527,2024-01-23 09:31:32.771277,32.81861,-86.61714,<NA>,<NA>,...,2023-ASO-27414-OE,<NA>,<NA>,"Off Logan Road , Clanton, AL 35046, UNITED STATES",2024-11-26,No,Active,Proximity Audit (0m),0.000000,-20.4
64691,1162574,ASR,<NA>,ASR,2025-07-24 21:02:34.966713,2025-07-28 09:49:51.734224,42.34739,-91.47869,<NA>,<NA>,...,2024-ACE-6738-OE,<NA>,<NA>,"3048 Hwy 13 IA-5254 , Ryan, IA 52330, UNITED S...",NaN,No,ASR-Granted,Focus Asset,NaN,NaN


In [14]:
case1_auto_merge_candidates

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference,created_at_month
33793,1142404,ASR,<NA>,ASR,2024-11-19 23:11:39.656464,2024-11-26 09:33:13.980674,38.96042,-76.21336,<NA>,<NA>,...,<NA>,<NA>,"115 PULLMAN CROSSING ROAD , GRASONVILLE, MD 21...",2025-05-27,No,Active,Focus Asset,NaN,NaN,2024-11
33794,1142404,ASR,1142066,ASR,2024-06-21 15:30:24.344816,2024-11-19 17:21:23.239832,38.96042,-76.21337,Crab Alley Bay,<NA>,...,<NA>,<NA>,"115 Pullman Crossing , Grasonville, MD 21638, ...",2025-05-27,No,Active,ASR Operator Mismatch,0.866746,0.0,2024-06
33795,1142405,ASR,<NA>,ASR,2024-11-19 23:11:39.656464,2024-11-26 09:33:15.186892,38.83472,-76.72933,<NA>,<NA>,...,<NA>,<NA>,"CRAIN HIGHWAY , UPPER MARLBORO, MD 20772, UNIT...",NaN,No,ASR-Granted,Focus Asset,NaN,NaN,2024-11
33796,1142405,ASR,1142072,ASR,2023-10-03 20:08:53.983413,2024-11-19 17:21:27.702541,38.83462,-76.72942,Moccasin,<NA>,...,<NA>,<NA>,"3610 Old Crain Hwy , Upper Marlboro, MD 20772,...",NaN,No,ASR-Granted,ASR Operator Mismatch,13.575846,0.6,2023-10
33802,1142444,ASR,<NA>,ASR,2024-11-19 23:11:39.656464,2024-12-03 10:12:04.760146,38.96042,-76.21336,<NA>,<NA>,...,<NA>,<NA>,"115 PULLMAN CROSSING ROAD , GRASONVILLE, MD 21...",2025-05-27,No,Active,Focus Asset,NaN,NaN,2024-11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34744,1147155,Everest Infrastructure Partners,833422,dledbetter,2018-09-20 10:16:32.231523,2018-09-21 14:49:34.622583,36.04094,-79.22089,Buckhhorn,00060,...,<NA>,<NA>,"2507 Buckhorn Road , Mebane, NC 27302, UNITED ...",2015-05-19,No,Active,Proximity Audit (0m),0.901171,-0.1,2018-09
34745,1147156,Everest Infrastructure Partners,<NA>,Everest Infrastructure Partners,2025-02-18 16:27:48.056800,2025-02-18 16:27:47.993715,35.40977,-77.98728,Goldsboro,US596436,...,<NA>,<NA>,"209z Hooks River Rd , Goldsboro, NC 27530, UNI...",NaN,No,Active,Focus Asset,NaN,NaN,2025-02
34746,1147156,Everest Infrastructure Partners,1137554,sgaither,2015-02-03 23:55:20.061725,2024-06-13 15:43:04.18398,35.40977,-77.98728,Goldsboro,US596436,...,<NA>,<NA>,"209 Hooks River Rd , GOLDSBORO, NC 27530, UNIT...",2015-04-19,No,Active,Manual Edit,0.000000,-0.1,2015-02
34749,1147158,Everest Infrastructure Partners,<NA>,Everest Infrastructure Partners,2025-02-18 16:27:57.187729,2025-02-18 16:27:57.124848,34.03211,-86.31711,ALTOONA - DWNTWN - THOMAS,US539594,...,<NA>,<NA>,"6650 Water Tank Town Road , Altoona, AL 35952,...",NaN,No,Active,Focus Asset,NaN,NaN,2025-02


In [15]:
initial_case1_prox_audits_post_auto_merge_table

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,faa_study_number,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference
0,942614,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:53:42.284115,2023-01-13 15:25:12.493704,33.94619,-118.28200,CA12409,160407,...,<NA>,<NA>,<NA>,"FIGUEROA ST WL 75F S OF 99TH ST S/N , August F...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN
1,942614,Phoenix Tower International,943927,Phoenix Tower International,2019-08-22 21:36:14.93064,2023-06-26 12:05:50.800659,33.94574,-118.28200,MAIN ST EL 185F N OF 110TH ST SF,161773,...,<NA>,<NA>,<NA>,"MAIN ST EL 185F N OF 110TH ST SF , , CA 90003,...",NaN,No,Unconfirmed,Proximity Audit (48m),49.914635,0.0
2,942664,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:55:21.34445,2023-01-13 11:49:35.586868,34.03929,-118.19900,CA12465,160463,...,<NA>,<NA>,<NA>,"1ST ST SL 2F E OF DACOTAH ST E/W , Los Angeles...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN
3,942664,Phoenix Tower International,942203,Phoenix Tower International,2019-08-22 20:40:05.047291,2023-01-13 13:04:52.922687,34.03910,-118.19900,CA11987,159985,...,<NA>,<NA>,<NA>,"1ST ST SL 53F W OF FRESNO ST WF , Los Angeles,...",NaN,No,Unconfirmed,Proximity Audit (21m),21.075388,0.0
4,942761,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:58:33.921034,2023-01-13 12:41:13.613007,33.98899,-118.31900,CA12570,160568,...,<NA>,<NA>,<NA>,"SLAUSON AVE NL 42F E OF 3RD AVE EF , Los Angel...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64688,1161828,ASR,264885,Crown Castle,2015-04-20 20:26:56.595238,2017-07-28 23:55:23.576292,39.00210,-95.73923,SKYLINE PARK/BURNETT'S MOUND,877829,...,<NA>,<NA>,<NA>,"3511 SW SKYLINE PARKWAY , TOPEKA, KS 66614, UN...",2007-01-12,No,Active,Proximity Audit (7m),7.276909,4.5
64689,1161829,ASR,<NA>,ASR,2025-04-01 15:38:18.798044,2025-04-09 11:29:48.458842,32.81861,-86.61714,<NA>,<NA>,...,2024-ASO-23463-OE,<NA>,<NA>,"Off Logan Road , Clanton, AL 35045, UNITED STATES",2024-11-26,No,Active,Focus Asset,NaN,NaN
64690,1161829,ASR,1136163,ASR,2024-01-23 15:24:06.649527,2024-01-23 09:31:32.771277,32.81861,-86.61714,<NA>,<NA>,...,2023-ASO-27414-OE,<NA>,<NA>,"Off Logan Road , Clanton, AL 35046, UNITED STATES",2024-11-26,No,Active,Proximity Audit (0m),0.000000,-20.4
64691,1162574,ASR,<NA>,ASR,2025-07-24 21:02:34.966713,2025-07-28 09:49:51.734224,42.34739,-91.47869,<NA>,<NA>,...,2024-ACE-6738-OE,<NA>,<NA>,"3048 Hwy 13 IA-5254 , Ryan, IA 52330, UNITED S...",NaN,No,ASR-Granted,Focus Asset,NaN,NaN


In [16]:
case1_auto_merge_further_filter

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,faa_study_number,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference
33622,1139789,ASR,<NA>,ASR,2021-02-22 14:55:46.271143,2024-07-16 10:20:53.166299,43.19161,-86.09733,<NA>,<NA>,...,2020-AGL-18544-OE,<NA>,<NA>,"5998 Heights Ravenna Road (MI-0028) , Fruitpor...",2020-06-25,No,Active,Focus Asset,NaN,NaN
33623,1139789,ASR,994367,ASR,2020-05-13 12:39:45.368753,2024-06-25 09:36:13.925015,43.19161,-86.09733,Heights Ravenna Rd,MI-0028,...,2020-AGL-18544-OE,<NA>,<NA>,"5998 Heights Ravenna Rd , Fruitport, MI 49415,...",2020-06-25,No,In development,ASR Operator Mismatch,0.000000,0.7
33658,1139997,ASR,<NA>,ASR,2021-02-22 14:55:46.271143,2024-08-06 08:52:53.298178,39.32000,-85.83764,<NA>,<NA>,...,2019-AGL-6221-OE,<NA>,<NA>,"7600 E 800 N (IN-0026) , Columbus, IN 47201, U...",2020-08-19,No,Active,Focus Asset,NaN,NaN
33659,1139997,ASR,1027096,ASR,2020-05-13 12:39:38.800827,2024-07-30 11:28:33.482098,39.32000,-85.83764,WS St Louis Crossing - C,<NA>,...,2019-AGL-6221-OE,<NA>,<NA>,"7600 E 800 N (IN-0026) , Columbus, IN 47201, U...",2020-08-19,No,Active,ASR Operator Mismatch,0.000000,0.0
33666,1140066,ASR,<NA>,ASR,2020-06-01 08:55:02.975387,2024-08-06 08:53:56.850984,35.19025,-87.04408,Hwy 11,<NA>,...,2018-ASO-12677-OE,<NA>,<NA>,"Hwy 11 , Pulaski, TN 38478, UNITED STATES",2020-12-08,No,Active,Focus Asset,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34744,1147155,Everest Infrastructure Partners,833422,dledbetter,2018-09-20 10:16:32.231523,2018-09-21 14:49:34.622583,36.04094,-79.22089,Buckhhorn,00060,...,2015-ASO-2179-OE,<NA>,<NA>,"2507 Buckhorn Road , Mebane, NC 27302, UNITED ...",2015-05-19,No,Active,Proximity Audit (0m),0.901171,-0.1
34745,1147156,Everest Infrastructure Partners,<NA>,Everest Infrastructure Partners,2025-02-18 16:27:48.056800,2025-02-18 16:27:47.993715,35.40977,-77.98728,Goldsboro,US596436,...,2020-ASO-23207-OE,<NA>,<NA>,"209z Hooks River Rd , Goldsboro, NC 27530, UNI...",NaN,No,Active,Focus Asset,NaN,NaN
34746,1147156,Everest Infrastructure Partners,1137554,sgaither,2015-02-03 23:55:20.061725,2024-06-13 15:43:04.18398,35.40977,-77.98728,Goldsboro,US596436,...,2020-ASO-23207-OE,<NA>,<NA>,"209 Hooks River Rd , GOLDSBORO, NC 27530, UNIT...",2015-04-19,No,Active,Manual Edit,0.000000,-0.1
34749,1147158,Everest Infrastructure Partners,<NA>,Everest Infrastructure Partners,2025-02-18 16:27:57.187729,2025-02-18 16:27:57.124848,34.03211,-86.31711,ALTOONA - DWNTWN - THOMAS,US539594,...,2006-ASO-6406-OE,<NA>,<NA>,"6650 Water Tank Town Road , Altoona, AL 35952,...",NaN,No,Active,Focus Asset,NaN,NaN


In [17]:
updated_case1_prox_audits_post_auto_merge_table

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,faa_study_number,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference
0,942614,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:53:42.284115,2023-01-13 15:25:12.493704,33.94619,-118.28200,CA12409,160407,...,<NA>,<NA>,<NA>,"FIGUEROA ST WL 75F S OF 99TH ST S/N , August F...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN
1,942614,Phoenix Tower International,943927,Phoenix Tower International,2019-08-22 21:36:14.93064,2023-06-26 12:05:50.800659,33.94574,-118.28200,MAIN ST EL 185F N OF 110TH ST SF,161773,...,<NA>,<NA>,<NA>,"MAIN ST EL 185F N OF 110TH ST SF , , CA 90003,...",NaN,No,Unconfirmed,Proximity Audit (48m),49.914635,0.0
2,942664,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:55:21.34445,2023-01-13 11:49:35.586868,34.03929,-118.19900,CA12465,160463,...,<NA>,<NA>,<NA>,"1ST ST SL 2F E OF DACOTAH ST E/W , Los Angeles...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN
3,942664,Phoenix Tower International,942203,Phoenix Tower International,2019-08-22 20:40:05.047291,2023-01-13 13:04:52.922687,34.03910,-118.19900,CA11987,159985,...,<NA>,<NA>,<NA>,"1ST ST SL 53F W OF FRESNO ST WF , Los Angeles,...",NaN,No,Unconfirmed,Proximity Audit (21m),21.075388,0.0
4,942761,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:58:33.921034,2023-01-13 12:41:13.613007,33.98899,-118.31900,CA12570,160568,...,<NA>,<NA>,<NA>,"SLAUSON AVE NL 42F E OF 3RD AVE EF , Los Angel...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64638,1142878,ASR,1142072,ASR,2023-10-03 20:08:53.983413,2024-11-19 17:21:27.702541,38.83462,-76.72942,Moccasin,<NA>,...,2024-AEA-937-OE,<NA>,<NA>,"3610 Old Crain Hwy , Upper Marlboro, MD 20772,...",NaN,No,ASR-Granted,ASR Operator Mismatch,13.575846,0.6
64639,1142878,ASR,<NA>,ASR,2024-11-19 23:11:39.656464,2025-01-28 10:07:53.386231,38.83472,-76.72933,<NA>,<NA>,...,2024-AEA-937-OE,<NA>,<NA>,"CRAIN HIGHWAY , UPPER MARLBORO, MD 20772, UNIT...",NaN,No,ASR-Granted,Focus Asset,NaN,NaN
64640,1146954,Everest Infrastructure Partners,324396,dledbetter,2015-02-03 23:55:20.061725,2018-08-15 19:41:28.859285,41.38514,-79.83375,Franklin #1,00766,...,2000-AEA-1487-OE,<NA>,<NA>,"Gurney Hill , Franklin, PA 16323, UNITED STATES",1980-04-19,No,Active,Proximity Audit (1m),1.390355,1.4
64641,1146954,Everest Infrastructure Partners,1146953,Everest Infrastructure Partners,2025-02-18 16:21:07.530334,2025-02-18 16:21:07.465378,41.38513,-79.83374,101 Gurney Rd 2 of 2,US1030132,...,2000-AEA-1487-OE,<NA>,<NA>,"101 Gurney Rd , Franklin, PA 16323, UNITED STATES",NaN,No,Active,Proximity Audit (0m),0.000000,0.0


In [18]:
case1_post_auto_merge_table

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,faa_study_number,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference
0,1139789,ASR,<NA>,Auto-Merged 12/2025,2021-02-22 14:55:46.271143,2025-12-02 22:19:09,43.19161,-86.09733,Heights Ravenna Rd,MI-0028,...,2023-AGL-21182-OE,<NA>,<NA>,"5998 Heights Ravenna Road (MI-0028) , Fruitpor...",2020-06-25,No,Active,Focus Asset,NaN,NaN
1,1139997,ASR,<NA>,Auto-Merged 12/2025,2021-02-22 14:55:46.271143,2025-12-02 22:19:09,39.32000,-85.83764,WS St Louis Crossing - C,<NA>,...,2025-AGL-7560-OE,<NA>,<NA>,"7600 E 800 N (IN-0026) , Columbus, IN 47201, U...",2020-08-19,No,Active,Focus Asset,NaN,NaN
2,1140066,ASR,<NA>,Auto-Merged 12/2025,2020-06-01 08:55:02.975387,2025-12-02 22:19:09,35.19025,-87.04408,Hwy 11,<NA>,...,2018-ASO-12677-OE,<NA>,<NA>,"Hwy 11 , Pulaski, TN 38478, UNITED STATES",2020-12-08,No,Active,Focus Asset,NaN,NaN
3,1142404,ASR,<NA>,Auto-Merged 12/2025,2024-11-19 23:11:39.656464,2025-12-02 22:19:09,38.96042,-76.21336,Crab Alley Bay,<NA>,...,2024-AEA-2062-OE,<NA>,<NA>,"115 PULLMAN CROSSING ROAD , GRASONVILLE, MD 21...",2025-05-27,No,Active,Focus Asset,NaN,NaN
4,1142405,ASR,<NA>,Auto-Merged 12/2025,2024-11-19 23:11:39.656464,2025-12-02 22:19:09,38.83472,-76.72933,Moccasin,<NA>,...,2024-AEA-937-OE,<NA>,<NA>,"CRAIN HIGHWAY , UPPER MARLBORO, MD 20772, UNIT...",NaN,No,ASR-Granted,Focus Asset,NaN,NaN
5,1146918,Everest Infrastructure Partners,<NA>,Auto-Merged 12/2025,2025-02-18 16:20:10.827803,2025-12-02 22:19:09,42.87342,-73.96573,36 Lolik Road,US727237,...,2018-AEA-351-OE,<NA>,<NA>,"36 Lolik Road , Glenville, NY 12302, UNITED ST...",1977-10-01,No,Active,Focus Asset,NaN,NaN
6,1146922,Everest Infrastructure Partners,<NA>,Auto-Merged 12/2025,2025-02-18 16:20:17.168162,2025-12-02 22:19:09,42.46120,-75.02930,176 Cemetery Hill Rd,US1030133,...,2000-AEA-4199-OE,<NA>,<NA>,"Oneonta Mountain Glenwood Hill , ONEONTA, NY 1...",2006-09-01,No,Active,Focus Asset,NaN,NaN
7,1146940,Everest Infrastructure Partners,<NA>,Auto-Merged 12/2025,2025-02-18 16:20:47.895860,2025-12-02 22:19:09,40.71623,-82.54584,1575 Lexington Ave,US596329,...,2021-AGL-15579-OE,<NA>,<NA>,"1575 Lexington Ave , Mansfield, OH 44904, UNIT...",1980-04-19,No,Active,Focus Asset,NaN,NaN
8,1146953,Everest Infrastructure Partners,<NA>,Auto-Merged 12/2025,2025-02-18 16:21:07.530334,2025-12-02 22:19:09,41.38513,-79.83374,101 Gurney Rd 2 of 2,US1030132,...,2025-AEA-4815-OE,<NA>,<NA>,"101 Gurney Rd , Franklin, PA 16323, UNITED STATES",1980-04-19,No,Active,Focus Asset,NaN,NaN
9,1146995,Everest Infrastructure Partners,<NA>,Auto-Merged 12/2025,2025-02-18 16:21:53.624158,2025-12-02 22:19:09,29.86634,-97.66603,117 Bufkin St,US596481,...,2020-ASW-6119-OE,<NA>,<NA>,"117 Bufkin St , Lockhart, TX 78644, UNITED STATES",1970-06-15,No,Active,Focus Asset,NaN,NaN


In [19]:
case1_raw_post_auto_merge_table

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,faa_study_number,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference
0,1139789,ASR,<NA>,Auto-Merged 12/2025,2021-02-22 14:55:46.271143,2025-12-02 22:19:09,43.19161,-86.09733,Heights Ravenna Rd,MI-0028,...,2023-AGL-21182-OE,NaN,NaN,"5998 Heights Ravenna Road (MI-0028) , Fruitpor...",2020-06-25,No,Active,Focus Asset,NaN,NaN
1,1139789,ASR,994367,ASR,2020-05-13 12:39:45.368753,2024-06-25 09:36:13.925015,43.19161,-86.09733,Heights Ravenna Rd,MI-0028,...,2020-AGL-18544-OE,NaN,NaN,"5998 Heights Ravenna Rd , Fruitport, MI 49415,...",2020-06-25,No,In development,ASR Operator Mismatch,0.0,0.7
2,1139789,ASR,<NA>,ASR,2021-02-22 14:55:46.271143,2024-07-16 10:20:53.166299,43.19161,-86.09733,NaN,NaN,...,2020-AGL-18544-OE,NaN,NaN,"5998 Heights Ravenna Road (MI-0028) , Fruitpor...",2020-06-25,No,Active,Focus Asset,NaN,NaN
3,1139997,ASR,<NA>,Auto-Merged 12/2025,2021-02-22 14:55:46.271143,2025-12-02 22:19:09,39.32000,-85.83764,WS St Louis Crossing - C,NaN,...,2025-AGL-7560-OE,NaN,NaN,"7600 E 800 N (IN-0026) , Columbus, IN 47201, U...",2020-08-19,No,Active,Focus Asset,NaN,NaN
4,1139997,ASR,1027096,ASR,2020-05-13 12:39:38.800827,2024-07-30 11:28:33.482098,39.32000,-85.83764,WS St Louis Crossing - C,NaN,...,2019-AGL-6221-OE,NaN,NaN,"7600 E 800 N (IN-0026) , Columbus, IN 47201, U...",2020-08-19,No,Active,ASR Operator Mismatch,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,1147156,Everest Infrastructure Partners,1137554,sgaither,2015-02-03 23:55:20.061725,2024-06-13 15:43:04.18398,35.40977,-77.98728,Goldsboro,US596436,...,2020-ASO-23207-OE,NaN,NaN,"209 Hooks River Rd , GOLDSBORO, NC 27530, UNIT...",2015-04-19,No,Active,Manual Edit,0.0,-0.1
146,1147156,Everest Infrastructure Partners,<NA>,Everest Infrastructure Partners,2025-02-18 16:27:48.056800,2025-02-18 16:27:47.993715,35.40977,-77.98728,Goldsboro,US596436,...,2020-ASO-23207-OE,NaN,NaN,"209z Hooks River Rd , Goldsboro, NC 27530, UNI...",NaN,No,Active,Focus Asset,NaN,NaN
147,1147158,Everest Infrastructure Partners,<NA>,Auto-Merged 12/2025,2025-02-18 16:27:57.187729,2025-12-02 22:19:09,34.03211,-86.31711,ALTOONA - DWNTWN - THOMAS,US539594,...,2006-ASO-6406-OE,NaN,NaN,"6650 Water Tank Town Road , Altoona, AL 35952,...",2007-07-12,No,Active,Focus Asset,NaN,NaN
148,1147158,Everest Infrastructure Partners,1137620,sgaither,2015-02-03 23:55:20.061725,2024-06-13 19:49:58.120249,34.03211,-86.31711,Altoona - Dwntwn - Thomas,US539594,...,2006-ASO-6406-OE,NaN,NaN,"6650 Water Tank Town Road , Altoona, AL 35952,...",2007-07-12,No,Active,Manual Edit,0.0,0.0


In [20]:
case1_aggregated_final_asset_table

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,faa_study_number,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference
0,1128609,Phoenix Tower International,<NA>,Phoenix Tower International,2023-05-08 20:00:48.103441,2023-05-08 16:00:48.024746,41.05333,-73.53527,555 Main St,US-CT-1255,...,<NA>,<NA>,<NA>,"555 Main St , Stamford, CT 06901, UNITED STATES",NaN,No,Unconfirmed,Focus Asset,NaN,NaN
1,1130226,Phoenix Tower International,<NA>,Phoenix Tower International,2023-05-08 20:30:04.261478,2023-05-08 16:30:04.153621,41.35416,-72.09805,26 Washington St,US-CT-1295,...,<NA>,<NA>,<NA>,"26 Washington St , New London, CT 06320, UNITE...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN
2,1139789,ASR,<NA>,Auto-Merged 12/2025,2021-02-22 14:55:46.271143,2025-12-02 22:19:09,43.19161,-86.09733,Heights Ravenna Rd,MI-0028,...,2023-AGL-21182-OE,<NA>,<NA>,"5998 Heights Ravenna Road (MI-0028) , Fruitpor...",2020-06-25,No,Active,Focus Asset,NaN,NaN
3,1139997,ASR,<NA>,Auto-Merged 12/2025,2021-02-22 14:55:46.271143,2025-12-02 22:19:09,39.32000,-85.83764,WS St Louis Crossing - C,<NA>,...,2025-AGL-7560-OE,<NA>,<NA>,"7600 E 800 N (IN-0026) , Columbus, IN 47201, U...",2020-08-19,No,Active,Focus Asset,NaN,NaN
4,1140066,ASR,<NA>,Auto-Merged 12/2025,2020-06-01 08:55:02.975387,2025-12-02 22:19:09,35.19025,-87.04408,Hwy 11,<NA>,...,2018-ASO-12677-OE,<NA>,<NA>,"Hwy 11 , Pulaski, TN 38478, UNITED STATES",2020-12-08,No,Active,Focus Asset,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72,1142794,ASR,<NA>,ASR,2024-11-19 23:11:39.656464,2025-01-14 10:03:14.401115,38.83472,-76.72933,<NA>,<NA>,...,2024-AEA-937-OE,<NA>,<NA>,"CRAIN HIGHWAY , UPPER MARLBORO, MD 20772, UNIT...",NaN,No,ASR-Granted,Focus Asset,NaN,NaN
73,1142829,ASR,<NA>,ASR,2024-11-19 23:11:39.656464,2025-01-21 10:49:18.717369,38.96042,-76.21336,<NA>,<NA>,...,2024-AEA-2062-OE,<NA>,<NA>,"115 PULLMAN CROSSING ROAD , GRASONVILLE, MD 21...",2025-05-27,No,Active,Focus Asset,NaN,NaN
74,1142830,ASR,<NA>,ASR,2024-11-19 23:11:39.656464,2025-01-21 10:49:19.903689,38.83472,-76.72933,<NA>,<NA>,...,2024-AEA-937-OE,<NA>,<NA>,"CRAIN HIGHWAY , UPPER MARLBORO, MD 20772, UNIT...",NaN,No,ASR-Granted,Focus Asset,NaN,NaN
75,1142877,ASR,<NA>,ASR,2024-11-19 23:11:39.656464,2025-01-28 10:07:52.188695,38.96042,-76.21336,<NA>,<NA>,...,2024-AEA-2062-OE,<NA>,<NA>,"115 PULLMAN CROSSING ROAD , GRASONVILLE, MD 21...",2025-05-27,No,Active,Focus Asset,NaN,NaN


In [21]:
final_case1_prox_audits_post_auto_merge_table

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,faa_study_number,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference
0,942614,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:53:42.284115,2023-01-13 15:25:12.493704,33.94619,-118.28200,CA12409,160407,...,<NA>,<NA>,<NA>,"FIGUEROA ST WL 75F S OF 99TH ST S/N , August F...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN
1,942614,Phoenix Tower International,943927,Phoenix Tower International,2019-08-22 21:36:14.93064,2023-06-26 12:05:50.800659,33.94574,-118.28200,MAIN ST EL 185F N OF 110TH ST SF,161773,...,<NA>,<NA>,<NA>,"MAIN ST EL 185F N OF 110TH ST SF , , CA 90003,...",NaN,No,Unconfirmed,Proximity Audit (48m),49.914635,0.0
2,942664,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:55:21.34445,2023-01-13 11:49:35.586868,34.03929,-118.19900,CA12465,160463,...,<NA>,<NA>,<NA>,"1ST ST SL 2F E OF DACOTAH ST E/W , Los Angeles...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN
3,942664,Phoenix Tower International,942203,Phoenix Tower International,2019-08-22 20:40:05.047291,2023-01-13 13:04:52.922687,34.03910,-118.19900,CA11987,159985,...,<NA>,<NA>,<NA>,"1ST ST SL 53F W OF FRESNO ST WF , Los Angeles,...",NaN,No,Unconfirmed,Proximity Audit (21m),21.075388,0.0
4,942761,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:58:33.921034,2023-01-13 12:41:13.613007,33.98899,-118.31900,CA12570,160568,...,<NA>,<NA>,<NA>,"SLAUSON AVE NL 42F E OF 3RD AVE EF , Los Angel...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64526,1161828,ASR,264885,Crown Castle,2015-04-20 20:26:56.595238,2017-07-28 23:55:23.576292,39.00210,-95.73923,SKYLINE PARK/BURNETT'S MOUND,877829,...,<NA>,<NA>,<NA>,"3511 SW SKYLINE PARKWAY , TOPEKA, KS 66614, UN...",2007-01-12,No,Active,Proximity Audit (7m),7.276909,4.5
64527,1161829,ASR,<NA>,ASR,2025-04-01 15:38:18.798044,2025-04-09 11:29:48.458842,32.81861,-86.61714,<NA>,<NA>,...,2024-ASO-23463-OE,<NA>,<NA>,"Off Logan Road , Clanton, AL 35045, UNITED STATES",2024-11-26,No,Active,Focus Asset,NaN,NaN
64528,1161829,ASR,1136163,ASR,2024-01-23 15:24:06.649527,2024-01-23 09:31:32.771277,32.81861,-86.61714,<NA>,<NA>,...,2023-ASO-27414-OE,<NA>,<NA>,"Off Logan Road , Clanton, AL 35046, UNITED STATES",2024-11-26,No,Active,Proximity Audit (0m),0.000000,-20.4
64529,1162574,ASR,<NA>,ASR,2025-07-24 21:02:34.966713,2025-07-28 09:49:51.734224,42.34739,-91.47869,<NA>,<NA>,...,2024-ACE-6738-OE,<NA>,<NA>,"3048 Hwy 13 IA-5254 , Ryan, IA 52330, UNITED S...",NaN,No,ASR-Granted,Focus Asset,NaN,NaN


In [22]:
final_case1_prox_audits_post_auto_merge_table[final_case1_prox_audits_post_auto_merge_table['source'] == 'Auto-Merged 11/2025']

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,faa_study_number,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference


In [23]:
case1_aggregated_final_asset_table[case1_aggregated_final_asset_table['source'] == 'Auto-Merged 11/2025']

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,faa_study_number,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference


In [24]:
case1_aggregated_final_asset_table.to_csv('case1_aggregated_final_asset_table.csv')
case1_auto_merge_candidates.to_csv('case1_auto_merge_candidates.csv')
case1_post_auto_merge_table.to_csv('case1_post_auto_merge_table.csv')
case1_raw_post_auto_merge_table.to_csv('case1_raw_post_auto_merge_table.csv')
final_case1_prox_audits_post_auto_merge_table.to_csv('final_case1_prox_audits_post_auto_merge_table.csv')
initial_case1_prox_audits_post_auto_merge_table.to_csv('initial_case1_prox_audits_post_auto_merge_table.csv')
updated_case1_prox_audits_post_auto_merge_table.to_csv('updated_case1_prox_audits_post_auto_merge_table.csv')
case1_auto_merge_further_filter.to_csv('case1_auto_merge_further_filter.csv')

---

## Case 2

In [25]:
case2_auto_merge_candidates, initial_case2_prox_audits_post_auto_merge_table = split_case_2_audits(final_case1_prox_audits_post_auto_merge_table)

In [26]:
case2_auto_merge_further_filter, updated_case2_prox_audits_post_auto_merge_table, case2_post_auto_merge_table, case2_raw_post_auto_merge_table = apply_case_2_full_processing(case2_auto_merge_candidates, initial_case2_prox_audits_post_auto_merge_table)

Starting Case 2 processing...


Processing Case 2: 100%|█████████████████████████| 2/2 [00:00<00:00, 489.73it/s]

--- Case 2 Processing completed in: 0.00 minutes ---


In [27]:
case2_aggregated_final_asset_table, final_case2_prox_audits_post_auto_merge_table = apply_case_2_maintenance_logic(prox_audits_table, updated_case2_prox_audits_post_auto_merge_table, case2_post_auto_merge_table, case2_raw_post_auto_merge_table, case1_aggregated_final_asset_table)

In [28]:
case2_auto_merge_candidates

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference,created_at_month
34433,1146953,Everest Infrastructure Partners,<NA>,Auto-Merged 12/2025,2025-02-18 16:21:07.530334,2025-12-02 22:19:09,41.38513,-79.83374,101 Gurney Rd 2 of 2,US1030132,...,<NA>,<NA>,"101 Gurney Rd , Franklin, PA 16323, UNITED STATES",1980-04-19,No,Active,Focus Asset,NaN,NaN,2025-02
34434,1146953,Everest Infrastructure Partners,823488,ASR,2018-04-13 10:12:57.464868,2019-10-16 10:25:59.233356,41.38514,-79.83365,Franklin #2,01321,...,<NA>,<NA>,"End of The Gurney Road , Franklin, PA 16323, U...",1980-04-19,No,Active,Proximity Audit (7m),7.609403,1.4,2018-04
34435,1146954,Everest Infrastructure Partners,<NA>,Everest Infrastructure Partners,2025-02-18 16:21:09.455953,2025-02-18 16:21:09.388917,41.38513,-79.83374,101 Gurney Rd 1 of 2,US596342,...,<NA>,<NA>,"101 Gurney Rd , Franklin, PA 16323, UNITED STATES",NaN,No,Active,Focus Asset,NaN,NaN,2025-02
34436,1146954,Everest Infrastructure Partners,823488,ASR,2018-04-13 10:12:57.464868,2019-10-16 10:25:59.233356,41.38514,-79.83365,Franklin #2,01321,...,<NA>,<NA>,"End of The Gurney Road , Franklin, PA 16323, U...",1980-04-19,No,Active,Proximity Audit (7m),7.609403,1.4,2018-04
34437,1146954,Everest Infrastructure Partners,1146953,Auto-Merged 12/2025,2025-02-18 16:21:07.530334,2025-12-02 22:19:09,41.38513,-79.83374,101 Gurney Rd 2 of 2,US1030132,...,<NA>,<NA>,"101 Gurney Rd , Franklin, PA 16323, UNITED STATES",1980-04-19,No,Active,Proximity Audit (0m),0.000000,0.0,2025-02


In [29]:
initial_case2_prox_audits_post_auto_merge_table

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,faa_study_number,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference
0,942614,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:53:42.284115,2023-01-13 15:25:12.493704,33.94619,-118.28200,CA12409,160407,...,<NA>,<NA>,<NA>,"FIGUEROA ST WL 75F S OF 99TH ST S/N , August F...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN
1,942614,Phoenix Tower International,943927,Phoenix Tower International,2019-08-22 21:36:14.93064,2023-06-26 12:05:50.800659,33.94574,-118.28200,MAIN ST EL 185F N OF 110TH ST SF,161773,...,<NA>,<NA>,<NA>,"MAIN ST EL 185F N OF 110TH ST SF , , CA 90003,...",NaN,No,Unconfirmed,Proximity Audit (48m),49.914635,0.0
2,942664,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:55:21.34445,2023-01-13 11:49:35.586868,34.03929,-118.19900,CA12465,160463,...,<NA>,<NA>,<NA>,"1ST ST SL 2F E OF DACOTAH ST E/W , Los Angeles...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN
3,942664,Phoenix Tower International,942203,Phoenix Tower International,2019-08-22 20:40:05.047291,2023-01-13 13:04:52.922687,34.03910,-118.19900,CA11987,159985,...,<NA>,<NA>,<NA>,"1ST ST SL 53F W OF FRESNO ST WF , Los Angeles,...",NaN,No,Unconfirmed,Proximity Audit (21m),21.075388,0.0
4,942761,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:58:33.921034,2023-01-13 12:41:13.613007,33.98899,-118.31900,CA12570,160568,...,<NA>,<NA>,<NA>,"SLAUSON AVE NL 42F E OF 3RD AVE EF , Los Angel...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64526,1161828,ASR,264885,Crown Castle,2015-04-20 20:26:56.595238,2017-07-28 23:55:23.576292,39.00210,-95.73923,SKYLINE PARK/BURNETT'S MOUND,877829,...,<NA>,<NA>,<NA>,"3511 SW SKYLINE PARKWAY , TOPEKA, KS 66614, UN...",2007-01-12,No,Active,Proximity Audit (7m),7.276909,4.5
64527,1161829,ASR,<NA>,ASR,2025-04-01 15:38:18.798044,2025-04-09 11:29:48.458842,32.81861,-86.61714,<NA>,<NA>,...,2024-ASO-23463-OE,<NA>,<NA>,"Off Logan Road , Clanton, AL 35045, UNITED STATES",2024-11-26,No,Active,Focus Asset,NaN,NaN
64528,1161829,ASR,1136163,ASR,2024-01-23 15:24:06.649527,2024-01-23 09:31:32.771277,32.81861,-86.61714,<NA>,<NA>,...,2023-ASO-27414-OE,<NA>,<NA>,"Off Logan Road , Clanton, AL 35046, UNITED STATES",2024-11-26,No,Active,Proximity Audit (0m),0.000000,-20.4
64529,1162574,ASR,<NA>,ASR,2025-07-24 21:02:34.966713,2025-07-28 09:49:51.734224,42.34739,-91.47869,<NA>,<NA>,...,2024-ACE-6738-OE,<NA>,<NA>,"3048 Hwy 13 IA-5254 , Ryan, IA 52330, UNITED S...",NaN,No,ASR-Granted,Focus Asset,NaN,NaN


In [30]:
case2_auto_merge_further_filter

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,faa_study_number,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference
34435,1146954,Everest Infrastructure Partners,<NA>,Everest Infrastructure Partners,2025-02-18 16:21:09.455953,2025-02-18 16:21:09.388917,41.38513,-79.83374,101 Gurney Rd 1 of 2,US596342,...,2000-AEA-1487-OE,<NA>,<NA>,"101 Gurney Rd , Franklin, PA 16323, UNITED STATES",NaN,No,Active,Focus Asset,NaN,NaN
34436,1146954,Everest Infrastructure Partners,823488,ASR,2018-04-13 10:12:57.464868,2019-10-16 10:25:59.233356,41.38514,-79.83365,Franklin #2,01321,...,00-AEA-1487-OE,<NA>,<NA>,"End of The Gurney Road , Franklin, PA 16323, U...",1980-04-19,No,Active,Proximity Audit (7m),7.609403,1.4


In [31]:
updated_case2_prox_audits_post_auto_merge_table

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,faa_study_number,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference
0,942614,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:53:42.284115,2023-01-13 15:25:12.493704,33.94619,-118.28200,CA12409,160407,...,<NA>,<NA>,<NA>,"FIGUEROA ST WL 75F S OF 99TH ST S/N , August F...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN
1,942614,Phoenix Tower International,943927,Phoenix Tower International,2019-08-22 21:36:14.93064,2023-06-26 12:05:50.800659,33.94574,-118.28200,MAIN ST EL 185F N OF 110TH ST SF,161773,...,<NA>,<NA>,<NA>,"MAIN ST EL 185F N OF 110TH ST SF , , CA 90003,...",NaN,No,Unconfirmed,Proximity Audit (48m),49.914635,0.0
2,942664,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:55:21.34445,2023-01-13 11:49:35.586868,34.03929,-118.19900,CA12465,160463,...,<NA>,<NA>,<NA>,"1ST ST SL 2F E OF DACOTAH ST E/W , Los Angeles...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN
3,942664,Phoenix Tower International,942203,Phoenix Tower International,2019-08-22 20:40:05.047291,2023-01-13 13:04:52.922687,34.03910,-118.19900,CA11987,159985,...,<NA>,<NA>,<NA>,"1ST ST SL 53F W OF FRESNO ST WF , Los Angeles,...",NaN,No,Unconfirmed,Proximity Audit (21m),21.075388,0.0
4,942761,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:58:33.921034,2023-01-13 12:41:13.613007,33.98899,-118.31900,CA12570,160568,...,<NA>,<NA>,<NA>,"SLAUSON AVE NL 42F E OF 3RD AVE EF , Los Angel...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64525,1162574,ASR,1136081,ASR,2024-01-16 15:01:37.007989,2024-01-16 09:05:38.763557,42.34739,-91.47872,<NA>,<NA>,...,2023-ACE-515-OE,<NA>,<NA>,"3074 State Hwy 13 , Ryan, IA 52330, UNITED STATES",NaN,No,ASR-Granted,Proximity Audit (2m),2.471958,-0.1
64526,1146954,Everest Infrastructure Partners,<NA>,Auto-Merged 12/2025,2025-02-18 16:21:09.455953,2025-12-02 22:19:48,41.38513,-79.83374,101 Gurney Rd 1 of 2,US596342,...,2025-AEA-4815-OE,<NA>,<NA>,"101 Gurney Rd , Franklin, PA 16323, UNITED STATES",1980-04-19,No,Active,Focus Asset,NaN,NaN
64527,1146953,Everest Infrastructure Partners,<NA>,Auto-Merged 12/2025,2025-02-18 16:21:07.530334,2025-12-02 22:19:09,41.38513,-79.83374,101 Gurney Rd 2 of 2,US1030132,...,2025-AEA-4815-OE,<NA>,<NA>,"101 Gurney Rd , Franklin, PA 16323, UNITED STATES",1980-04-19,No,Active,Focus Asset,NaN,NaN
64528,1146953,Everest Infrastructure Partners,823488,ASR,2018-04-13 10:12:57.464868,2019-10-16 10:25:59.233356,41.38514,-79.83365,Franklin #2,01321,...,00-AEA-1487-OE,<NA>,<NA>,"End of The Gurney Road , Franklin, PA 16323, U...",1980-04-19,No,Active,Proximity Audit (7m),7.609403,1.4


In [32]:
case2_post_auto_merge_table

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,faa_study_number,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference
0,1146954,Everest Infrastructure Partners,<NA>,Auto-Merged 12/2025,2025-02-18 16:21:09.455953,2025-12-02 22:19:48,41.38513,-79.83374,101 Gurney Rd 1 of 2,US596342,...,2025-AEA-4815-OE,<NA>,<NA>,"101 Gurney Rd , Franklin, PA 16323, UNITED STATES",1980-04-19,No,Active,Focus Asset,NaN,NaN


In [33]:
case2_raw_post_auto_merge_table

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,faa_study_number,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference
0,1146954,Everest Infrastructure Partners,<NA>,Auto-Merged 12/2025,2025-02-18 16:21:09.455953,2025-12-02 22:19:48,41.38513,-79.83374,101 Gurney Rd 1 of 2,US596342,...,2025-AEA-4815-OE,NaN,NaN,"101 Gurney Rd , Franklin, PA 16323, UNITED STATES",1980-04-19,No,Active,Focus Asset,NaN,NaN
1,1146954,Everest Infrastructure Partners,823488,ASR,2018-04-13 10:12:57.464868,2019-10-16 10:25:59.233356,41.38514,-79.83365,Franklin #2,01321,...,00-AEA-1487-OE,NaN,NaN,"End of The Gurney Road , Franklin, PA 16323, U...",1980-04-19,No,Active,Proximity Audit (7m),7.609403,1.4
2,1146954,Everest Infrastructure Partners,<NA>,Everest Infrastructure Partners,2025-02-18 16:21:09.455953,2025-02-18 16:21:09.388917,41.38513,-79.83374,101 Gurney Rd 1 of 2,US596342,...,2000-AEA-1487-OE,NaN,NaN,"101 Gurney Rd , Franklin, PA 16323, UNITED STATES",NaN,No,Active,Focus Asset,NaN,NaN


In [34]:
case2_aggregated_final_asset_table

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,faa_study_number,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference
0,1128609,Phoenix Tower International,<NA>,Phoenix Tower International,2023-05-08 20:00:48.103441,2023-05-08 16:00:48.024746,41.05333,-73.53527,555 Main St,US-CT-1255,...,<NA>,<NA>,<NA>,"555 Main St , Stamford, CT 06901, UNITED STATES",NaN,No,Unconfirmed,Focus Asset,NaN,NaN
1,1130226,Phoenix Tower International,<NA>,Phoenix Tower International,2023-05-08 20:30:04.261478,2023-05-08 16:30:04.153621,41.35416,-72.09805,26 Washington St,US-CT-1295,...,<NA>,<NA>,<NA>,"26 Washington St , New London, CT 06320, UNITE...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN
2,1139789,ASR,<NA>,Auto-Merged 12/2025,2021-02-22 14:55:46.271143,2025-12-02 22:19:09,43.19161,-86.09733,Heights Ravenna Rd,MI-0028,...,2023-AGL-21182-OE,<NA>,<NA>,"5998 Heights Ravenna Road (MI-0028) , Fruitpor...",2020-06-25,No,Active,Focus Asset,NaN,NaN
3,1139997,ASR,<NA>,Auto-Merged 12/2025,2021-02-22 14:55:46.271143,2025-12-02 22:19:09,39.32000,-85.83764,WS St Louis Crossing - C,<NA>,...,2025-AGL-7560-OE,<NA>,<NA>,"7600 E 800 N (IN-0026) , Columbus, IN 47201, U...",2020-08-19,No,Active,Focus Asset,NaN,NaN
4,1140066,ASR,<NA>,Auto-Merged 12/2025,2020-06-01 08:55:02.975387,2025-12-02 22:19:09,35.19025,-87.04408,Hwy 11,<NA>,...,2018-ASO-12677-OE,<NA>,<NA>,"Hwy 11 , Pulaski, TN 38478, UNITED STATES",2020-12-08,No,Active,Focus Asset,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73,1142829,ASR,<NA>,ASR,2024-11-19 23:11:39.656464,2025-01-21 10:49:18.717369,38.96042,-76.21336,<NA>,<NA>,...,2024-AEA-2062-OE,<NA>,<NA>,"115 PULLMAN CROSSING ROAD , GRASONVILLE, MD 21...",2025-05-27,No,Active,Focus Asset,NaN,NaN
74,1142830,ASR,<NA>,ASR,2024-11-19 23:11:39.656464,2025-01-21 10:49:19.903689,38.83472,-76.72933,<NA>,<NA>,...,2024-AEA-937-OE,<NA>,<NA>,"CRAIN HIGHWAY , UPPER MARLBORO, MD 20772, UNIT...",NaN,No,ASR-Granted,Focus Asset,NaN,NaN
75,1142877,ASR,<NA>,ASR,2024-11-19 23:11:39.656464,2025-01-28 10:07:52.188695,38.96042,-76.21336,<NA>,<NA>,...,2024-AEA-2062-OE,<NA>,<NA>,"115 PULLMAN CROSSING ROAD , GRASONVILLE, MD 21...",2025-05-27,No,Active,Focus Asset,NaN,NaN
76,1142878,ASR,<NA>,ASR,2024-11-19 23:11:39.656464,2025-01-28 10:07:53.386231,38.83472,-76.72933,<NA>,<NA>,...,2024-AEA-937-OE,<NA>,<NA>,"CRAIN HIGHWAY , UPPER MARLBORO, MD 20772, UNIT...",NaN,No,ASR-Granted,Focus Asset,NaN,NaN


In [35]:
final_case2_prox_audits_post_auto_merge_table

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,faa_study_number,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference
0,942614,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:53:42.284115,2023-01-13 15:25:12.493704,33.94619,-118.28200,CA12409,160407,...,<NA>,<NA>,<NA>,"FIGUEROA ST WL 75F S OF 99TH ST S/N , August F...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN
1,942614,Phoenix Tower International,943927,Phoenix Tower International,2019-08-22 21:36:14.93064,2023-06-26 12:05:50.800659,33.94574,-118.28200,MAIN ST EL 185F N OF 110TH ST SF,161773,...,<NA>,<NA>,<NA>,"MAIN ST EL 185F N OF 110TH ST SF , , CA 90003,...",NaN,No,Unconfirmed,Proximity Audit (48m),49.914635,0.0
2,942664,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:55:21.34445,2023-01-13 11:49:35.586868,34.03929,-118.19900,CA12465,160463,...,<NA>,<NA>,<NA>,"1ST ST SL 2F E OF DACOTAH ST E/W , Los Angeles...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN
3,942664,Phoenix Tower International,942203,Phoenix Tower International,2019-08-22 20:40:05.047291,2023-01-13 13:04:52.922687,34.03910,-118.19900,CA11987,159985,...,<NA>,<NA>,<NA>,"1ST ST SL 53F W OF FRESNO ST WF , Los Angeles,...",NaN,No,Unconfirmed,Proximity Audit (21m),21.075388,0.0
4,942761,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:58:33.921034,2023-01-13 12:41:13.613007,33.98899,-118.31900,CA12570,160568,...,<NA>,<NA>,<NA>,"SLAUSON AVE NL 42F E OF 3RD AVE EF , Los Angel...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64523,1161828,ASR,264885,Crown Castle,2015-04-20 20:26:56.595238,2017-07-28 23:55:23.576292,39.00210,-95.73923,SKYLINE PARK/BURNETT'S MOUND,877829,...,<NA>,<NA>,<NA>,"3511 SW SKYLINE PARKWAY , TOPEKA, KS 66614, UN...",2007-01-12,No,Active,Proximity Audit (7m),7.276909,4.5
64524,1161829,ASR,<NA>,ASR,2025-04-01 15:38:18.798044,2025-04-09 11:29:48.458842,32.81861,-86.61714,<NA>,<NA>,...,2024-ASO-23463-OE,<NA>,<NA>,"Off Logan Road , Clanton, AL 35045, UNITED STATES",2024-11-26,No,Active,Focus Asset,NaN,NaN
64525,1161829,ASR,1136163,ASR,2024-01-23 15:24:06.649527,2024-01-23 09:31:32.771277,32.81861,-86.61714,<NA>,<NA>,...,2023-ASO-27414-OE,<NA>,<NA>,"Off Logan Road , Clanton, AL 35046, UNITED STATES",2024-11-26,No,Active,Proximity Audit (0m),0.000000,-20.4
64526,1162574,ASR,<NA>,ASR,2025-07-24 21:02:34.966713,2025-07-28 09:49:51.734224,42.34739,-91.47869,<NA>,<NA>,...,2024-ACE-6738-OE,<NA>,<NA>,"3048 Hwy 13 IA-5254 , Ryan, IA 52330, UNITED S...",NaN,No,ASR-Granted,Focus Asset,NaN,NaN


In [36]:
case2_aggregated_final_asset_table[case2_aggregated_final_asset_table['source'] == 'Auto-Merged 11/2025']

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,faa_study_number,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference


In [37]:
final_case2_prox_audits_post_auto_merge_table[final_case2_prox_audits_post_auto_merge_table['source'] == 'Auto-Merged 11/2025']

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,faa_study_number,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference


In [38]:
case2_aggregated_final_asset_table.to_csv('case2_aggregated_final_asset_table.csv')
case2_auto_merge_candidates.to_csv('case2_auto_merge_candidates.csv')
case2_post_auto_merge_table.to_csv('case2_post_auto_merge_table.csv')
case2_raw_post_auto_merge_table.to_csv('case2_raw_post_auto_merge_table.csv')
final_case2_prox_audits_post_auto_merge_table.to_csv('final_case2_prox_audits_post_auto_merge_table.csv')
initial_case2_prox_audits_post_auto_merge_table.to_csv('initial_case2_prox_audits_post_auto_merge_table.csv')
updated_case2_prox_audits_post_auto_merge_table.to_csv('updated_case2_prox_audits_post_auto_merge_table.csv')
case2_auto_merge_further_filter.to_csv('case2_auto_merge_further_filter.csv')

---

## Case 3

In [39]:
case3_auto_merge_candidates, initial_case3_prox_audits_post_auto_merge_table = split_case_3_audits(final_case2_prox_audits_post_auto_merge_table)

In [40]:
case3_auto_merge_further_filter, updated_case3_prox_audits_post_auto_merge_table, case3_post_auto_merge_table, case3_raw_post_auto_merge_table = apply_case_3_full_processing(case3_auto_merge_candidates, initial_case3_prox_audits_post_auto_merge_table)

Starting Case 3 processing...


Processing Case 3: 100%|██████████████████████████| 3/3 [00:02<00:00,  1.45it/s]

--- Case 3 Processing completed in: 0.04 minutes ---


In [41]:
case3_aggregated_final_asset_table, final_case3_prox_audits_post_auto_merge_table = apply_case_3_maintenance_logic(prox_audits_table, updated_case3_prox_audits_post_auto_merge_table, case3_post_auto_merge_table, case3_raw_post_auto_merge_table, case2_aggregated_final_asset_table)

In [42]:
case3_auto_merge_candidates

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference,created_at_month
34432,1146945,Everest Infrastructure Partners,329248,dledbetter,2015-02-03 23:55:20.061725,2018-09-21 18:48:54.285743,41.56339,-84.13133,Waueson,01016,...,<NA>,<NA>,"1302 North Shoop Avenue , Wauseon, OH 43567, U...",1965-01-01,No,Active,Proximity Audit (32m),32.197440,-0.1,2015-02
34435,1147003,Everest Infrastructure Partners,<NA>,Everest Infrastructure Partners,2025-02-18 16:22:01.924127,2025-02-18 16:22:01.857562,33.93656,-98.53389,3225 Maurine St,US596472,...,<NA>,<NA>,"3225 Maurine St , WICHITA FALLS, TX 76306, UNI...",NaN,No,Active,Focus Asset,NaN,NaN,2025-02
34436,1147003,Everest Infrastructure Partners,327206,dledbetter,2015-02-03 23:55:20.061725,2018-09-21 19:48:20.430233,33.93653,-98.53391,Maurine Street,00583,...,<NA>,<NA>,"3225 Maurine Street , Wichita Falls, TX 76305,...",1979-10-04,No,Active,Proximity Audit (3m),3.806865,-0.1,2015-02
34503,1147109,Everest Infrastructure Partners,<NA>,Everest Infrastructure Partners,2025-02-18 16:25:50.136163,2025-02-18 16:25:50.074986,33.97196,-84.50239,Blackjack Mountain 2,US701324,...,<NA>,<NA>,"1480 Barnes Mill Road , Marietta, GA 30062, UN...",NaN,No,Dismantled,Focus Asset,NaN,NaN,2025-02
34504,1147109,Everest Infrastructure Partners,1136825,sgaither,2015-02-03 23:55:20.061725,2024-04-30 22:05:28.468952,33.97196,-84.50239,Blackjack Mountain 2,US701324,...,<NA>,Southeast,"1480 Barnes Mill Road , Marietta, GA 30062, UN...",1987-02-01,No,Dismantled,Manual Edit,0.000000,0.0,2015-02
34431,1146945,Everest Infrastructure Partners,<NA>,Everest Infrastructure Partners,2025-02-18 16:20:56.672347,2025-02-18 16:20:56.605248,41.56311,-84.13123,1152 N Shoop Ave,US596309,...,<NA>,<NA>,"1152 N Shoop Ave , Wauseon, OH 43567, UNITED S...",NaN,No,Active,Focus Asset,NaN,NaN,2025-02


In [43]:
initial_case3_prox_audits_post_auto_merge_table

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,faa_study_number,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference
0,942614,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:53:42.284115,2023-01-13 15:25:12.493704,33.94619,-118.28200,CA12409,160407,...,<NA>,<NA>,<NA>,"FIGUEROA ST WL 75F S OF 99TH ST S/N , August F...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN
1,942614,Phoenix Tower International,943927,Phoenix Tower International,2019-08-22 21:36:14.93064,2023-06-26 12:05:50.800659,33.94574,-118.28200,MAIN ST EL 185F N OF 110TH ST SF,161773,...,<NA>,<NA>,<NA>,"MAIN ST EL 185F N OF 110TH ST SF , , CA 90003,...",NaN,No,Unconfirmed,Proximity Audit (48m),49.914635,0.0
2,942664,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:55:21.34445,2023-01-13 11:49:35.586868,34.03929,-118.19900,CA12465,160463,...,<NA>,<NA>,<NA>,"1ST ST SL 2F E OF DACOTAH ST E/W , Los Angeles...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN
3,942664,Phoenix Tower International,942203,Phoenix Tower International,2019-08-22 20:40:05.047291,2023-01-13 13:04:52.922687,34.03910,-118.19900,CA11987,159985,...,<NA>,<NA>,<NA>,"1ST ST SL 53F W OF FRESNO ST WF , Los Angeles,...",NaN,No,Unconfirmed,Proximity Audit (21m),21.075388,0.0
4,942761,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:58:33.921034,2023-01-13 12:41:13.613007,33.98899,-118.31900,CA12570,160568,...,<NA>,<NA>,<NA>,"SLAUSON AVE NL 42F E OF 3RD AVE EF , Los Angel...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64523,1161828,ASR,264885,Crown Castle,2015-04-20 20:26:56.595238,2017-07-28 23:55:23.576292,39.00210,-95.73923,SKYLINE PARK/BURNETT'S MOUND,877829,...,<NA>,<NA>,<NA>,"3511 SW SKYLINE PARKWAY , TOPEKA, KS 66614, UN...",2007-01-12,No,Active,Proximity Audit (7m),7.276909,4.5
64524,1161829,ASR,<NA>,ASR,2025-04-01 15:38:18.798044,2025-04-09 11:29:48.458842,32.81861,-86.61714,<NA>,<NA>,...,2024-ASO-23463-OE,<NA>,<NA>,"Off Logan Road , Clanton, AL 35045, UNITED STATES",2024-11-26,No,Active,Focus Asset,NaN,NaN
64525,1161829,ASR,1136163,ASR,2024-01-23 15:24:06.649527,2024-01-23 09:31:32.771277,32.81861,-86.61714,<NA>,<NA>,...,2023-ASO-27414-OE,<NA>,<NA>,"Off Logan Road , Clanton, AL 35046, UNITED STATES",2024-11-26,No,Active,Proximity Audit (0m),0.000000,-20.4
64526,1162574,ASR,<NA>,ASR,2025-07-24 21:02:34.966713,2025-07-28 09:49:51.734224,42.34739,-91.47869,<NA>,<NA>,...,2024-ACE-6738-OE,<NA>,<NA>,"3048 Hwy 13 IA-5254 , Ryan, IA 52330, UNITED S...",NaN,No,ASR-Granted,Focus Asset,NaN,NaN


In [44]:
case3_auto_merge_further_filter

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,faa_study_number,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference
34431,1146945,Everest Infrastructure Partners,<NA>,Everest Infrastructure Partners,2025-02-18 16:20:56.672347,2025-02-18 16:20:56.605248,41.56311,-84.13123,1152 N Shoop Ave,US596309,...,1968-CLE-342-OE,<NA>,<NA>,"1152 N Shoop Ave , Wauseon, OH 43567, UNITED S...",NaN,No,Active,Focus Asset,NaN,NaN
34432,1146945,Everest Infrastructure Partners,329248,dledbetter,2015-02-03 23:55:20.061725,2018-09-21 18:48:54.285743,41.56339,-84.13133,Waueson,01016,...,<NA>,<NA>,<NA>,"1302 North Shoop Avenue , Wauseon, OH 43567, U...",1965-01-01,No,Active,Proximity Audit (32m),32.197440,-0.1
34435,1147003,Everest Infrastructure Partners,<NA>,Everest Infrastructure Partners,2025-02-18 16:22:01.924127,2025-02-18 16:22:01.857562,33.93656,-98.53389,3225 Maurine St,US596472,...,2015-ASW-7170-OE,<NA>,<NA>,"3225 Maurine St , WICHITA FALLS, TX 76306, UNI...",NaN,No,Active,Focus Asset,NaN,NaN
34436,1147003,Everest Infrastructure Partners,327206,dledbetter,2015-02-03 23:55:20.061725,2018-09-21 19:48:20.430233,33.93653,-98.53391,Maurine Street,00583,...,<NA>,<NA>,<NA>,"3225 Maurine Street , Wichita Falls, TX 76305,...",1979-10-04,No,Active,Proximity Audit (3m),3.806865,-0.1
34503,1147109,Everest Infrastructure Partners,<NA>,Everest Infrastructure Partners,2025-02-18 16:25:50.136163,2025-02-18 16:25:50.074986,33.97196,-84.50239,Blackjack Mountain 2,US701324,...,<NA>,<NA>,<NA>,"1480 Barnes Mill Road , Marietta, GA 30062, UN...",NaN,No,Dismantled,Focus Asset,NaN,NaN
34504,1147109,Everest Infrastructure Partners,1136825,sgaither,2015-02-03 23:55:20.061725,2024-04-30 22:05:28.468952,33.97196,-84.50239,Blackjack Mountain 2,US701324,...,2002-ASO-293-OE,<NA>,Southeast,"1480 Barnes Mill Road , Marietta, GA 30062, UN...",1987-02-01,No,Dismantled,Manual Edit,0.000000,0.0


In [45]:
updated_case3_prox_audits_post_auto_merge_table

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,faa_study_number,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference
0,942614,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:53:42.284115,2023-01-13 15:25:12.493704,33.94619,-118.28200,CA12409,160407,...,<NA>,<NA>,<NA>,"FIGUEROA ST WL 75F S OF 99TH ST S/N , August F...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN
1,942614,Phoenix Tower International,943927,Phoenix Tower International,2019-08-22 21:36:14.93064,2023-06-26 12:05:50.800659,33.94574,-118.28200,MAIN ST EL 185F N OF 110TH ST SF,161773,...,<NA>,<NA>,<NA>,"MAIN ST EL 185F N OF 110TH ST SF , , CA 90003,...",NaN,No,Unconfirmed,Proximity Audit (48m),49.914635,0.0
2,942664,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:55:21.34445,2023-01-13 11:49:35.586868,34.03929,-118.19900,CA12465,160463,...,<NA>,<NA>,<NA>,"1ST ST SL 2F E OF DACOTAH ST E/W , Los Angeles...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN
3,942664,Phoenix Tower International,942203,Phoenix Tower International,2019-08-22 20:40:05.047291,2023-01-13 13:04:52.922687,34.03910,-118.19900,CA11987,159985,...,<NA>,<NA>,<NA>,"1ST ST SL 53F W OF FRESNO ST WF , Los Angeles,...",NaN,No,Unconfirmed,Proximity Audit (21m),21.075388,0.0
4,942761,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:58:33.921034,2023-01-13 12:41:13.613007,33.98899,-118.31900,CA12570,160568,...,<NA>,<NA>,<NA>,"SLAUSON AVE NL 42F E OF 3RD AVE EF , Los Angel...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64520,1162574,ASR,<NA>,ASR,2025-07-24 21:02:34.966713,2025-07-28 09:49:51.734224,42.34739,-91.47869,<NA>,<NA>,...,2024-ACE-6738-OE,<NA>,<NA>,"3048 Hwy 13 IA-5254 , Ryan, IA 52330, UNITED S...",NaN,No,ASR-Granted,Focus Asset,NaN,NaN
64521,1162574,ASR,1136081,ASR,2024-01-16 15:01:37.007989,2024-01-16 09:05:38.763557,42.34739,-91.47872,<NA>,<NA>,...,2023-ACE-515-OE,<NA>,<NA>,"3074 State Hwy 13 , Ryan, IA 52330, UNITED STATES",NaN,No,ASR-Granted,Proximity Audit (2m),2.471958,-0.1
64522,1146945,Everest Infrastructure Partners,<NA>,Auto-Merged 12/2025,2025-02-18 16:20:56.672347,2025-12-02 22:19:57,41.56311,-84.13123,1152 N Shoop Ave,US596309,...,2015-AGL-13257-OE,<NA>,<NA>,"1152 N Shoop Ave , Wauseon, OH 43567, UNITED S...",1965-01-01,No,Active,Focus Asset,NaN,NaN
64523,1147003,Everest Infrastructure Partners,<NA>,Auto-Merged 12/2025,2025-02-18 16:22:01.924127,2025-12-02 22:19:57,33.93656,-98.53389,3225 Maurine St,US596472,...,2015-ASW-7170-OE,<NA>,<NA>,"3225 Maurine St , WICHITA FALLS, TX 76306, UNI...",1979-10-04,No,Active,Focus Asset,NaN,NaN


In [46]:
case3_post_auto_merge_table

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,faa_study_number,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference
0,1146945,Everest Infrastructure Partners,<NA>,Auto-Merged 12/2025,2025-02-18 16:20:56.672347,2025-12-02 22:19:57,41.56311,-84.13123,1152 N Shoop Ave,US596309,...,2015-AGL-13257-OE,<NA>,<NA>,"1152 N Shoop Ave , Wauseon, OH 43567, UNITED S...",1965-01-01,No,Active,Focus Asset,NaN,NaN
1,1147003,Everest Infrastructure Partners,<NA>,Auto-Merged 12/2025,2025-02-18 16:22:01.924127,2025-12-02 22:19:57,33.93656,-98.53389,3225 Maurine St,US596472,...,2015-ASW-7170-OE,<NA>,<NA>,"3225 Maurine St , WICHITA FALLS, TX 76306, UNI...",1979-10-04,No,Active,Focus Asset,NaN,NaN
2,1147109,Everest Infrastructure Partners,<NA>,Auto-Merged 12/2025,2025-02-18 16:25:50.136163,2025-12-02 22:19:57,33.97196,-84.50239,Blackjack Mountain 2,US701324,...,2002-ASO-293-OE,<NA>,Southeast,"1480 Barnes Mill Road , Marietta, GA 30062, UN...",1987-02-01,No,Active,Focus Asset,NaN,NaN


In [47]:
case3_raw_post_auto_merge_table

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,faa_study_number,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference
0,1146945,Everest Infrastructure Partners,<NA>,Auto-Merged 12/2025,2025-02-18 16:20:56.672347,2025-12-02 22:19:57,41.56311,-84.13123,1152 N Shoop Ave,US596309,...,2015-AGL-13257-OE,NaN,NaN,"1152 N Shoop Ave , Wauseon, OH 43567, UNITED S...",1965-01-01,No,Active,Focus Asset,NaN,NaN
1,1146945,Everest Infrastructure Partners,329248,dledbetter,2015-02-03 23:55:20.061725,2018-09-21 18:48:54.285743,41.56339,-84.13133,Waueson,01016,...,<NA>,NaN,NaN,"1302 North Shoop Avenue , Wauseon, OH 43567, U...",1965-01-01,No,Active,Proximity Audit (32m),32.197440,-0.1
2,1146945,Everest Infrastructure Partners,<NA>,Everest Infrastructure Partners,2025-02-18 16:20:56.672347,2025-02-18 16:20:56.605248,41.56311,-84.13123,1152 N Shoop Ave,US596309,...,1968-CLE-342-OE,NaN,NaN,"1152 N Shoop Ave , Wauseon, OH 43567, UNITED S...",NaN,No,Active,Focus Asset,NaN,NaN
3,1147003,Everest Infrastructure Partners,<NA>,Auto-Merged 12/2025,2025-02-18 16:22:01.924127,2025-12-02 22:19:57,33.93656,-98.53389,3225 Maurine St,US596472,...,2015-ASW-7170-OE,NaN,NaN,"3225 Maurine St , WICHITA FALLS, TX 76306, UNI...",1979-10-04,No,Active,Focus Asset,NaN,NaN
4,1147003,Everest Infrastructure Partners,327206,dledbetter,2015-02-03 23:55:20.061725,2018-09-21 19:48:20.430233,33.93653,-98.53391,Maurine Street,00583,...,<NA>,NaN,NaN,"3225 Maurine Street , Wichita Falls, TX 76305,...",1979-10-04,No,Active,Proximity Audit (3m),3.806865,-0.1
5,1147003,Everest Infrastructure Partners,<NA>,Everest Infrastructure Partners,2025-02-18 16:22:01.924127,2025-02-18 16:22:01.857562,33.93656,-98.53389,3225 Maurine St,US596472,...,2015-ASW-7170-OE,NaN,NaN,"3225 Maurine St , WICHITA FALLS, TX 76306, UNI...",NaN,No,Active,Focus Asset,NaN,NaN
6,1147109,Everest Infrastructure Partners,<NA>,Auto-Merged 12/2025,2025-02-18 16:25:50.136163,2025-12-02 22:19:57,33.97196,-84.50239,Blackjack Mountain 2,US701324,...,2002-ASO-293-OE,NaN,Southeast,"1480 Barnes Mill Road , Marietta, GA 30062, UN...",1987-02-01,No,Active,Focus Asset,NaN,NaN
7,1147109,Everest Infrastructure Partners,1136825,sgaither,2015-02-03 23:55:20.061725,2024-04-30 22:05:28.468952,33.97196,-84.50239,Blackjack Mountain 2,US701324,...,2002-ASO-293-OE,NaN,Southeast,"1480 Barnes Mill Road , Marietta, GA 30062, UN...",1987-02-01,No,Dismantled,Manual Edit,0.000000,0.0
8,1147109,Everest Infrastructure Partners,<NA>,Everest Infrastructure Partners,2025-02-18 16:25:50.136163,2025-02-18 16:25:50.074986,33.97196,-84.50239,Blackjack Mountain 2,US701324,...,<NA>,NaN,NaN,"1480 Barnes Mill Road , Marietta, GA 30062, UN...",NaN,No,Dismantled,Focus Asset,NaN,NaN


In [48]:
case3_aggregated_final_asset_table

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,faa_study_number,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference
0,1128609,Phoenix Tower International,<NA>,Phoenix Tower International,2023-05-08 20:00:48.103441,2023-05-08 16:00:48.024746,41.05333,-73.53527,555 Main St,US-CT-1255,...,<NA>,<NA>,NaN,"555 Main St , Stamford, CT 06901, UNITED STATES",NaN,No,Unconfirmed,Focus Asset,NaN,NaN
1,1130226,Phoenix Tower International,<NA>,Phoenix Tower International,2023-05-08 20:30:04.261478,2023-05-08 16:30:04.153621,41.35416,-72.09805,26 Washington St,US-CT-1295,...,<NA>,<NA>,NaN,"26 Washington St , New London, CT 06320, UNITE...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN
2,1139789,ASR,<NA>,Auto-Merged 12/2025,2021-02-22 14:55:46.271143,2025-12-02 22:19:09,43.19161,-86.09733,Heights Ravenna Rd,MI-0028,...,2023-AGL-21182-OE,<NA>,NaN,"5998 Heights Ravenna Road (MI-0028) , Fruitpor...",2020-06-25,No,Active,Focus Asset,NaN,NaN
3,1139997,ASR,<NA>,Auto-Merged 12/2025,2021-02-22 14:55:46.271143,2025-12-02 22:19:09,39.32000,-85.83764,WS St Louis Crossing - C,<NA>,...,2025-AGL-7560-OE,<NA>,NaN,"7600 E 800 N (IN-0026) , Columbus, IN 47201, U...",2020-08-19,No,Active,Focus Asset,NaN,NaN
4,1140066,ASR,<NA>,Auto-Merged 12/2025,2020-06-01 08:55:02.975387,2025-12-02 22:19:09,35.19025,-87.04408,Hwy 11,<NA>,...,2018-ASO-12677-OE,<NA>,NaN,"Hwy 11 , Pulaski, TN 38478, UNITED STATES",2020-12-08,No,Active,Focus Asset,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76,1142878,ASR,<NA>,ASR,2024-11-19 23:11:39.656464,2025-01-28 10:07:53.386231,38.83472,-76.72933,<NA>,<NA>,...,2024-AEA-937-OE,<NA>,NaN,"CRAIN HIGHWAY , UPPER MARLBORO, MD 20772, UNIT...",NaN,No,ASR-Granted,Focus Asset,NaN,NaN
77,1146953,Everest Infrastructure Partners,<NA>,Auto-Merged 12/2025,2025-02-18 16:21:07.530334,2025-12-02 22:19:09,41.38513,-79.83374,101 Gurney Rd 2 of 2,US1030132,...,2025-AEA-4815-OE,<NA>,NaN,"101 Gurney Rd , Franklin, PA 16323, UNITED STATES",1980-04-19,No,Active,Focus Asset,NaN,NaN
78,1146945,Everest Infrastructure Partners,<NA>,Auto-Merged 12/2025,2025-02-18 16:20:56.672347,2025-12-02 22:19:57,41.56311,-84.13123,1152 N Shoop Ave,US596309,...,2015-AGL-13257-OE,<NA>,<NA>,"1152 N Shoop Ave , Wauseon, OH 43567, UNITED S...",1965-01-01,No,Active,Focus Asset,NaN,NaN
79,1147003,Everest Infrastructure Partners,<NA>,Auto-Merged 12/2025,2025-02-18 16:22:01.924127,2025-12-02 22:19:57,33.93656,-98.53389,3225 Maurine St,US596472,...,2015-ASW-7170-OE,<NA>,<NA>,"3225 Maurine St , WICHITA FALLS, TX 76306, UNI...",1979-10-04,No,Active,Focus Asset,NaN,NaN


In [49]:
final_case3_prox_audits_post_auto_merge_table

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,faa_study_number,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference
0,942614,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:53:42.284115,2023-01-13 15:25:12.493704,33.94619,-118.28200,CA12409,160407,...,<NA>,<NA>,<NA>,"FIGUEROA ST WL 75F S OF 99TH ST S/N , August F...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN
1,942614,Phoenix Tower International,943927,Phoenix Tower International,2019-08-22 21:36:14.93064,2023-06-26 12:05:50.800659,33.94574,-118.28200,MAIN ST EL 185F N OF 110TH ST SF,161773,...,<NA>,<NA>,<NA>,"MAIN ST EL 185F N OF 110TH ST SF , , CA 90003,...",NaN,No,Unconfirmed,Proximity Audit (48m),49.914635,0.0
2,942664,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:55:21.34445,2023-01-13 11:49:35.586868,34.03929,-118.19900,CA12465,160463,...,<NA>,<NA>,<NA>,"1ST ST SL 2F E OF DACOTAH ST E/W , Los Angeles...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN
3,942664,Phoenix Tower International,942203,Phoenix Tower International,2019-08-22 20:40:05.047291,2023-01-13 13:04:52.922687,34.03910,-118.19900,CA11987,159985,...,<NA>,<NA>,<NA>,"1ST ST SL 53F W OF FRESNO ST WF , Los Angeles,...",NaN,No,Unconfirmed,Proximity Audit (21m),21.075388,0.0
4,942761,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:58:33.921034,2023-01-13 12:41:13.613007,33.98899,-118.31900,CA12570,160568,...,<NA>,<NA>,<NA>,"SLAUSON AVE NL 42F E OF 3RD AVE EF , Los Angel...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64517,1161828,ASR,264885,Crown Castle,2015-04-20 20:26:56.595238,2017-07-28 23:55:23.576292,39.00210,-95.73923,SKYLINE PARK/BURNETT'S MOUND,877829,...,<NA>,<NA>,<NA>,"3511 SW SKYLINE PARKWAY , TOPEKA, KS 66614, UN...",2007-01-12,No,Active,Proximity Audit (7m),7.276909,4.5
64518,1161829,ASR,<NA>,ASR,2025-04-01 15:38:18.798044,2025-04-09 11:29:48.458842,32.81861,-86.61714,<NA>,<NA>,...,2024-ASO-23463-OE,<NA>,<NA>,"Off Logan Road , Clanton, AL 35045, UNITED STATES",2024-11-26,No,Active,Focus Asset,NaN,NaN
64519,1161829,ASR,1136163,ASR,2024-01-23 15:24:06.649527,2024-01-23 09:31:32.771277,32.81861,-86.61714,<NA>,<NA>,...,2023-ASO-27414-OE,<NA>,<NA>,"Off Logan Road , Clanton, AL 35046, UNITED STATES",2024-11-26,No,Active,Proximity Audit (0m),0.000000,-20.4
64520,1162574,ASR,<NA>,ASR,2025-07-24 21:02:34.966713,2025-07-28 09:49:51.734224,42.34739,-91.47869,<NA>,<NA>,...,2024-ACE-6738-OE,<NA>,<NA>,"3048 Hwy 13 IA-5254 , Ryan, IA 52330, UNITED S...",NaN,No,ASR-Granted,Focus Asset,NaN,NaN


In [50]:
case3_aggregated_final_asset_table[case3_aggregated_final_asset_table['source'] == 'Auto-Merged 11/2025']

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,faa_study_number,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference


In [51]:
final_case3_prox_audits_post_auto_merge_table[final_case3_prox_audits_post_auto_merge_table['source'] == 'Auto-Merged 11/2025']

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,faa_study_number,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference


In [52]:
case3_aggregated_final_asset_table.to_csv('case3_aggregated_final_asset_table.csv')
case3_auto_merge_candidates.to_csv('case3_auto_merge_candidates.csv')
case3_post_auto_merge_table.to_csv('case3_post_auto_merge_table.csv')
case3_raw_post_auto_merge_table.to_csv('case3_raw_post_auto_merge_table.csv')
final_case3_prox_audits_post_auto_merge_table.to_csv('final_case3_prox_audits_post_auto_merge_table.csv')
initial_case3_prox_audits_post_auto_merge_table.to_csv('initial_case3_prox_audits_post_auto_merge_table.csv')
updated_case3_prox_audits_post_auto_merge_table.to_csv('updated_case3_prox_audits_post_auto_merge_table.csv')
case3_auto_merge_further_filter.to_csv('case3_auto_merge_further_filter.csv')

---

## Case 4

In [53]:
case4_auto_merge_candidates, initial_case4_prox_audits_post_auto_merge_table = split_case_4_audits(final_case3_prox_audits_post_auto_merge_table)

In [54]:
case4_auto_merge_further_filter, updated_case4_prox_audits_post_auto_merge_table, case4_post_auto_merge_table, case4_raw_post_auto_merge_table = apply_case_4_full_processing(case4_auto_merge_candidates, initial_case4_prox_audits_post_auto_merge_table)

Starting Case 4 processing...


Processing Case 4: 100%|██████████████████| 7841/7841 [1:07:01<00:00,  1.95it/s]


--- Case 4 Processing completed in: 67.13 minutes ---


In [55]:
case4_aggregated_final_asset_table, final_case4_prox_audits_post_auto_merge_table = apply_case_4_maintenance_logic(prox_audits_table, updated_case4_prox_audits_post_auto_merge_table, case4_post_auto_merge_table, case4_raw_post_auto_merge_table, case3_aggregated_final_asset_table)

In [56]:
case4_auto_merge_candidates

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference,created_at_month
34580,1147159,SBA,<NA>,SBA,2025-02-19 20:47:14.736408,2025-02-19 20:47:14.612784,61.11381,-149.86151,Canine Castle,AK12386-A,...,<NA>,<NA>,"11801 Old Seward Hwy , Anchorage, AK 99515, UN...",NaN,No,Active,Focus Asset,NaN,NaN,2025-02
34581,1147159,SBA,117947,dledbetter,2015-02-03 20:22:46.310560,2019-07-30 10:15:20.762731,61.11382,-149.86151,Canine Castle,AK12386-A,...,<NA>,<NA>,", Anchorage, AK 99515, UNITED STATES",2009-05-06,No,Active,Proximity Audit (1m),1.114310,0.0,2015-02
34582,1147160,SBA,<NA>,SBA,2025-02-19 20:47:16.835983,2025-02-19 20:47:16.766783,61.16028,-149.85808,Brayton Pax Soto,AK12387-A,...,<NA>,<NA>,"6646 Homer Drive , Anchorage, AK 99518, UNITED...",NaN,No,Active,Focus Asset,NaN,NaN,2025-02
34583,1147160,SBA,117948,dledbetter,2015-02-03 20:22:46.346536,2019-07-30 10:15:20.762731,61.16028,-149.85809,Brayton Pax Soto,AK12387-A,...,<NA>,<NA>,", Anchorage, AK 99518, UNITED STATES",2009-05-06,No,Active,Proximity Audit (0m),0.538346,0.0,2015-02
34584,1147161,SBA,<NA>,SBA,2025-02-19 20:47:18.928215,2025-02-19 20:47:18.862005,61.13868,-149.84196,Tikigaq,AK12388-A,...,<NA>,<NA>,"2121 Abbott Road , Anchorage, AK 99507, UNITED...",NaN,No,Active,Focus Asset,NaN,NaN,2025-02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64466,1161552,SBA,118020,SBA,2015-02-03 20:22:46.431704,2017-11-08 13:04:11.713717,33.75752,-85.78392,Hamiltons Lake,AL04110-B,...,<NA>,<NA>,", Anniston, AL 36206-7568, UNITED STATES",2000-08-03,No,Active,Proximity Audit (1m),1.109180,-0.1,2015-02
64469,1161554,SBA,<NA>,SBA,2025-02-20 05:42:05.608936,2025-02-20 05:42:05.545048,34.11235,-88.02328,West Hamilton,AL04756-A,...,<NA>,<NA>,"596 Fire Tower Road , Hamilton, AL 35570-9721,...",NaN,No,Active,Focus Asset,NaN,NaN,2025-02
64470,1161554,SBA,118022,SBA,2015-02-03 20:22:46.433772,2017-11-08 13:03:54.697111,34.11235,-88.02329,West Hamilton,AL04756-A,...,<NA>,<NA>,", Hamilton, AL 35570-9721, UNITED STATES",2000-06-24,No,Active,Proximity Audit (0m),0.922630,-3.1,2015-02
64474,1161556,SBA,<NA>,SBA,2025-02-20 05:42:15.236684,2025-02-20 05:42:15.173002,34.58570,-86.45280,Hamptons Cove,AL04832-B,...,<NA>,<NA>,"238 Wilson Mann Road , Owens Cross Roads, AL 3...",NaN,No,Active,Focus Asset,NaN,NaN,2025-02


In [57]:
initial_case4_prox_audits_post_auto_merge_table

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,faa_study_number,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference
0,942614,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:53:42.284115,2023-01-13 15:25:12.493704,33.94619,-118.28200,CA12409,160407,...,<NA>,<NA>,<NA>,"FIGUEROA ST WL 75F S OF 99TH ST S/N , August F...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN
1,942614,Phoenix Tower International,943927,Phoenix Tower International,2019-08-22 21:36:14.93064,2023-06-26 12:05:50.800659,33.94574,-118.28200,MAIN ST EL 185F N OF 110TH ST SF,161773,...,<NA>,<NA>,<NA>,"MAIN ST EL 185F N OF 110TH ST SF , , CA 90003,...",NaN,No,Unconfirmed,Proximity Audit (48m),49.914635,0.0
2,942664,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:55:21.34445,2023-01-13 11:49:35.586868,34.03929,-118.19900,CA12465,160463,...,<NA>,<NA>,<NA>,"1ST ST SL 2F E OF DACOTAH ST E/W , Los Angeles...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN
3,942664,Phoenix Tower International,942203,Phoenix Tower International,2019-08-22 20:40:05.047291,2023-01-13 13:04:52.922687,34.03910,-118.19900,CA11987,159985,...,<NA>,<NA>,<NA>,"1ST ST SL 53F W OF FRESNO ST WF , Los Angeles,...",NaN,No,Unconfirmed,Proximity Audit (21m),21.075388,0.0
4,942761,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:58:33.921034,2023-01-13 12:41:13.613007,33.98899,-118.31900,CA12570,160568,...,<NA>,<NA>,<NA>,"SLAUSON AVE NL 42F E OF 3RD AVE EF , Los Angel...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64517,1161828,ASR,264885,Crown Castle,2015-04-20 20:26:56.595238,2017-07-28 23:55:23.576292,39.00210,-95.73923,SKYLINE PARK/BURNETT'S MOUND,877829,...,<NA>,<NA>,<NA>,"3511 SW SKYLINE PARKWAY , TOPEKA, KS 66614, UN...",2007-01-12,No,Active,Proximity Audit (7m),7.276909,4.5
64518,1161829,ASR,<NA>,ASR,2025-04-01 15:38:18.798044,2025-04-09 11:29:48.458842,32.81861,-86.61714,<NA>,<NA>,...,2024-ASO-23463-OE,<NA>,<NA>,"Off Logan Road , Clanton, AL 35045, UNITED STATES",2024-11-26,No,Active,Focus Asset,NaN,NaN
64519,1161829,ASR,1136163,ASR,2024-01-23 15:24:06.649527,2024-01-23 09:31:32.771277,32.81861,-86.61714,<NA>,<NA>,...,2023-ASO-27414-OE,<NA>,<NA>,"Off Logan Road , Clanton, AL 35046, UNITED STATES",2024-11-26,No,Active,Proximity Audit (0m),0.000000,-20.4
64520,1162574,ASR,<NA>,ASR,2025-07-24 21:02:34.966713,2025-07-28 09:49:51.734224,42.34739,-91.47869,<NA>,<NA>,...,2024-ACE-6738-OE,<NA>,<NA>,"3048 Hwy 13 IA-5254 , Ryan, IA 52330, UNITED S...",NaN,No,ASR-Granted,Focus Asset,NaN,NaN


In [58]:
case4_auto_merge_further_filter

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,faa_study_number,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference
34580,1147159,SBA,<NA>,SBA,2025-02-19 20:47:14.736408,2025-02-19 20:47:14.612784,61.11381,-149.86151,Canine Castle,AK12386-A,...,NaN,<NA>,<NA>,"11801 Old Seward Hwy , Anchorage, AK 99515, UN...",NaN,No,Active,Focus Asset,NaN,NaN
34581,1147159,SBA,117947,dledbetter,2015-02-03 20:22:46.310560,2019-07-30 10:15:20.762731,61.11382,-149.86151,Canine Castle,AK12386-A,...,2010-AAL-202-OE,<NA>,<NA>,", Anchorage, AK 99515, UNITED STATES",2009-05-06,No,Active,Proximity Audit (1m),1.114310,0.0
34582,1147160,SBA,<NA>,SBA,2025-02-19 20:47:16.835983,2025-02-19 20:47:16.766783,61.16028,-149.85808,Brayton Pax Soto,AK12387-A,...,NaN,<NA>,<NA>,"6646 Homer Drive , Anchorage, AK 99518, UNITED...",NaN,No,Active,Focus Asset,NaN,NaN
34583,1147160,SBA,117948,dledbetter,2015-02-03 20:22:46.346536,2019-07-30 10:15:20.762731,61.16028,-149.85809,Brayton Pax Soto,AK12387-A,...,2010-AAL-203-OE,<NA>,<NA>,", Anchorage, AK 99518, UNITED STATES",2009-05-06,No,Active,Proximity Audit (0m),0.538346,0.0
34584,1147161,SBA,<NA>,SBA,2025-02-19 20:47:18.928215,2025-02-19 20:47:18.862005,61.13868,-149.84196,Tikigaq,AK12388-A,...,NaN,<NA>,<NA>,"2121 Abbott Road , Anchorage, AK 99507, UNITED...",NaN,No,Active,Focus Asset,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64466,1161552,SBA,118020,SBA,2015-02-03 20:22:46.431704,2017-11-08 13:04:11.713717,33.75752,-85.78392,Hamiltons Lake,AL04110-B,...,2022-ASO-34760-OE,<NA>,<NA>,", Anniston, AL 36206-7568, UNITED STATES",2000-08-03,No,Active,Proximity Audit (1m),1.109180,-0.1
64469,1161554,SBA,<NA>,SBA,2025-02-20 05:42:05.608936,2025-02-20 05:42:05.545048,34.11235,-88.02328,West Hamilton,AL04756-A,...,NaN,<NA>,<NA>,"596 Fire Tower Road , Hamilton, AL 35570-9721,...",NaN,No,Active,Focus Asset,NaN,NaN
64470,1161554,SBA,118022,SBA,2015-02-03 20:22:46.433772,2017-11-08 13:03:54.697111,34.11235,-88.02329,West Hamilton,AL04756-A,...,2024-ASO-19958-OE,<NA>,<NA>,", Hamilton, AL 35570-9721, UNITED STATES",2000-06-24,No,Active,Proximity Audit (0m),0.922630,-3.1
64474,1161556,SBA,<NA>,SBA,2025-02-20 05:42:15.236684,2025-02-20 05:42:15.173002,34.58570,-86.45280,Hamptons Cove,AL04832-B,...,NaN,<NA>,<NA>,"238 Wilson Mann Road , Owens Cross Roads, AL 3...",NaN,No,Active,Focus Asset,NaN,NaN


In [59]:
updated_case4_prox_audits_post_auto_merge_table

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference,abs_agldiff
0,942614,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:53:42.284115,2023-01-13 15:25:12.493704,33.94619,-118.28200,CA12409,160407,...,<NA>,<NA>,"FIGUEROA ST WL 75F S OF 99TH ST S/N , August F...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN,NaN
1,942614,Phoenix Tower International,943927,Phoenix Tower International,2019-08-22 21:36:14.93064,2023-06-26 12:05:50.800659,33.94574,-118.28200,MAIN ST EL 185F N OF 110TH ST SF,161773,...,<NA>,<NA>,"MAIN ST EL 185F N OF 110TH ST SF , , CA 90003,...",NaN,No,Unconfirmed,Proximity Audit (48m),49.914635,0.0,NaN
2,942664,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:55:21.34445,2023-01-13 11:49:35.586868,34.03929,-118.19900,CA12465,160463,...,<NA>,<NA>,"1ST ST SL 2F E OF DACOTAH ST E/W , Los Angeles...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN,NaN
3,942664,Phoenix Tower International,942203,Phoenix Tower International,2019-08-22 20:40:05.047291,2023-01-13 13:04:52.922687,34.03910,-118.19900,CA11987,159985,...,<NA>,<NA>,"1ST ST SL 53F W OF FRESNO ST WF , Los Angeles,...",NaN,No,Unconfirmed,Proximity Audit (21m),21.075388,0.0,NaN
4,942761,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:58:33.921034,2023-01-13 12:41:13.613007,33.98899,-118.31900,CA12570,160568,...,<NA>,<NA>,"SLAUSON AVE NL 42F E OF 3RD AVE EF , Los Angel...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56833,1161485,SBA,118519,SBA,2015-02-03 20:22:46.957858,2017-11-08 10:41:29.22358,33.23192,-85.75846,MELLOW VALLEY,AL21818-A,...,<NA>,<NA>,", ASHLAND, AL 36251, UNITED STATES",2005-11-04,No,Active,Proximity Audit (1m),1.448739,0.0,NaN
56834,1161504,SBA,648125,ASR,2015-02-03 23:55:20.061725,2018-08-06 15:47:36.82213,31.47442,-87.34879,Bellview - Wiggins,AL40490-T,...,<NA>,<NA>,", Monroeville, AL 36460, UNITED STATES",2012-03-29,No,Active,Proximity Audit (1m),1.460281,-27.2,NaN
56835,1161504,SBA,<NA>,SBA,2025-02-20 05:40:08.032426,2025-02-20 05:40:07.968054,31.47441,-87.34878,Bellview - Wiggins,AL40490-T,...,<NA>,<NA>,"XX Charlie Sawyer Road , Monroeville, AL 36460...",NaN,No,Active,Focus Asset,NaN,NaN,NaN
56836,1161525,SBA,<NA>,SBA,2025-02-20 05:40:55.429298,2025-02-20 05:40:55.36567,34.62724,-86.09185,Hwy 72,AL45024-A,...,<NA>,<NA>,"3639 Porter Road , Scottsboro, AL 35768, UNITE...",NaN,No,Active,Focus Asset,NaN,NaN,NaN


In [60]:
case4_post_auto_merge_table

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,faa_study_number,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference
0,1147159,SBA,<NA>,Auto-Merged 12/2025,2025-02-19 20:47:14.736408,2025-12-02 22:20:15,61.11381,-149.86151,Canine Castle,AK12386-A,...,2020-AAL-161-OE,<NA>,<NA>,"11801 Old Seward Hwy , Anchorage, AK 99515, UN...",2009-05-06,No,Active,Focus Asset,NaN,NaN
1,1147160,SBA,<NA>,Auto-Merged 12/2025,2025-02-19 20:47:16.835983,2025-12-02 22:20:15,61.16028,-149.85808,Brayton Pax Soto,AK12387-A,...,2020-AAL-162-OE,<NA>,<NA>,"6646 Homer Drive , Anchorage, AK 99518, UNITED...",2009-05-06,No,Active,Focus Asset,NaN,NaN
2,1147161,SBA,<NA>,Auto-Merged 12/2025,2025-02-19 20:47:18.928215,2025-12-02 22:20:15,61.13868,-149.84196,Tikigaq,AK12388-A,...,2020-AAL-165-OE,<NA>,<NA>,"2121 Abbott Road , Anchorage, AK 99507, UNITED...",2009-05-08,No,Active,Focus Asset,NaN,NaN
3,1147162,SBA,<NA>,Auto-Merged 12/2025,2025-02-19 20:47:22.075974,2025-12-02 22:20:15,61.13942,-149.91288,Dimond House,AK14602-A,...,2020-AAL-159-OE,<NA>,<NA>,"1767 W. Dimond Blvd. , Anchorage, AK 99502, UN...",2011-05-27,No,Active,Focus Asset,NaN,NaN
4,1147163,SBA,<NA>,Auto-Merged 12/2025,2025-02-19 20:47:24.192863,2025-12-02 22:20:15,58.28950,-134.39446,Juneau Port,AK14604-A,...,2011-AAL-84-OE,<NA>,<NA>,"1076 Jacobsen Drive , Juneau, AK 99801, UNITED...",2011-07-28,No,Active,Focus Asset,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7679,1161550,SBA,<NA>,Auto-Merged 12/2025,2025-02-20 05:41:57.044402,2025-12-02 22:20:15,32.97265,-86.70287,Jemison,AL03077-S,...,2024-ASO-22463-OE,<NA>,<NA>,"11650 County Road 42 , Jemison, AL 35085, UNIT...",2001-06-26,No,Active,Focus Asset,NaN,NaN
7680,1161551,SBA,<NA>,Auto-Merged 12/2025,2025-02-20 05:41:59.208766,2025-12-02 22:20:15,33.63757,-85.80192,Coleman Road,AL04109-B,...,2022-ASO-28528-OE,<NA>,<NA>,"2510 Coleman Road , Anniston, AL 36207, UNITED...",2000-07-25,No,Active,Focus Asset,NaN,NaN
7681,1161552,SBA,<NA>,Auto-Merged 12/2025,2025-02-20 05:42:01.344507,2025-12-02 22:20:15,33.75751,-85.78392,Hamiltons Lake,AL04110-B,...,2022-ASO-34760-OE,<NA>,<NA>,"8505 Old Jacksonville Hwy , Anniston, AL 36206...",2000-08-03,No,Active,Focus Asset,NaN,NaN
7682,1161554,SBA,<NA>,Auto-Merged 12/2025,2025-02-20 05:42:05.608936,2025-12-02 22:20:15,34.11235,-88.02328,West Hamilton,AL04756-A,...,2024-ASO-19958-OE,<NA>,<NA>,"596 Fire Tower Road , Hamilton, AL 35570-9721,...",2000-06-24,No,Active,Focus Asset,NaN,NaN


In [61]:
case4_raw_post_auto_merge_table

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,faa_study_number,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference
0,1147159,SBA,<NA>,Auto-Merged 12/2025,2025-02-19 20:47:14.736408,2025-12-02 22:20:15,61.11381,-149.86151,Canine Castle,AK12386-A,...,2020-AAL-161-OE,NaN,<NA>,"11801 Old Seward Hwy , Anchorage, AK 99515, UN...",2009-05-06,No,Active,Focus Asset,NaN,NaN
1,1147159,SBA,117947,dledbetter,2015-02-03 20:22:46.310560,2019-07-30 10:15:20.762731,61.11382,-149.86151,Canine Castle,AK12386-A,...,2010-AAL-202-OE,NaN,<NA>,", Anchorage, AK 99515, UNITED STATES",2009-05-06,No,Active,Proximity Audit (1m),1.114310,0.0
2,1147159,SBA,<NA>,SBA,2025-02-19 20:47:14.736408,2025-02-19 20:47:14.612784,61.11381,-149.86151,Canine Castle,AK12386-A,...,NaN,NaN,<NA>,"11801 Old Seward Hwy , Anchorage, AK 99515, UN...",NaN,No,Active,Focus Asset,NaN,NaN
3,1147160,SBA,<NA>,Auto-Merged 12/2025,2025-02-19 20:47:16.835983,2025-12-02 22:20:15,61.16028,-149.85808,Brayton Pax Soto,AK12387-A,...,2020-AAL-162-OE,NaN,<NA>,"6646 Homer Drive , Anchorage, AK 99518, UNITED...",2009-05-06,No,Active,Focus Asset,NaN,NaN
4,1147160,SBA,117948,dledbetter,2015-02-03 20:22:46.346536,2019-07-30 10:15:20.762731,61.16028,-149.85809,Brayton Pax Soto,AK12387-A,...,2010-AAL-203-OE,NaN,<NA>,", Anchorage, AK 99518, UNITED STATES",2009-05-06,No,Active,Proximity Audit (0m),0.538346,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23047,1161554,SBA,118022,SBA,2015-02-03 20:22:46.433772,2017-11-08 13:03:54.697111,34.11235,-88.02329,West Hamilton,AL04756-A,...,2024-ASO-19958-OE,NaN,<NA>,", Hamilton, AL 35570-9721, UNITED STATES",2000-06-24,No,Active,Proximity Audit (0m),0.922630,-3.1
23048,1161554,SBA,<NA>,SBA,2025-02-20 05:42:05.608936,2025-02-20 05:42:05.545048,34.11235,-88.02328,West Hamilton,AL04756-A,...,NaN,NaN,<NA>,"596 Fire Tower Road , Hamilton, AL 35570-9721,...",NaN,No,Active,Focus Asset,NaN,NaN
23049,1161556,SBA,<NA>,Auto-Merged 12/2025,2025-02-20 05:42:15.236684,2025-12-02 22:20:15,34.58570,-86.45280,Hamptons Cove,AL04832-B,...,2024-ASO-19484-OE,NaN,<NA>,"238 Wilson Mann Road , Owens Cross Roads, AL 3...",2000-12-11,No,Active,Focus Asset,NaN,NaN
23050,1161556,SBA,118029,SBA,2015-02-03 20:22:46.443693,2017-11-08 13:03:58.876764,34.58571,-86.45280,Hamptons Cove,AL04832-B,...,2024-ASO-19484-OE,NaN,<NA>,", Owens Cross Roads, AL 35763-8605, UNITED ST...",2000-12-11,No,Active,Proximity Audit (1m),1.109330,0.3


In [62]:
case4_aggregated_final_asset_table

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference,abs_agldiff
0,1128609,Phoenix Tower International,<NA>,Phoenix Tower International,2023-05-08 20:00:48.103441,2023-05-08 16:00:48.024746,41.05333,-73.53527,555 Main St,US-CT-1255,...,<NA>,NaN,"555 Main St , Stamford, CT 06901, UNITED STATES",NaN,No,Unconfirmed,Focus Asset,NaN,NaN,NaN
1,1130226,Phoenix Tower International,<NA>,Phoenix Tower International,2023-05-08 20:30:04.261478,2023-05-08 16:30:04.153621,41.35416,-72.09805,26 Washington St,US-CT-1295,...,<NA>,NaN,"26 Washington St , New London, CT 06320, UNITE...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN,NaN
2,1139789,ASR,<NA>,Auto-Merged 12/2025,2021-02-22 14:55:46.271143,2025-12-02 22:19:09,43.19161,-86.09733,Heights Ravenna Rd,MI-0028,...,<NA>,NaN,"5998 Heights Ravenna Road (MI-0028) , Fruitpor...",2020-06-25,No,Active,Focus Asset,NaN,NaN,NaN
3,1139997,ASR,<NA>,Auto-Merged 12/2025,2021-02-22 14:55:46.271143,2025-12-02 22:19:09,39.32000,-85.83764,WS St Louis Crossing - C,<NA>,...,<NA>,NaN,"7600 E 800 N (IN-0026) , Columbus, IN 47201, U...",2020-08-19,No,Active,Focus Asset,NaN,NaN,NaN
4,1140066,ASR,<NA>,Auto-Merged 12/2025,2020-06-01 08:55:02.975387,2025-12-02 22:19:09,35.19025,-87.04408,Hwy 11,<NA>,...,<NA>,NaN,"Hwy 11 , Pulaski, TN 38478, UNITED STATES",2020-12-08,No,Active,Focus Asset,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7124,1161552,SBA,<NA>,Auto-Merged 12/2025,2025-02-20 05:42:01.344507,2025-12-02 22:20:15,33.75751,-85.78392,Hamiltons Lake,AL04110-B,...,<NA>,<NA>,"8505 Old Jacksonville Hwy , Anniston, AL 36206...",2000-08-03,No,Active,Focus Asset,NaN,NaN,NaN
7125,1161556,SBA,<NA>,Auto-Merged 12/2025,2025-02-20 05:42:15.236684,2025-12-02 22:20:15,34.58570,-86.45280,Hamptons Cove,AL04832-B,...,<NA>,<NA>,"238 Wilson Mann Road , Owens Cross Roads, AL 3...",2000-12-11,No,Active,Focus Asset,NaN,NaN,NaN
7126,1148196,SBA,<NA>,SBA,2025-02-19 21:24:55.452334,2025-02-19 21:24:55.40274,36.50733,-121.90939,Lobos Ridge,CA20557-A,...,<NA>,<NA>,"3400 Red Wolf Drive , Carmel, CA 93924, UNITED...",NaN,No,Active,Focus Asset,NaN,NaN,NaN
7127,1153450,SBA,<NA>,SBA,2025-02-20 00:29:41.727753,2025-02-20 00:29:41.633851,42.16512,-70.88884,Rockland 1,MA01115-A,...,<NA>,<NA>,"55 Accord Park Drive , Rockland, MA 02370-1070...",NaN,No,Active,Focus Asset,NaN,NaN,NaN


In [63]:
final_case4_prox_audits_post_auto_merge_table

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference,abs_agldiff
0,942614,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:53:42.284115,2023-01-13 15:25:12.493704,33.94619,-118.28200,CA12409,160407,...,<NA>,<NA>,"FIGUEROA ST WL 75F S OF 99TH ST S/N , August F...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN,NaN
1,942614,Phoenix Tower International,943927,Phoenix Tower International,2019-08-22 21:36:14.93064,2023-06-26 12:05:50.800659,33.94574,-118.28200,MAIN ST EL 185F N OF 110TH ST SF,161773,...,<NA>,<NA>,"MAIN ST EL 185F N OF 110TH ST SF , , CA 90003,...",NaN,No,Unconfirmed,Proximity Audit (48m),49.914635,0.0,NaN
2,942664,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:55:21.34445,2023-01-13 11:49:35.586868,34.03929,-118.19900,CA12465,160463,...,<NA>,<NA>,"1ST ST SL 2F E OF DACOTAH ST E/W , Los Angeles...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN,NaN
3,942664,Phoenix Tower International,942203,Phoenix Tower International,2019-08-22 20:40:05.047291,2023-01-13 13:04:52.922687,34.03910,-118.19900,CA11987,159985,...,<NA>,<NA>,"1ST ST SL 53F W OF FRESNO ST WF , Los Angeles,...",NaN,No,Unconfirmed,Proximity Audit (21m),21.075388,0.0,NaN
4,942761,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:58:33.921034,2023-01-13 12:41:13.613007,33.98899,-118.31900,CA12570,160568,...,<NA>,<NA>,"SLAUSON AVE NL 42F E OF 3RD AVE EF , Los Angel...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49688,1161828,ASR,264885,Crown Castle,2015-04-20 20:26:56.595238,2017-07-28 23:55:23.576292,39.00210,-95.73923,SKYLINE PARK/BURNETT'S MOUND,877829,...,<NA>,<NA>,"3511 SW SKYLINE PARKWAY , TOPEKA, KS 66614, UN...",2007-01-12,No,Active,Proximity Audit (7m),7.276909,4.5,NaN
49689,1161829,ASR,<NA>,ASR,2025-04-01 15:38:18.798044,2025-04-09 11:29:48.458842,32.81861,-86.61714,<NA>,<NA>,...,<NA>,<NA>,"Off Logan Road , Clanton, AL 35045, UNITED STATES",2024-11-26,No,Active,Focus Asset,NaN,NaN,NaN
49690,1161829,ASR,1136163,ASR,2024-01-23 15:24:06.649527,2024-01-23 09:31:32.771277,32.81861,-86.61714,<NA>,<NA>,...,<NA>,<NA>,"Off Logan Road , Clanton, AL 35046, UNITED STATES",2024-11-26,No,Active,Proximity Audit (0m),0.000000,-20.4,NaN
49691,1162574,ASR,<NA>,ASR,2025-07-24 21:02:34.966713,2025-07-28 09:49:51.734224,42.34739,-91.47869,<NA>,<NA>,...,<NA>,<NA>,"3048 Hwy 13 IA-5254 , Ryan, IA 52330, UNITED S...",NaN,No,ASR-Granted,Focus Asset,NaN,NaN,NaN


In [64]:
case4_aggregated_final_asset_table[case4_aggregated_final_asset_table['source'] == 'Auto-Merged 11/2025']

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference,abs_agldiff


In [65]:
final_case4_prox_audits_post_auto_merge_table[final_case4_prox_audits_post_auto_merge_table['source'] == 'Auto-Merged 11/2025']

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference,abs_agldiff


In [66]:
case4_aggregated_final_asset_table.to_csv('case4_aggregated_final_asset_table.csv')
case4_auto_merge_candidates.to_csv('case4_auto_merge_candidates.csv')
case4_post_auto_merge_table.to_csv('case4_post_auto_merge_table.csv')
case4_raw_post_auto_merge_table.to_csv('case4_raw_post_auto_merge_table.csv')
final_case4_prox_audits_post_auto_merge_table.to_csv('final_case4_prox_audits_post_auto_merge_table.csv')
initial_case4_prox_audits_post_auto_merge_table.to_csv('initial_case4_prox_audits_post_auto_merge_table.csv')
updated_case4_prox_audits_post_auto_merge_table.to_csv('updated_case4_prox_audits_post_auto_merge_table.csv')
case4_auto_merge_further_filter.to_csv('case4_auto_merge_further_filter.csv')

---

## Case 5

In [67]:
case5_auto_merge_candidates, initial_case5_prox_audits_post_auto_merge_table = split_case_5_audits(final_case4_prox_audits_post_auto_merge_table)

In [68]:
case5_auto_merge_candidates

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference,abs_agldiff
49152,1161017,SBA,<NA>,SBA,2025-02-20 05:21:22.940506,2025-02-20 05:21:22.873848,43.00667,-87.98071,Electric Avenue,WI22396-A,...,<NA>,<NA>,"2142 S. 55 Street , West Allis, WI 53219, UNIT...",NaN,No,Active,Focus Asset,NaN,NaN,NaN
49153,1161017,SBA,873745,SBA,2019-07-29 22:22:29.134802,2019-07-29 22:22:29.131642,43.00668,-87.98072,Electric Avenue,WI22396-A,...,<NA>,<NA>,", West Allis, WI 53219, UNITED STATES",NaN,No,Active,Proximity Audit (1m),1.378010,0.0,NaN
40962,1153396,SBA,<NA>,SBA,2025-02-20 00:27:35.151741,2025-02-20 00:27:35.045297,30.52633,-93.03866,Reeves,LA20713-A,...,<NA>,<NA>,"254 Sawmill Road , Reeves, LA 70658, UNITED ST...",NaN,No,Active,Focus Asset,NaN,NaN,NaN
49154,1161019,SBA,<NA>,SBA,2025-02-20 05:21:26.096716,2025-02-20 05:21:26.031644,43.18326,-88.07972,Warren Street,WI22398-A,...,<NA>,<NA>,"N91W13749 Warren Street , Menomonee Falls, WI ...",NaN,No,Active,Focus Asset,NaN,NaN,NaN
40964,1153396,SBA,124496,SBA,2015-02-03 20:22:53.079725,2017-11-08 11:44:50.122239,30.52633,-93.03867,Reeves,LA20713-A,...,<NA>,<NA>,", Reeves, LA 70658, UNITED STATES",NaN,No,Active,Proximity Audit (0m),0.959731,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49147,1161016,SBA,<NA>,SBA,2025-02-20 05:21:20.596661,2025-02-20 05:21:20.53261,42.95804,-88.04900,Hale Interchange,WI22395-A,...,<NA>,<NA>,"4717 S. 108th Street , Greenfield, WI 53228, U...",NaN,No,Active,Focus Asset,NaN,NaN,NaN
40956,1153392,SBA,124492,SBA,2015-02-03 20:22:53.075807,2017-11-08 11:44:24.572451,30.30013,-93.23730,Moss Bluff West,LA20708-A,...,<NA>,<NA>,", MOSS BLUFF, LA 70611, UNITED STATES",NaN,No,Active,Proximity Audit (1m),1.108575,0.0,NaN
40957,1153393,SBA,<NA>,SBA,2025-02-20 00:27:28.828565,2025-02-20 00:27:28.728445,30.39915,-93.23360,Moss Bluff,LA20709-A,...,<NA>,<NA>,"5607A North Perkins Ferry Road , Lake Charles,...",NaN,No,Active,Focus Asset,NaN,NaN,NaN
49150,1161016,SBA,873744,SBA,2019-07-29 22:22:26.743259,2019-07-29 22:22:26.742142,42.95805,-88.04900,Hale Interchange,WI22395-A,...,<NA>,<NA>,", Greenfield, WI 53228, UNITED STATES",NaN,No,Active,Proximity Audit (1m),1.110919,-0.1,NaN


In [69]:
case5_auto_merge_candidates['focus_asset_id'].nunique()

1507

In [70]:
initial_case5_prox_audits_post_auto_merge_table

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference,abs_agldiff
0,942614,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:53:42.284115,2023-01-13 15:25:12.493704,33.94619,-118.28200,CA12409,160407,...,<NA>,<NA>,"FIGUEROA ST WL 75F S OF 99TH ST S/N , August F...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN,NaN
1,942614,Phoenix Tower International,943927,Phoenix Tower International,2019-08-22 21:36:14.93064,2023-06-26 12:05:50.800659,33.94574,-118.28200,MAIN ST EL 185F N OF 110TH ST SF,161773,...,<NA>,<NA>,"MAIN ST EL 185F N OF 110TH ST SF , , CA 90003,...",NaN,No,Unconfirmed,Proximity Audit (48m),49.914635,0.0,NaN
2,942664,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:55:21.34445,2023-01-13 11:49:35.586868,34.03929,-118.19900,CA12465,160463,...,<NA>,<NA>,"1ST ST SL 2F E OF DACOTAH ST E/W , Los Angeles...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN,NaN
3,942664,Phoenix Tower International,942203,Phoenix Tower International,2019-08-22 20:40:05.047291,2023-01-13 13:04:52.922687,34.03910,-118.19900,CA11987,159985,...,<NA>,<NA>,"1ST ST SL 53F W OF FRESNO ST WF , Los Angeles,...",NaN,No,Unconfirmed,Proximity Audit (21m),21.075388,0.0,NaN
4,942761,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:58:33.921034,2023-01-13 12:41:13.613007,33.98899,-118.31900,CA12570,160568,...,<NA>,<NA>,"SLAUSON AVE NL 42F E OF 3RD AVE EF , Los Angel...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49688,1161828,ASR,264885,Crown Castle,2015-04-20 20:26:56.595238,2017-07-28 23:55:23.576292,39.00210,-95.73923,SKYLINE PARK/BURNETT'S MOUND,877829,...,<NA>,<NA>,"3511 SW SKYLINE PARKWAY , TOPEKA, KS 66614, UN...",2007-01-12,No,Active,Proximity Audit (7m),7.276909,4.5,NaN
49689,1161829,ASR,<NA>,ASR,2025-04-01 15:38:18.798044,2025-04-09 11:29:48.458842,32.81861,-86.61714,<NA>,<NA>,...,<NA>,<NA>,"Off Logan Road , Clanton, AL 35045, UNITED STATES",2024-11-26,No,Active,Focus Asset,NaN,NaN,NaN
49690,1161829,ASR,1136163,ASR,2024-01-23 15:24:06.649527,2024-01-23 09:31:32.771277,32.81861,-86.61714,<NA>,<NA>,...,<NA>,<NA>,"Off Logan Road , Clanton, AL 35046, UNITED STATES",2024-11-26,No,Active,Proximity Audit (0m),0.000000,-20.4,NaN
49691,1162574,ASR,<NA>,ASR,2025-07-24 21:02:34.966713,2025-07-28 09:49:51.734224,42.34739,-91.47869,<NA>,<NA>,...,<NA>,<NA>,"3048 Hwy 13 IA-5254 , Ryan, IA 52330, UNITED S...",NaN,No,ASR-Granted,Focus Asset,NaN,NaN,NaN


In [71]:
case5_auto_merge_further_filter, updated_case5_prox_audits_post_auto_merge_table, case5_post_auto_merge_table, case5_raw_post_auto_merge_table = apply_case_5_full_processing(case5_auto_merge_candidates, initial_case5_prox_audits_post_auto_merge_table)

Starting Case 5 processing...


Processing Case 5: 100%|███████████████████| 1507/1507 [00:05<00:00, 264.48it/s]


--- Case 5 Processing completed in: 0.11 minutes ---


In [72]:
case5_auto_merge_further_filter

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference,abs_agldiff
33998,1146141,SBA,<NA>,SBA,2025-02-07 18:48:28.996003,2025-02-07 18:48:28.902566,29.63527,-98.20194,UNION PACIFIC - NEW BRAUNFELS (MIC),TX30414-M,...,<NA>,<NA>,"200' SOUTH OF FROBOESE AND ECKHART , NEW BRAUN...",NaN,No,Active,Focus Asset,NaN,NaN,NaN
33999,1146141,SBA,812742,SBA,2017-11-08 15:26:05.520871,2017-11-08 12:11:42.265212,29.63528,-98.20194,UNION PACIFIC - NEW BRAUNFELS (MIC),TX30414-M,...,<NA>,<NA>,", NEW BRAUNFELS, TX 78132, UNITED STATES",NaN,No,Active,Proximity Audit (1m),1.108463,-0.1,NaN
34431,1147082,Everest Infrastructure Partners,<NA>,Everest Infrastructure Partners,2025-02-18 16:24:43.128157,2025-02-18 16:24:43.060276,39.33861,-120.53593,Signal Peak Tower,US948577,...,<NA>,<NA>,"Fordyce Lake Road , Signal Peak, CA 95728, UNI...",NaN,No,Active,Focus Asset,NaN,NaN,NaN
34434,1147082,Everest Infrastructure Partners,1136768,Everest Infrastructure Partners,2024-03-11 19:52:07.801014,2024-03-11 19:52:07.752987,39.33861,-120.53592,Signal Peak Tower,US948577,...,<NA>,<NA>,"Fordyce Lake Road , Signal Peak, CA 95728, UNI...",NaN,No,Active,Proximity Audit (0m),0.862120,0.0,NaN
34435,1147083,Everest Infrastructure Partners,<NA>,Everest Infrastructure Partners,2025-02-18 16:24:45.764095,2025-02-18 16:24:45.69894,37.64691,-122.40442,DoubleTree San Francisco Airpt South,US1021205,...,<NA>,<NA>,"275 S Airport Blvd , South San Francisco, CA 9...",NaN,No,Active,Focus Asset,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49524,1161425,SBA,463695,SBA,2015-10-07 14:53:10.478301,2017-11-08 10:42:49.585368,30.61561,-88.13534,Rivere du Chein,AL17028-A,...,<NA>,<NA>,", Mobile, AL 36693, UNITED STATES",NaN,No,Active,Proximity Audit (1m),1.108629,0.0,NaN
49525,1161426,SBA,<NA>,SBA,2025-02-20 05:37:04.504778,2025-02-20 05:37:04.442489,30.79995,-88.08447,Saraland,AL17029-A,...,<NA>,<NA>,"350 Industrial Parkway , Saraland, AL 36571, U...",NaN,No,Active,Focus Asset,NaN,NaN,NaN
49526,1161426,SBA,463696,SBA,2015-10-07 14:53:10.478301,2017-11-08 10:42:33.923615,30.79996,-88.08448,Saraland,AL17029-A,...,<NA>,<NA>,", Saraland, AL 36571, UNITED STATES",NaN,No,Active,Proximity Audit (1m),1.464594,0.0,NaN
49599,1161490,SBA,<NA>,SBA,2025-02-20 05:39:36.736587,2025-02-20 05:39:36.668613,34.83366,-86.64343,Burwell Springs,AL40003-A,...,<NA>,<NA>,"7485 Pulaski Pike , Huntsville, AL 35773, UNIT...",NaN,No,Active,Focus Asset,NaN,NaN,NaN


In [73]:
updated_case5_prox_audits_post_auto_merge_table

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference,abs_agldiff
0,942614,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:53:42.284115,2023-01-13 15:25:12.493704,33.94619,-118.28200,CA12409,160407,...,<NA>,<NA>,"FIGUEROA ST WL 75F S OF 99TH ST S/N , August F...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN,NaN
1,942614,Phoenix Tower International,943927,Phoenix Tower International,2019-08-22 21:36:14.93064,2023-06-26 12:05:50.800659,33.94574,-118.28200,MAIN ST EL 185F N OF 110TH ST SF,161773,...,<NA>,<NA>,"MAIN ST EL 185F N OF 110TH ST SF , , CA 90003,...",NaN,No,Unconfirmed,Proximity Audit (48m),49.914635,0.0,NaN
2,942664,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:55:21.34445,2023-01-13 11:49:35.586868,34.03929,-118.19900,CA12465,160463,...,<NA>,<NA>,"1ST ST SL 2F E OF DACOTAH ST E/W , Los Angeles...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN,NaN
3,942664,Phoenix Tower International,942203,Phoenix Tower International,2019-08-22 20:40:05.047291,2023-01-13 13:04:52.922687,34.03910,-118.19900,CA11987,159985,...,<NA>,<NA>,"1ST ST SL 53F W OF FRESNO ST WF , Los Angeles,...",NaN,No,Unconfirmed,Proximity Audit (21m),21.075388,0.0,NaN
4,942761,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:58:33.921034,2023-01-13 12:41:13.613007,33.98899,-118.31900,CA12570,160568,...,<NA>,<NA>,"SLAUSON AVE NL 42F E OF 3RD AVE EF , Los Angel...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48270,1160589,SBA,132619,dledbetter,2015-02-03 20:23:01.362696,2019-09-29 15:11:42.209365,48.90865,-117.86947,Flagstaff Mountain,WA21271-A,...,<NA>,<NA>,", North Port, WA 99157, UNITED STATES",NaN,No,Active,Proximity Audit (21m),21.142226,-0.4,NaN
48271,1160589,SBA,1160588,SBA,2025-02-20 05:05:29.302221,2025-02-20 05:05:29.249701,48.90865,-117.86947,Flagstaff Mountain,WA21271-A,...,<NA>,<NA>,"4851 Sheep Creek Road , North Port, WA 99157, ...",NaN,No,Active,Proximity Audit (21m),21.142226,0.0,NaN
48272,1160589,SBA,<NA>,SBA,2025-02-20 05:05:30.270248,2025-02-20 05:05:30.221438,48.90846,-117.86946,Flagstaff Mountain,WA21271-A,...,<NA>,<NA>,"4851 Sheep Creek Road , North Port, WA 99157, ...",NaN,No,Active,Focus Asset,NaN,NaN,NaN
48273,1161026,SBA,873756,SBA,2019-07-29 22:22:56.800970,2019-07-29 22:22:56.79387,45.89304,-89.85133,Sunday Lake,WI22407-A,...,<NA>,<NA>,", Minocqua, WI 54548, UNITED STATES",NaN,No,Active,Proximity Audit (1m),1.355649,14.0,NaN


In [74]:
case5_post_auto_merge_table

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference,abs_agldiff
0,1146141,SBA,<NA>,Auto-Merged 12/2025,2025-02-07 18:48:28.996003,2025-12-02 23:27:35,29.63527,-98.20194,UNION PACIFIC - NEW BRAUNFELS (MIC),TX30414-M,...,<NA>,<NA>,"200' SOUTH OF FROBOESE AND ECKHART , NEW BRAUN...",NaN,No,Active,Focus Asset,NaN,NaN,NaN
1,1147082,Everest Infrastructure Partners,<NA>,Auto-Merged 12/2025,2025-02-18 16:24:43.128157,2025-12-02 23:27:35,39.33861,-120.53593,Signal Peak Tower,US948577,...,<NA>,<NA>,"Fordyce Lake Road , Signal Peak, CA 95728, UNI...",NaN,No,Active,Focus Asset,NaN,NaN,NaN
2,1147083,Everest Infrastructure Partners,<NA>,Auto-Merged 12/2025,2025-02-18 16:24:45.764095,2025-12-02 23:27:35,37.64691,-122.40442,DoubleTree San Francisco Airpt South,US1021205,...,<NA>,<NA>,"275 S Airport Blvd , South San Francisco, CA 9...",NaN,No,Active,Focus Asset,NaN,NaN,NaN
3,1147084,Everest Infrastructure Partners,<NA>,Auto-Merged 12/2025,2025-02-18 16:24:47.667864,2025-12-02 23:27:35,34.25933,-118.30727,Sunland CO,US701800,...,<NA>,<NA>,"8000 Foothill Boulevard , Sunland, CA 91040, U...",NaN,No,Active,Focus Asset,NaN,NaN,NaN
4,1147086,Everest Infrastructure Partners,<NA>,Auto-Merged 12/2025,2025-02-18 16:24:53.079755,2025-12-02 23:27:35,41.09801,-73.41934,Residence Inn by Marriott Norwalk,US1021152,...,<NA>,<NA>,"45 S Main St , Norwalk, CT 06854, UNITED STATES",NaN,No,Active,Focus Asset,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1413,1161379,SBA,<NA>,Auto-Merged 12/2025,2025-02-20 05:35:06.852633,2025-12-02 23:27:35,31.65832,-85.27469,Lawrenceville,AL13028-A,...,<NA>,<NA>,"483 County Road 211 , Abbeville, AL 36310, UNI...",NaN,No,Active,Focus Asset,NaN,NaN,NaN
1414,1161401,SBA,<NA>,Auto-Merged 12/2025,2025-02-20 05:36:06.064562,2025-12-02 23:27:35,33.30568,-88.08415,Downtown Carrollton,AL14646-A,...,<NA>,<NA>,"352 William E. Hill Dr. , Carrollton, AL 35447...",NaN,No,Active,Focus Asset,NaN,NaN,NaN
1415,1161425,SBA,<NA>,Auto-Merged 12/2025,2025-02-20 05:37:02.360914,2025-12-02 23:27:35,30.61560,-88.13534,Rivere du Chein,AL17028-A,...,<NA>,<NA>,"4070 Lloyd Station Road , Mobile, AL 36693, UN...",NaN,No,Active,Focus Asset,NaN,NaN,NaN
1416,1161426,SBA,<NA>,Auto-Merged 12/2025,2025-02-20 05:37:04.504778,2025-12-02 23:27:35,30.79995,-88.08447,Saraland,AL17029-A,...,<NA>,<NA>,"350 Industrial Parkway , Saraland, AL 36571, U...",NaN,No,Active,Focus Asset,NaN,NaN,NaN


In [75]:
case5_raw_post_auto_merge_table

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference,abs_agldiff
0,1146141,SBA,<NA>,Auto-Merged 12/2025,2025-02-07 18:48:28.996003,2025-12-02 23:27:35,29.63527,-98.20194,UNION PACIFIC - NEW BRAUNFELS (MIC),TX30414-M,...,NaN,<NA>,"200' SOUTH OF FROBOESE AND ECKHART , NEW BRAUN...",NaN,No,Active,Focus Asset,NaN,NaN,NaN
1,1146141,SBA,812742,SBA,2017-11-08 15:26:05.520871,2017-11-08 12:11:42.265212,29.63528,-98.20194,UNION PACIFIC - NEW BRAUNFELS (MIC),TX30414-M,...,NaN,<NA>,", NEW BRAUNFELS, TX 78132, UNITED STATES",NaN,No,Active,Proximity Audit (1m),1.108463,-0.1,NaN
2,1146141,SBA,<NA>,SBA,2025-02-07 18:48:28.996003,2025-02-07 18:48:28.902566,29.63527,-98.20194,UNION PACIFIC - NEW BRAUNFELS (MIC),TX30414-M,...,NaN,<NA>,"200' SOUTH OF FROBOESE AND ECKHART , NEW BRAUN...",NaN,No,Active,Focus Asset,NaN,NaN,NaN
3,1147082,Everest Infrastructure Partners,<NA>,Auto-Merged 12/2025,2025-02-18 16:24:43.128157,2025-12-02 23:27:35,39.33861,-120.53593,Signal Peak Tower,US948577,...,NaN,<NA>,"Fordyce Lake Road , Signal Peak, CA 95728, UNI...",NaN,No,Active,Focus Asset,NaN,NaN,NaN
4,1147082,Everest Infrastructure Partners,1136768,Everest Infrastructure Partners,2024-03-11 19:52:07.801014,2024-03-11 19:52:07.752987,39.33861,-120.53592,Signal Peak Tower,US948577,...,NaN,<NA>,"Fordyce Lake Road , Signal Peak, CA 95728, UNI...",NaN,No,Active,Proximity Audit (0m),0.862120,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4249,1161426,SBA,463696,SBA,2015-10-07 14:53:10.478301,2017-11-08 10:42:33.923615,30.79996,-88.08448,Saraland,AL17029-A,...,NaN,<NA>,", Saraland, AL 36571, UNITED STATES",NaN,No,Active,Proximity Audit (1m),1.464594,0.0,NaN
4250,1161426,SBA,<NA>,SBA,2025-02-20 05:37:04.504778,2025-02-20 05:37:04.442489,30.79995,-88.08447,Saraland,AL17029-A,...,NaN,<NA>,"350 Industrial Parkway , Saraland, AL 36571, U...",NaN,No,Active,Focus Asset,NaN,NaN,NaN
4251,1161490,SBA,<NA>,Auto-Merged 12/2025,2025-02-20 05:39:36.736587,2025-12-02 23:27:35,34.83366,-86.64343,Burwell Springs,AL40003-A,...,NaN,<NA>,"7485 Pulaski Pike , Huntsville, AL 35773, UNIT...",NaN,No,Active,Focus Asset,NaN,NaN,NaN
4252,1161490,SBA,118525,SBA,2015-02-03 20:22:46.964012,2017-11-08 10:40:17.274368,34.83366,-86.64344,Burwell Springs,AL40003-A,...,NaN,<NA>,", Huntsville, AL 35773, UNITED STATES",NaN,No,Active,Proximity Audit (0m),0.914725,0.9,NaN


In [76]:
case5_aggregated_final_asset_table, final_case5_prox_audits_post_auto_merge_table = apply_case_5_maintenance_logic(prox_audits_table, updated_case5_prox_audits_post_auto_merge_table, case5_post_auto_merge_table, case5_raw_post_auto_merge_table, case4_aggregated_final_asset_table)

In [77]:
case5_aggregated_final_asset_table

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference,abs_agldiff
0,1128609,Phoenix Tower International,<NA>,Phoenix Tower International,2023-05-08 20:00:48.103441,2023-05-08 16:00:48.024746,41.05333,-73.53527,555 Main St,US-CT-1255,...,<NA>,NaN,"555 Main St , Stamford, CT 06901, UNITED STATES",NaN,No,Unconfirmed,Focus Asset,NaN,NaN,NaN
1,1130226,Phoenix Tower International,<NA>,Phoenix Tower International,2023-05-08 20:30:04.261478,2023-05-08 16:30:04.153621,41.35416,-72.09805,26 Washington St,US-CT-1295,...,<NA>,NaN,"26 Washington St , New London, CT 06320, UNITE...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN,NaN
2,1139789,ASR,<NA>,Auto-Merged 12/2025,2021-02-22 14:55:46.271143,2025-12-02 22:19:09,43.19161,-86.09733,Heights Ravenna Rd,MI-0028,...,<NA>,NaN,"5998 Heights Ravenna Road (MI-0028) , Fruitpor...",2020-06-25,No,Active,Focus Asset,NaN,NaN,NaN
3,1139997,ASR,<NA>,Auto-Merged 12/2025,2021-02-22 14:55:46.271143,2025-12-02 22:19:09,39.32000,-85.83764,WS St Louis Crossing - C,<NA>,...,<NA>,NaN,"7600 E 800 N (IN-0026) , Columbus, IN 47201, U...",2020-08-19,No,Active,Focus Asset,NaN,NaN,NaN
4,1140066,ASR,<NA>,Auto-Merged 12/2025,2020-06-01 08:55:02.975387,2025-12-02 22:19:09,35.19025,-87.04408,Hwy 11,<NA>,...,<NA>,NaN,"Hwy 11 , Pulaski, TN 38478, UNITED STATES",2020-12-08,No,Active,Focus Asset,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8019,1161324,SBA,<NA>,Auto-Merged 12/2025,2025-02-20 05:32:54.944660,2025-12-02 23:27:35,30.61600,-88.15756,Knollwood,AL08876-A,...,<NA>,NaN,"3620 Demetropolis Road , Mobile, AL 36693, UNI...",NaN,No,Active,Focus Asset,NaN,NaN,NaN
8020,1161401,SBA,<NA>,Auto-Merged 12/2025,2025-02-20 05:36:06.064562,2025-12-02 23:27:35,33.30568,-88.08415,Downtown Carrollton,AL14646-A,...,<NA>,NaN,"352 William E. Hill Dr. , Carrollton, AL 35447...",NaN,No,Active,Focus Asset,NaN,NaN,NaN
8021,1161426,SBA,<NA>,Auto-Merged 12/2025,2025-02-20 05:37:04.504778,2025-12-02 23:27:35,30.79995,-88.08447,Saraland,AL17029-A,...,<NA>,NaN,"350 Industrial Parkway , Saraland, AL 36571, U...",NaN,No,Active,Focus Asset,NaN,NaN,NaN
8022,1161490,SBA,<NA>,Auto-Merged 12/2025,2025-02-20 05:39:36.736587,2025-12-02 23:27:35,34.83366,-86.64343,Burwell Springs,AL40003-A,...,<NA>,NaN,"7485 Pulaski Pike , Huntsville, AL 35773, UNIT...",NaN,No,Active,Focus Asset,NaN,NaN,NaN


In [78]:
final_case5_prox_audits_post_auto_merge_table

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference,abs_agldiff
0,942614,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:53:42.284115,2023-01-13 15:25:12.493704,33.94619,-118.28200,CA12409,160407,...,<NA>,<NA>,"FIGUEROA ST WL 75F S OF 99TH ST S/N , August F...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN,NaN
1,942614,Phoenix Tower International,943927,Phoenix Tower International,2019-08-22 21:36:14.93064,2023-06-26 12:05:50.800659,33.94574,-118.28200,MAIN ST EL 185F N OF 110TH ST SF,161773,...,<NA>,<NA>,"MAIN ST EL 185F N OF 110TH ST SF , , CA 90003,...",NaN,No,Unconfirmed,Proximity Audit (48m),49.914635,0.0,NaN
2,942664,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:55:21.34445,2023-01-13 11:49:35.586868,34.03929,-118.19900,CA12465,160463,...,<NA>,<NA>,"1ST ST SL 2F E OF DACOTAH ST E/W , Los Angeles...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN,NaN
3,942664,Phoenix Tower International,942203,Phoenix Tower International,2019-08-22 20:40:05.047291,2023-01-13 13:04:52.922687,34.03910,-118.19900,CA11987,159985,...,<NA>,<NA>,"1ST ST SL 53F W OF FRESNO ST WF , Los Angeles,...",NaN,No,Unconfirmed,Proximity Audit (21m),21.075388,0.0,NaN
4,942761,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:58:33.921034,2023-01-13 12:41:13.613007,33.98899,-118.31900,CA12570,160568,...,<NA>,<NA>,"SLAUSON AVE NL 42F E OF 3RD AVE EF , Los Angel...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47291,1161828,ASR,264885,Crown Castle,2015-04-20 20:26:56.595238,2017-07-28 23:55:23.576292,39.00210,-95.73923,SKYLINE PARK/BURNETT'S MOUND,877829,...,<NA>,<NA>,"3511 SW SKYLINE PARKWAY , TOPEKA, KS 66614, UN...",2007-01-12,No,Active,Proximity Audit (7m),7.276909,4.5,NaN
47292,1161829,ASR,<NA>,ASR,2025-04-01 15:38:18.798044,2025-04-09 11:29:48.458842,32.81861,-86.61714,<NA>,<NA>,...,<NA>,<NA>,"Off Logan Road , Clanton, AL 35045, UNITED STATES",2024-11-26,No,Active,Focus Asset,NaN,NaN,NaN
47293,1161829,ASR,1136163,ASR,2024-01-23 15:24:06.649527,2024-01-23 09:31:32.771277,32.81861,-86.61714,<NA>,<NA>,...,<NA>,<NA>,"Off Logan Road , Clanton, AL 35046, UNITED STATES",2024-11-26,No,Active,Proximity Audit (0m),0.000000,-20.4,NaN
47294,1162574,ASR,<NA>,ASR,2025-07-24 21:02:34.966713,2025-07-28 09:49:51.734224,42.34739,-91.47869,<NA>,<NA>,...,<NA>,<NA>,"3048 Hwy 13 IA-5254 , Ryan, IA 52330, UNITED S...",NaN,No,ASR-Granted,Focus Asset,NaN,NaN,NaN


In [79]:
case5_aggregated_final_asset_table[case5_aggregated_final_asset_table['source'] == 'Auto-Merged 11/2025']

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference,abs_agldiff


In [80]:
final_case5_prox_audits_post_auto_merge_table[final_case5_prox_audits_post_auto_merge_table['source'] == 'Auto-Merged 11/2025']

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference,abs_agldiff


In [81]:
case5_aggregated_final_asset_table.to_csv('case5_aggregated_final_asset_table.csv')
case5_auto_merge_candidates.to_csv('case5_auto_merge_candidates.csv')
case5_post_auto_merge_table.to_csv('case5_post_auto_merge_table.csv')
case5_raw_post_auto_merge_table.to_csv('case5_raw_post_auto_merge_table.csv')
final_case5_prox_audits_post_auto_merge_table.to_csv('final_case5_prox_audits_post_auto_merge_table.csv')
initial_case5_prox_audits_post_auto_merge_table.to_csv('initial_case5_prox_audits_post_auto_merge_table.csv')
updated_case5_prox_audits_post_auto_merge_table.to_csv('updated_case5_prox_audits_post_auto_merge_table.csv')
case5_auto_merge_further_filter.to_csv('case5_auto_merge_further_filter.csv')

---

## Whole Code

The below line is just the whole code; same as the above lines

In [82]:
# import pandas as pd
# from geopy.distance import geodesic
# import numpy as np
# from typing import Tuple, List
# from datetime import datetime
# import numpy as np
# import requests
# import json
# import re
# import time 
# import warnings
# from requests.adapters import HTTPAdapter
# from urllib3.util.retry import Retry
# from concurrent.futures import ThreadPoolExecutor
# from tqdm import tqdm # Used to show progress during long-running tasks (like pre-caching)


# # loading the complete list of proximity audits and loading it in a pandas DataFrame
# df = pd.read_csv('Result 2025-09-23 05-01-51.csv')


# # Casting the data types
# df['focus_asset_id'] = df['focus_asset_id'].astype('Int64')
# df['focus_asset'] = df['focus_asset'].astype('string')
# df['associated_asset_id'] = df['associated_asset_id'].astype('Int64')
# df['source'] = df['source'].astype('string')
# df['name'] = df['name'].astype('string')
# df['operator_site_id'] = df['operator_site_id'].astype('string')
# df['type'] = df['type'].astype('string')
# df['description'] = df['description'].astype('string')
# df['operator_name'] = df['operator_name'].astype('string')
# df['manager_name'] = df['manager_name'].astype('string')
# df['fcc_owner_name'] = df['fcc_owner_name'].astype('string')
# df['shelter'] = df['shelter'].astype('string')
# df['power'] = df['power'].astype('string')
# df['fcc_asr_number'] = df['fcc_asr_number'].astype('string')
# df['faa_study_number'] = df['faa_study_number'].astype('string')
# df['cdbs_facility_id'] = df['cdbs_facility_id'].astype('string')
# df['region'] = df['region'].astype('string')
# df['address'] = df['address'].astype('string')
# df['stealth'] = df['stealth'].astype('string')
# df['asset_status'] = df['asset_status'].astype('string')
# df['audit_reason'] = df['audit_reason'].astype('string')


# def calculate_distances_to_reference(df):
#     """
#     Calculates the geodesic distance (in meters) from the reference record 
#     to every associated record within the same focus_asset_id group.
    
#     This function populates the 'distance_to_reference' column.
#     """
#     df_copy = df.copy()
#     # Initialize the new column with NaN. Only associated records will be populated.
#     df_copy['distance_to_reference'] = np.nan 

#     # Step 1: Iterate through each distinct grouping in the DataFrame
#     for group_name, group_df in df_copy.groupby('focus_asset_id'):
#         # Step 2: Identify the single reference record
#         reference_record = group_df[group_df['associated_asset_id'].isna()]

#         if not reference_record.empty:
#             # Step 3: Extract the coordinates for the reference record
#             ref_lat = reference_record['latitude'].iloc[0]
#             ref_lon = reference_record['longitude'].iloc[0]
#             reference_coords = (ref_lat, ref_lon)

#             # Step 4: Iterate through all records in the group (looking for associated records)
#             for index, row in group_df.iterrows():
#                 # Check if the current row is an associated record
#                 if pd.notna(row['associated_asset_id']):
#                     record_coords = (row['latitude'], row['longitude'])
                    
#                     # Step 5: Calculate the geodesic distance in meters
#                     distance = geodesic(reference_coords, record_coords).meters
                    
#                     # Step 6: Update the distance_to_reference field for the associated record
#                     df_copy.loc[index, 'distance_to_reference'] = distance
    
#     return df_copy


# def calculate_agldiff_to_reference(df):
#     """
#     Calculates the difference in Height Above Ground Level (AGL) between the
#     reference record and every associated record (Ref AGL - Assoc AGL).
    
#     This function populates the 'agldiff_to_reference' column.
#     """
#     df_copy = df.copy() 
#     # Initialize the new column with NaN
#     df_copy['agldiff_to_reference'] = np.nan 

#     # Step 1: Iterate through each distinct grouping
#     for group_name, group_df in df_copy.groupby('focus_asset_id'):
#         # Step 2: Identify the reference record
#         reference_record = group_df[group_df['associated_asset_id'].isna()]

#         if not reference_record.empty:
#             # Step 3: Extract the AGL for the reference record
#             ref_agl = reference_record['agl'].iloc[0]

#             # Step 4: Iterate through all associated records in the group
#             for index, row in group_df.iterrows():
#                 # Check if the current row is an associated record
#                 if pd.notna(row['associated_asset_id']):
#                     record_agl = row['agl']
                    
#                     # Step 5: Calculate the AGL difference (Ref AGL minus Assoc AGL)
#                     distance = ref_agl - record_agl
                    
#                     # Step 6: Update the agldiff_to_reference field
#                     df_copy.loc[index, 'agldiff_to_reference'] = distance
                    
#     return df_copy

# # Calculating the distance of the associated record from the corresponding focus/reference record in a group.
# df_with_distances = calculate_distances_to_reference(df)

# # Calculating the AGL difference of the associated record from the corresponding focus/reference record in a group.
# df_with_differences = calculate_agldiff_to_reference(df_with_distances)

# # redefining the variable to create the prox_audits_table DataFrame.
# prox_audits_table = df_with_differences



# # Suppress all FutureWarning messages
# # This ensures a clean terminal output by hiding warnings related to DataFrame operations.
# warnings.simplefilter(action='ignore', category=FutureWarning)


# # Global Caches for API Calls
# asn_cache = {} # Cache for ASR -> ASN lookups
# asr_cache = {} # Cache for ASN -> ASR lookups

# # API Configuration
# api_endpoint_asn = "https://oeaaa.faa.gov/oeaaa/tools-api/namedOperation.do"
# api_endpoint_asr = "https://oeaaa.faa.gov/oeaaa/oe3a/external/portal-api/caseFiling/dynamicCaseDataByAsn.do"

# # Payload templates, headers, and response keys
# api_payload_template_asn = {
#     "areaType": "id", "timeSpan": "120", "formLat": "", "formLon": "",
#     "radiusNM": 25, "structureType": "ANY", "allStatusSelected": True,
#     "criteria": {}, "fcc": "", "opName": "GET_CASE_BY_FCC",
#     "placement": "OFF_AIRPORT", "status": {}, "structureTypes": ["ANY"]
# }
# api_payload_template_asr = {
#     "areaType": "id", "timeSpan": "120", "formLat": "", "formLon": "",
#     "radiusNM": 25, "structureType": "ANY", "allStatusSelected": True,
#     "criteria": {}, "asnRegion": "", "asnYear": 0, "asnSequence": "",
#     "asnCaseType": "", "placement": "OFF_AIRPORT", "status": {},
#     "structureTypes": ["ANY"]
# }
# headers = {
#     "Content-Type": "application/json",
#     "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36"
# }
# asn_key_in_response = "ASN"
# date_key_in_response_asn = "SUBMITTED_DATE"
# fcc_asr_key_in_response = "fccAsr"

# # Robust Request Session (Retries, Timeout)
# def create_robust_session():
#     """Initializes a global requests.Session object configured for resilience."""
#     session = requests.Session()
#     session.headers.update(headers)
    
#     retry_strategy = Retry(
#         total=5, 
#         backoff_factor=1, 
#         status_forcelist=[429, 500, 502, 503, 504],
#     )
    
#     adapter = HTTPAdapter(max_retries=retry_strategy)
#     session.mount("http://", adapter)
#     session.mount("https://", adapter)
#     session.timeout = 60 # Set a 60-second timeout
    
#     return session

# api_session = create_robust_session()



# # ==============================================================================
# # I. CORE API UTILITIES AND CACHING LOGIC
# # ==============================================================================

# def parse_asn(asn_string):
#     """Parses a FAA Study Number (ASN) string into its constituent parts."""
#     match = re.match(r"(\d{4})-([A-Z]{3})-([\d\w]+)-([A-Z]{2})", str(asn_string))
#     if match:
#         year, region, sequence, casetype = match.groups()
#         return {
#             "asnYear": int(year), "asnRegion": region, "asnSequence": sequence,
#             "asnCaseType": casetype
#         }
#     return None


# def get_asn_via_api(asr_number):
#     """Fetches FAA Study Number (ASN) based on FCC ASR Number, utilizing cache."""
#     asr_number_str = str(asr_number)
#     if asr_number_str in asn_cache: return asn_cache[asr_number_str]

#     payload = api_payload_template_asn.copy()
#     payload["fcc"] = asr_number_str
    
#     try:
#         response = api_session.post(api_endpoint_asn, data=json.dumps(payload))
#         if response.status_code == 200:
#             data = response.json()
#             if data and isinstance(data, list) and len(data) > 0:
#                 def parse_date(record):
#                     date_str = record.get(date_key_in_response_asn)
#                     if date_str and date_str != "N/A":
#                         try: return datetime.strptime(date_str, "%m/%d/%Y")
#                         except ValueError: return datetime.min
#                     return datetime.min
#                 latest_record = max(data, key=parse_date)
                
#                 if asn_key_in_response in latest_record:
#                     result = latest_record[asn_key_in_response]
#                     asn_cache[asr_number_str] = result 
#                     return result
#     except Exception: pass
#     asn_cache[asr_number_str] = None
#     return None


# def get_asr_via_api(asn_number):
#     """Fetches FCC ASR Number based on FAA Study Number (ASN), utilizing cache."""
#     asn_number_str = str(asn_number)
#     if asn_number_str in asr_cache: return asr_cache[asn_number_str]

#     asn_parts = parse_asn(asn_number_str)
#     if not asn_parts:
#         asr_cache[asn_number_str] = None
#         return None

#     payload = api_payload_template_asr.copy()
#     payload.update(asn_parts)

#     try:
#         response = api_session.post(api_endpoint_asr, data=json.dumps(payload))
#         if response.status_code == 200:
#             data = response.json()
#             result = None
#             if isinstance(data, dict) and fcc_asr_key_in_response in data:
#                 result = str(data[fcc_asr_key_in_response])
#             elif isinstance(data, list) and len(data) > 0 and fcc_asr_key_in_response in data[0]:
#                 result = str(data[0][fcc_asr_key_in_response])
#             if result:
#                 asr_cache[asn_number_str] = result 
#                 return result
#     except Exception: pass
#     asr_cache[asn_number_str] = None
#     return None


# def clean_asr_in_dataframe(df: pd.DataFrame) -> pd.DataFrame:
#     """
#     Standardizes the 'fcc_asr_number' column by removing the '.0' suffix from clean integer values.
#     """
#     if 'fcc_asr_number' in df.columns:
#         original_strings = df['fcc_asr_number'].copy()
#         numeric_fcc = pd.to_numeric(df['fcc_asr_number'], errors='coerce')
        
#         def to_clean_str(x, original_val):
#             if pd.notnull(x) and x == int(x): return str(int(x))
#             if pd.notnull(original_val): return str(original_val)
#             return np.nan
        
#         df['fcc_asr_number'] = [to_clean_str(num, orig) for num, orig in zip(numeric_fcc, original_strings)]
#     return df


# def determine_faa_study_number(ref_fcc, ref_faa, assoc_faa):
#     """
#     Complex logic for Cases 2, 3, 4: Attempts to scrape the optimal FAA Study Number (ASN).
#     1. ASR -> ASN lookup (primary). 2. If null, cross-validate existing Ref/Assoc ASN.
#     """
#     # 1. Primary Check: ASR -> ASN
#     new_faa = get_asn_via_api(ref_fcc)
#     if pd.notnull(new_faa) and new_faa != 'N/A': return str(new_faa) 

#     # 2. Secondary Check: Cross-Validation (ASN -> ASR)
#     faa_ids_to_check = set()
#     if pd.notnull(ref_faa) and parse_asn(str(ref_faa)): faa_ids_to_check.add(str(ref_faa))
#     if pd.notnull(assoc_faa) and parse_asn(str(assoc_faa)): faa_ids_to_check.add(str(assoc_faa))
    
#     for faa_id in faa_ids_to_check:
#         found_asr = get_asr_via_api(faa_id)
#         if pd.notnull(found_asr) and str(found_asr) == str(ref_fcc):
#             return faa_id 

#     return None


# def get_case_1_final_faa(ref_fcc, ref_faa, assoc_faa):
#     """
#     Simple logic for Case 1: Attempts ASR -> ASN lookup. 
#     If scrape fails, it falls back to the existing Reference FAA Study Number.
#     """
#     new_faa = get_asn_via_api(ref_fcc)
    
#     if pd.notnull(new_faa) and new_faa != 'N/A': return str(new_faa)
#     else: return ref_faa


# def merge_records(reference_record, associated_record, merge_timestamp, faa_study_number=None):
#     """
#     Core merging logic: combines two records, applies field coalescing, operator priority, 
#     status update, and metadata updates.
#     """
#     merged_record = reference_record.copy()

#     # Operator Name Logic: Use associated name if ref is 'Unassigned'/'Unkown' and assoc is valid
#     ref_op = reference_record['operator_name']
#     assoc_op = associated_record['operator_name']
    
#     if pd.notnull(ref_op) and ref_op in ["Unassigned", "Unkown"] and \
#        pd.notnull(assoc_op) and assoc_op not in ["Unassigned", "Unkown"]:
#         merged_record['operator_name'] = assoc_op
    
#     # Update Source with 'Auto-Merged MM/YYYY'
#     merged_record['source'] = f"Auto-Merged {merge_timestamp.strftime('%m/%Y')}"
#     # Update Timestamp (updated_at = now)
#     merged_record['created_at'] = reference_record['created_at']
#     merged_record['updated_at'] = merge_timestamp.strftime('%Y-%m-%d %H:%M:%S')

#     # Coalesce fields: if reference field is null, fill from associated record
#     fields_to_check = [
#         "latitude", "longitude", "name", "operator_site_id", "type", "description", 
#         "manager_name", "fcc_owner_name", "agl", "amsl", "ground_elevation", "haat", 
#         "shelter", "power", "stories", "fcc_asr_number", "cdbs_facility_id", "region", 
#         "address", "construction_date", "stealth", "asset_status"
#     ]
#     for field in fields_to_check:
#         if pd.isnull(merged_record[field]) and pd.notnull(associated_record[field]):
#             merged_record[field] = associated_record[field]

#     # Update FAA Study Number with the newly scraped/validated value
#     if faa_study_number is not None: merged_record['faa_study_number'] = faa_study_number
        
#     # Final ASR Cleaning for merged record
#     if pd.notnull(merged_record['fcc_asr_number']):
#         try:
#             numeric_val = pd.to_numeric(merged_record['fcc_asr_number'], errors='coerce')
#             if pd.notnull(numeric_val) and numeric_val == int(numeric_val):
#                 merged_record['fcc_asr_number'] = str(int(numeric_val))
#             else:
#                 merged_record['fcc_asr_number'] = str(merged_record['fcc_asr_number'])
#         except Exception: pass
            
#     # Final check: If construction_date is valid, set asset_status to 'Active'
#     if pd.notnull(merged_record['construction_date']):
#         try:
#             pd.to_datetime(merged_record['construction_date'])
#             merged_record['asset_status'] = "Active"
#         except: pass
            
#     return merged_record

# # ==============================================================================
# # III. ORCHESTRATION AND CASE LOGIC (CASES 1 - 5)
# # ==============================================================================

# # Worker functions for pre-caching
# def _cache_asn(asr): get_asn_via_api(asr)
# def _cache_asr(asn): get_asr_via_api(asn)

# def pre_populate_api_caches(prox_audits_table: pd.DataFrame, max_workers: int = 3):
#     """
#     CRITICAL OPTIMIZATION STEP. Populates the global caches by executing all required
#     unique API calls safely in parallel BEFORE the main processing loop starts.
#     """
#     start_time = time.time()
#     print(f"Starting API pre-caching with max_workers={max_workers}...")
    
#     unique_asrs = prox_audits_table['fcc_asr_number'].dropna().unique()
#     unique_asns_raw = prox_audits_table['faa_study_number'].dropna().unique()
#     unique_asns = [asn for asn in unique_asns_raw if parse_asn(asn)]
    
#     # Execute ASR Caching in Parallel Threads
#     print(f"Caching {len(unique_asrs)} unique ASR numbers...")
#     with ThreadPoolExecutor(max_workers=max_workers) as executor:
#         list(tqdm(executor.map(_cache_asn, unique_asrs), total=len(unique_asrs), desc="Caching ASRs"))
            
#     # Execute ASN Caching in Parallel Threads
#     print(f"Caching {len(unique_asns)} unique ASN numbers...")
#     with ThreadPoolExecutor(max_workers=max_workers) as executor:
#         list(tqdm(executor.map(_cache_asr, unique_asns), total=len(unique_asns), desc="Caching ASNs"))

#     end_time = time.time()
#     duration_minutes = (end_time - start_time) / 60
#     print(f"--- API Pre-caching completed in: {duration_minutes:.2f} minutes ---")
#     print(f"Total items cached: {len(asn_cache) + len(asr_cache)}")


# # CASE 1: Looking for Merging Candidates
# def split_case_1_audits(prox_audits_table: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
#     """CASE 1: Candidates require matching, non-null FCC ASR AND FAA ASN."""
#     working_table = prox_audits_table.copy(); working_table = clean_asr_in_dataframe(working_table)
#     candidates_indices = set(); grouped = working_table.groupby('focus_asset_id')
    
#     for focus_asset_id, group in grouped:
#         reference_record_df = group[group['associated_asset_id'].isnull()]
#         associated_records_df = group[group['associated_asset_id'].notnull()]
#         ref_index = reference_record_df.index[0] if len(reference_record_df) == 1 else None

#         if len(reference_record_df) == 1 and ref_index is not None:
#             reference_record = reference_record_df.iloc[0]
#             ref_fcc = reference_record['fcc_asr_number']; ref_faa = reference_record['faa_study_number']
            
#             if pd.notnull(ref_fcc) and pd.notnull(ref_faa):
#                 matching_mask = (associated_records_df['fcc_asr_number'] == ref_fcc) & \
#                                 (associated_records_df['faa_study_number'] == ref_faa)
#                 matching_associated_records = associated_records_df[matching_mask]
                
#                 if not matching_associated_records.empty:
#                     candidates_indices.add(ref_index); candidates_indices.update(matching_associated_records.index)

#     case1_auto_merge_candidates = working_table.loc[list(candidates_indices)].copy()
#     all_original_indices = set(working_table.index)
#     final_post_merge_indices = all_original_indices.difference(candidates_indices)
#     case1_prox_audits_post_auto_merge_table = working_table.loc[list(final_post_merge_indices)].copy()

#     cols = working_table.columns
#     if case1_auto_merge_candidates.empty: case1_auto_merge_candidates = pd.DataFrame(columns=cols)
#     if case1_prox_audits_post_auto_merge_table.empty: case1_prox_audits_post_auto_merge_table = pd.DataFrame(columns=cols)
#     return case1_auto_merge_candidates, case1_prox_audits_post_auto_merge_table

# # CASE 1: Further Filter & Merging Process
# def apply_case_1_full_processing(
#     candidates_table: pd.DataFrame, initial_prox_audits_table: pd.DataFrame
# ) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
#     """Executes Case 1 merging logic. Applies sequential filters and simple FAA scraping."""
#     start_time = time.time(); print("Starting Case 1 processing...")
#     merging_timestamp = datetime.now(); cols = candidates_table.columns
    
#     case1_auto_merge_further_filter_list = []; case1_raw_post_auto_merge_list = []
#     case1_post_auto_merge_list = []; new_prox_audit_records_list = [] 
#     failed_prox_audit_records_list = []; case1_prox_audits_post_auto_merge_table = initial_prox_audits_table.copy()
#     processed_focus_ids = set(); processed_assoc_ids = set()
    
#     if not pd.api.types.is_datetime64_any_dtype(candidates_table['created_at']):
#         candidates_table['created_at'] = pd.to_datetime(candidates_table['created_at'], errors='coerce')
#     candidates_table['created_at_month'] = candidates_table['created_at'].dt.to_period('M')
#     grouped = candidates_table.groupby('focus_asset_id')
    
#     for focus_asset_id, group in tqdm(grouped, desc="Processing Case 1"):
#         reference_record_df = group[group['associated_asset_id'].isnull()]
#         associated_records_df = group[group['associated_asset_id'].notnull()].copy()
        
#         if len(reference_record_df) != 1: continue
            
#         reference_record = reference_record_df.iloc[0]
#         ref_fcc = reference_record['fcc_asr_number']; ref_faa = reference_record['faa_study_number']
#         ref_source = reference_record['source']; ref_year_month = reference_record['created_at_month']
        
#         remaining_associated_records_df = associated_records_df.copy(); indices_to_remove = set()
        
#         if processed_focus_ids:
#             focus_id_match_mask = remaining_associated_records_df['associated_asset_id'].isin(processed_focus_ids)
#             indices_to_remove.update(remaining_associated_records_df[focus_id_match_mask].index)
#         if processed_assoc_ids:
#             assoc_id_match_mask = remaining_associated_records_df['associated_asset_id'].isin(processed_assoc_ids)
#             indices_to_remove.update(remaining_associated_records_df[assoc_id_match_mask].index)

#         remaining_associated_records_df['created_at_month'] = remaining_associated_records_df['created_at'].dt.to_period('M')
#         date_source_match_mask = (remaining_associated_records_df['source'] == ref_source) & \
#                                  (remaining_associated_records_df['created_at_month'] == ref_year_month)
#         indices_to_remove.update(remaining_associated_records_df[date_source_match_mask].index)
        
#         removed_associated_records_df = remaining_associated_records_df.loc[list(indices_to_remove)].copy()
#         remaining_associated_records_df = remaining_associated_records_df.loc[~remaining_associated_records_df.index.isin(indices_to_remove)]
        
#         if 'created_at_month' in removed_associated_records_df.columns: removed_associated_records_df.drop(columns=['created_at_month'], inplace=True)
#         if 'created_at_month' in remaining_associated_records_df.columns: remaining_associated_records_df.drop(columns=['created_at_month'], inplace=True)
#         ref_record_cleaned = reference_record_df.copy()
#         if 'created_at_month' in ref_record_cleaned.columns: ref_record_cleaned.drop(columns=['created_at_month'], inplace=True)
        
#         if not removed_associated_records_df.empty: failed_prox_audit_records_list.append(removed_associated_records_df)
            
#         if len(remaining_associated_records_df) > 0:
#             current_group_to_merge = pd.concat([ref_record_cleaned, remaining_associated_records_df], ignore_index=False)
#             case1_auto_merge_further_filter_list.append(current_group_to_merge)

#             for index, assoc_record in remaining_associated_records_df.iterrows():
#                 final_faa = get_case_1_final_faa(ref_fcc, ref_faa, assoc_record['faa_study_number'])

#                 merged_record_series = merge_records(reference_record, assoc_record, merging_timestamp, faa_study_number=final_faa)
#                 merged_record_df = pd.DataFrame([merged_record_series], columns=cols)
                
#                 new_prox_audit_records_list.append(merged_record_df); case1_post_auto_merge_list.append(merged_record_df)
#                 case1_raw_post_auto_merge_list.append(merged_record_df)
#                 assoc_record_df = pd.DataFrame([assoc_record], columns=cols); case1_raw_post_auto_merge_list.append(assoc_record_df)
#                 ref_record_df_cleaned_raw = ref_record_cleaned.copy(); case1_raw_post_auto_merge_list.append(ref_record_df_cleaned_raw)
                
#                 if pd.notnull(merged_record_series['focus_asset_id']): processed_focus_ids.add(merged_record_series['focus_asset_id'])
#                 if pd.notnull(assoc_record['associated_asset_id']): processed_assoc_ids.add(assoc_record['associated_asset_id'])
                
#         else:
#             failed_prox_audit_records_list.append(ref_record_cleaned)
            
#     # Concatenate all results from lists ONCE at the end
#     if new_prox_audit_records_list:
#         new_records_df = pd.concat(new_prox_audit_records_list, ignore_index=True)
#         case1_prox_audits_post_auto_merge_table = pd.concat([case1_prox_audits_post_auto_merge_table, new_records_df], ignore_index=True)
#     if failed_prox_audit_records_list:
#         failed_records_df = pd.concat(failed_prox_audit_records_list, ignore_index=True)
#         case1_prox_audits_post_auto_merge_table = pd.concat([case1_prox_audits_post_auto_merge_table, failed_records_df], ignore_index=True)
    
#     if case1_auto_merge_further_filter_list: case1_auto_merge_further_filter = pd.concat(case1_auto_merge_further_filter_list, ignore_index=False)
#     else: case1_auto_merge_further_filter = pd.DataFrame(columns=cols)
#     if case1_post_auto_merge_list: case1_post_auto_merge_table = pd.concat(case1_post_auto_merge_list, ignore_index=True)
#     else: case1_post_auto_merge_table = pd.DataFrame(columns=cols)
#     if case1_raw_post_auto_merge_list: case1_raw_post_auto_merge_table = pd.concat(case1_raw_post_auto_merge_list, ignore_index=True)
#     else: case1_raw_post_auto_merge_table = pd.DataFrame(columns=cols)

#     end_time = time.time()
#     duration_minutes = (end_time - start_time) / 60
#     print(f"--- Case 1 Processing completed in: {duration_minutes:.2f} minutes ---")
    
#     return (
#         case1_auto_merge_further_filter, 
#         case1_prox_audits_post_auto_merge_table.drop_duplicates(ignore_index=True),
#         case1_post_auto_merge_table.drop_duplicates(ignore_index=True), 
#         case1_raw_post_auto_merge_table.drop_duplicates(ignore_index=True)
#     )

# # CASE 1: Maintenance Conditions for cleanup and preparation of remaining proximity audits, which will be fed for Case 2.
# def apply_case_1_maintenance_logic(
#     prox_audits_table: pd.DataFrame, post_auto_merge_table: pd.DataFrame, 
#     post_merge_table: pd.DataFrame, raw_post_merge_table: pd.DataFrame
# ) -> Tuple[pd.DataFrame, pd.DataFrame]:
#     """Performs final cleanup, updates associated records, and sorts the output for Case 2."""
#     if post_auto_merge_table.empty:
#         empty_df = pd.DataFrame(columns=prox_audits_table.columns)
#         return empty_df, empty_df
    
#     working_post_merge = post_auto_merge_table.reset_index(drop=True)
#     final_asset_table_list = []
#     associated_records_mask = working_post_merge['associated_asset_id'].notnull()
    
#     # Filter A: Update associated records based on matching focus_asset_id in post_merge_table
#     if not post_merge_table.empty:
#         post_merge_lookup = post_merge_table.set_index('focus_asset_id')
#         all_cols = working_post_merge.columns.tolist()
#         cols_to_exclude = ["audit_reason", "distance_to_reference", "agldiff_to_reference", "associated_asset_id", "index"]
#         cols_to_update = [col for col in all_cols if col not in cols_to_exclude]
#         for idx, assoc_record in working_post_merge[associated_records_mask].iterrows():
#             assoc_asset_id = assoc_record['associated_asset_id']
#             if assoc_asset_id in post_merge_lookup.index:
#                 matching_merged_record = post_merge_lookup.loc[assoc_asset_id]
#                 if isinstance(matching_merged_record, pd.DataFrame): matching_merged_record = matching_merged_record.iloc[0]
#                 for col in cols_to_update:
#                     if col in matching_merged_record.index:
#                         if col in working_post_merge.columns:
#                             working_post_merge.loc[idx, col] = matching_merged_record[col]

#     # Filter B: Remove associated records whose associated_asset_id is in the raw_post_merge_table (fully processed)
#     if not raw_post_merge_table.empty:
#         raw_assoc_ids = set(raw_post_merge_table['associated_asset_id'].dropna())
#         removal_mask = (working_post_merge['associated_asset_id'].notnull()) & \
#                        (working_post_merge['associated_asset_id'].isin(raw_assoc_ids))
#         working_post_merge = working_post_merge[~removal_mask]

#     # Step C: Identify groups with only 1 record (finalized) and move them to final_asset_table
#     valid_groups = working_post_merge['focus_asset_id'].dropna()
#     if not valid_groups.empty:
#         group_sizes = working_post_merge.groupby('focus_asset_id').size()
#         single_record_groups = group_sizes[group_sizes == 1].index
#         final_asset_table_df = working_post_merge[
#             working_post_merge['focus_asset_id'].isin(single_record_groups)
#         ].copy()
#         final_asset_table_list.append(final_asset_table_df)
#         working_post_merge = working_post_merge[
#             ~working_post_merge['focus_asset_id'].isin(single_record_groups)
#         ]

#     # Step D: Sort the remaining records by focus_asset_id and associated_asset_id (NULLs first)
#     sorted_post_merge_table = working_post_merge.sort_values(
#         by=['focus_asset_id', 'associated_asset_id'], 
#         ascending=[True, True],
#         na_position='first'
#     ).reset_index(drop=True)

#     final_asset_table = pd.concat(final_asset_table_list, ignore_index=True)
#     if final_asset_table.empty: final_asset_table = pd.DataFrame(columns=prox_audits_table.columns) 

#     return final_asset_table, sorted_post_merge_table

# # CASE 2: Looking for Merging Candidates
# def split_case_2_audits(case1_sorted_post_merge_table: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
#     """CASE 2: Candidates require matching, non-null FCC ASR, but DIFFERENT, non-null FAA ASN values."""
#     working_table = case1_sorted_post_merge_table.copy(); working_table = clean_asr_in_dataframe(working_table) 
#     candidates_indices = set(); grouped = working_table.groupby('focus_asset_id')

#     for focus_asset_id, group in grouped:
#         reference_record_df = group[group['associated_asset_id'].isnull()]
#         associated_records_df = group[group['associated_asset_id'].notnull()]
#         ref_index = reference_record_df.index[0] if len(reference_record_df) == 1 else None

#         if len(reference_record_df) == 1 and ref_index is not None:
#             reference_record = reference_record_df.iloc[0]
#             ref_fcc = reference_record['fcc_asr_number']; ref_faa = reference_record['faa_study_number']

#             if pd.notnull(ref_fcc) and pd.notnull(ref_faa):
#                 matching_mask = (associated_records_df['fcc_asr_number'] == ref_fcc) & \
#                                 (associated_records_df['faa_study_number'] != ref_faa) & \
#                                 (associated_records_df['fcc_asr_number'].notnull()) & \
#                                 (associated_records_df['faa_study_number'].notnull())
#                 matching_associated_records = associated_records_df[matching_mask]
                
#                 if not matching_associated_records.empty:
#                     candidates_indices.add(ref_index); candidates_indices.update(matching_associated_records.index)

#     case2_auto_merge_candidates = working_table.loc[list(candidates_indices)].copy()
#     all_original_indices = set(working_table.index) 
#     final_post_merge_indices = all_original_indices.difference(candidates_indices)
#     case2_prox_audits_post_auto_merge_table = working_table.loc[list(final_post_merge_indices)].copy()
    
#     cols = working_table.columns
#     if case2_auto_merge_candidates.empty: case2_auto_merge_candidates = pd.DataFrame(columns=cols)
#     if case2_prox_audits_post_auto_merge_table.empty: case2_prox_audits_post_auto_merge_table = pd.DataFrame(columns=cols)

#     return case2_auto_merge_candidates, case2_prox_audits_post_auto_merge_table

# # CASE 2: Further Filter & Merging Process
# def apply_case_2_full_processing(
#     candidates_table: pd.DataFrame,
#     initial_prox_audits_table: pd.DataFrame
# ) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
#     """Executes Case 2 merging logic. Applies expanded filters and complex FAA scraping."""
#     start_time = time.time(); print("Starting Case 2 processing...")
#     merging_timestamp = datetime.now(); cols = candidates_table.columns
#     merge_source_check = f"Auto-Merged {merging_timestamp.strftime('%m/%Y')}"
    
#     # Initialize lists and sets
#     case2_auto_merge_further_filter_list = []; case2_raw_post_auto_merge_list = []
#     case2_post_auto_merge_list = []; new_prox_audit_records_list = []
#     failed_prox_audit_records_list = []; case2_prox_audits_post_auto_merge_table = initial_prox_audits_table.copy()
#     processed_focus_ids = set(); processed_assoc_ids = set() 
    
#     if not pd.api.types.is_datetime64_any_dtype(candidates_table['created_at']):
#         candidates_table['created_at'] = pd.to_datetime(candidates_table['created_at'], errors='coerce')
#     candidates_table['created_at_month'] = candidates_table['created_at'].dt.to_period('M')
#     grouped = candidates_table.groupby('focus_asset_id')
    
#     for focus_asset_id, group in tqdm(grouped, desc="Processing Case 2"):
#         reference_record_df = group[group['associated_asset_id'].isnull()]
#         associated_records_df = group[group['associated_asset_id'].notnull()].copy()
        
#         if len(reference_record_df) != 1: continue
            
#         reference_record = reference_record_df.iloc[0]
#         ref_fcc = reference_record['fcc_asr_number']; ref_faa = reference_record['faa_study_number']
#         ref_source = reference_record['source']; ref_year_month = reference_record['created_at_month']
        
#         remaining_associated_records_df = associated_records_df.copy(); indices_to_remove = set()
        
#         # Filters A, B, C, D, E
#         if processed_focus_ids:
#             focus_id_match_mask = remaining_associated_records_df['associated_asset_id'].isin(processed_focus_ids)
#             indices_to_remove.update(remaining_associated_records_df[focus_id_match_mask].index)
#         if processed_assoc_ids:
#             assoc_id_match_mask = remaining_associated_records_df['associated_asset_id'].isin(processed_assoc_ids)
#             indices_to_remove.update(remaining_associated_records_df[assoc_id_match_mask].index)

#         source_match_mask = (remaining_associated_records_df['source'] == merge_source_check)
#         indices_to_remove.update(remaining_associated_records_df[source_match_mask].index)
        
#         ref_source_match = reference_record['source'] == merge_source_check
        
#         ref_record_cleaned = reference_record_df.copy()
#         if 'created_at_month' in ref_record_cleaned.columns: ref_record_cleaned.drop(columns=['created_at_month'], inplace=True)
        
#         if ref_source_match:
#             group_associated_cleaned = associated_records_df.copy()
#             if 'created_at_month' in group_associated_cleaned.columns: group_associated_cleaned.drop(columns=['created_at_month'], inplace=True)
#             failed_prox_audit_records_list.append(ref_record_cleaned); failed_prox_audit_records_list.append(group_associated_cleaned)
#             continue 

#         remaining_associated_records_df['created_at_month'] = remaining_associated_records_df['created_at'].dt.to_period('M')
#         date_source_match_mask = (remaining_associated_records_df['source'] == ref_source) & \
#                                  (remaining_associated_records_df['created_at_month'] == ref_year_month)
#         indices_to_remove.update(remaining_associated_records_df[date_source_match_mask].index)

#         removed_associated_records_df = remaining_associated_records_df.loc[list(indices_to_remove)].copy()
#         remaining_associated_records_df = remaining_associated_records_df.loc[~remaining_associated_records_df.index.isin(indices_to_remove)]
#         if 'created_at_month' in removed_associated_records_df.columns: removed_associated_records_df.drop(columns=['created_at_month'], inplace=True)
#         if 'created_at_month' in remaining_associated_records_df.columns: remaining_associated_records_df.drop(columns=['created_at_month'], inplace=True)

#         if not removed_associated_records_df.empty: failed_prox_audit_records_list.append(removed_associated_records_df)

#         if len(remaining_associated_records_df) > 0:
#             current_group_to_merge = pd.concat([ref_record_cleaned, remaining_associated_records_df], ignore_index=False)
#             case2_auto_merge_further_filter_list.append(current_group_to_merge)

#             for index, assoc_record in remaining_associated_records_df.iterrows():
#                 final_faa = determine_faa_study_number(ref_fcc, ref_faa, assoc_record['faa_study_number'])

#                 if pd.notnull(final_faa):
#                     merged_record_series = merge_records(reference_record, assoc_record, merging_timestamp, faa_study_number=final_faa)
#                     merged_record_df = pd.DataFrame([merged_record_series], columns=cols)

#                     new_prox_audit_records_list.append(merged_record_df); case2_post_auto_merge_list.append(merged_record_df)
#                     case2_raw_post_auto_merge_list.append(merged_record_df)
#                     assoc_record_df = pd.DataFrame([assoc_record], columns=cols); case2_raw_post_auto_merge_list.append(assoc_record_df)
#                     ref_record_df_cleaned_raw = ref_record_cleaned.copy(); case2_raw_post_auto_merge_list.append(ref_record_df_cleaned_raw)
                    
#                     if pd.notnull(merged_record_series['focus_asset_id']): processed_focus_ids.add(merged_record_series['focus_asset_id'])
#                     if pd.notnull(assoc_record['associated_asset_id']): processed_assoc_ids.add(assoc_record['associated_asset_id'])

#                 else:
#                     assoc_record_df = pd.DataFrame([assoc_record], columns=cols)
#                     failed_prox_audit_records_list.append(ref_record_cleaned); failed_prox_audit_records_list.append(assoc_record_df)
                    
#         else:
#             failed_prox_audit_records_list.append(ref_record_cleaned)
            
#     # Concatenate all results from lists ONCE at the end
#     if new_prox_audit_records_list:
#         new_records_df = pd.concat(new_prox_audit_records_list, ignore_index=True)
#         case2_prox_audits_post_auto_merge_table = pd.concat([case2_prox_audits_post_auto_merge_table, new_records_df], ignore_index=True)
#     if failed_prox_audit_records_list:
#         failed_records_df = pd.concat(failed_prox_audit_records_list, ignore_index=True)
#         case2_prox_audits_post_auto_merge_table = pd.concat([case2_prox_audits_post_auto_merge_table, failed_records_df], ignore_index=True)
    
#     if case2_auto_merge_further_filter_list: case2_auto_merge_further_filter = pd.concat(case2_auto_merge_further_filter_list, ignore_index=False)
#     else: case2_auto_merge_further_filter = pd.DataFrame(columns=cols)
#     if case2_post_auto_merge_list: case2_post_auto_merge_table = pd.concat(case2_post_auto_merge_list, ignore_index=True)
#     else: case2_post_auto_merge_table = pd.DataFrame(columns=cols)
#     if case2_raw_post_auto_merge_list: case2_raw_post_auto_merge_table = pd.concat(case2_raw_post_auto_merge_list, ignore_index=True)
#     else: case2_raw_post_auto_merge_table = pd.DataFrame(columns=cols)

#     end_time = time.time()
#     duration_minutes = (end_time - start_time) / 60
#     print(f"--- Case 2 Processing completed in: {duration_minutes:.2f} minutes ---")
    
#     return (
#         case2_auto_merge_further_filter, 
#         case2_prox_audits_post_auto_merge_table.drop_duplicates(ignore_index=True),
#         case2_post_auto_merge_table.drop_duplicates(ignore_index=True), 
#         case2_raw_post_auto_merge_table.drop_duplicates(ignore_index=True)
#     )

# # CASE 2: Maintenance Conditions for cleanup and preparation of remaining proximity audits, which will be fed for Case 3.
# def apply_case_2_maintenance_logic(
#     prox_audits_table: pd.DataFrame, post_auto_merge_table: pd.DataFrame, 
#     post_merge_table: pd.DataFrame, raw_post_merge_table: pd.DataFrame,
#     running_final_asset_table: pd.DataFrame
# ) -> Tuple[pd.DataFrame, pd.DataFrame]:
#     """Performs final cleanup for Case 2. Updates associated records, and aggregates output for Case 3."""
#     if post_auto_merge_table.empty:
#         empty_df = pd.DataFrame(columns=prox_audits_table.columns)
#         return running_final_asset_table, empty_df
    
#     working_post_merge = post_auto_merge_table.reset_index(drop=True)
#     final_asset_table_list = []
#     associated_records_mask = working_post_merge['associated_asset_id'].notnull()
    
#     if not post_merge_table.empty:
#         post_merge_lookup = post_merge_table.set_index('focus_asset_id')
#         all_cols = working_post_merge.columns.tolist()
#         cols_to_exclude = ["audit_reason", "distance_to_reference", "agldiff_to_reference", "associated_asset_id", "index"]
#         cols_to_update = [col for col in all_cols if col not in cols_to_exclude]
#         for idx, assoc_record in working_post_merge[associated_records_mask].iterrows():
#             assoc_asset_id = assoc_record['associated_asset_id']
#             if assoc_asset_id in post_merge_lookup.index:
#                 matching_merged_record = post_merge_lookup.loc[assoc_asset_id]
#                 if isinstance(matching_merged_record, pd.DataFrame): matching_merged_record = matching_merged_record.iloc[0]
#                 for col in cols_to_update:
#                     if col in matching_merged_record.index:
#                         if col in working_post_merge.columns:
#                             working_post_merge.loc[idx, col] = matching_merged_record[col]

#     if not raw_post_merge_table.empty:
#         raw_assoc_ids = set(raw_post_merge_table['associated_asset_id'].dropna())
#         removal_mask = (working_post_merge['associated_asset_id'].notnull()) & \
#                        (working_post_merge['associated_asset_id'].isin(raw_assoc_ids))
#         working_post_merge = working_post_merge[~removal_mask]

#     valid_groups = working_post_merge['focus_asset_id'].dropna()
#     if not valid_groups.empty:
#         group_sizes = working_post_merge.groupby('focus_asset_id').size()
#         single_record_groups = group_sizes[group_sizes == 1].index
#         final_asset_table_df = working_post_merge[
#             working_post_merge['focus_asset_id'].isin(single_record_groups)
#         ].copy()
#         final_asset_table_list.append(final_asset_table_df)
#         working_post_merge = working_post_merge[
#             ~working_post_merge['focus_asset_id'].isin(single_record_groups)
#         ]

#     sorted_post_merge_table = working_post_merge.sort_values(
#         by=['focus_asset_id', 'associated_asset_id'], 
#         ascending=[True, True],
#         na_position='first'
#     ).reset_index(drop=True)

#     case2_final_assets = pd.concat(final_asset_table_list, ignore_index=True)
#     if case2_final_assets.empty: case2_final_assets = pd.DataFrame(columns=prox_audits_table.columns) 
        
#     aggregated_final_asset_table = pd.concat([running_final_asset_table, case2_final_assets], ignore_index=True)

#     return aggregated_final_asset_table, sorted_post_merge_table

# # CASE 3: Looking for Merging Candidates
# def split_case_3_audits(case2_sorted_post_merge_table: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
#     """CASE 3: Candidates require matching, non-null FCC ASR, but Ref NULL XOR Assoc NULL FAA ASN."""
#     working_table = case2_sorted_post_merge_table.copy(); working_table = clean_asr_in_dataframe(working_table) 
#     candidates_indices = set(); grouped = working_table.groupby('focus_asset_id')

#     for focus_asset_id, group in grouped:
#         reference_record_df = group[group['associated_asset_id'].isnull()]
#         associated_records_df = group[group['associated_asset_id'].notnull()]
#         ref_index = reference_record_df.index[0] if len(reference_record_df) == 1 else None

#         if len(reference_record_df) == 1 and ref_index is not None:
#             reference_record = reference_record_df.iloc[0]
#             ref_fcc = reference_record['fcc_asr_number']; ref_faa = reference_record['faa_study_number']

#             if pd.notnull(ref_fcc):
#                 mask_ref_null = pd.isnull(ref_faa); mask_assoc_not_null = associated_records_df['faa_study_number'].notnull()
#                 mask_ref_not_null = pd.notnull(ref_faa); mask_assoc_null = associated_records_df['faa_study_number'].isnull()
                
#                 matching_mask = (associated_records_df['fcc_asr_number'] == ref_fcc) & \
#                                 (associated_records_df['fcc_asr_number'].notnull()) & \
#                                 (
#                                     (mask_ref_null & mask_assoc_not_null) | 
#                                     (mask_ref_not_null & mask_assoc_null)
#                                 )
                
#                 matching_associated_records = associated_records_df[matching_mask]
                
#                 if not matching_associated_records.empty:
#                     candidates_indices.add(ref_index); candidates_indices.update(matching_associated_records.index)

#     case3_auto_merge_candidates = working_table.loc[list(candidates_indices)].copy()
#     all_original_indices = set(working_table.index) 
#     final_post_merge_indices = all_original_indices.difference(candidates_indices)
#     case3_prox_audits_post_auto_merge_table = working_table.loc[list(final_post_merge_indices)].copy()
    
#     cols = working_table.columns
#     if case3_auto_merge_candidates.empty: case3_auto_merge_candidates = pd.DataFrame(columns=cols)
#     if case3_prox_audits_post_auto_merge_table.empty: case3_prox_audits_post_auto_merge_table = pd.DataFrame(columns=cols)

#     return case3_auto_merge_candidates, case3_prox_audits_post_auto_merge_table

# # CASE 3: Further Filter & Merging Process
# def apply_case_3_full_processing(
#     candidates_table: pd.DataFrame,
#     initial_prox_audits_table: pd.DataFrame
# ) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
#     """Executes Case 3 merging logic. Identical to Case 2: applies expanded filters and complex FAA scraping."""
#     start_time = time.time(); print("Starting Case 3 processing...")
#     merging_timestamp = datetime.now(); cols = candidates_table.columns
#     merge_source_check = f"Auto-Merged {merging_timestamp.strftime('%m/%Y')}"
    
#     # Initialize lists and sets
#     case3_auto_merge_further_filter_list = []; case3_raw_post_auto_merge_list = []
#     case3_post_auto_merge_list = []; new_prox_audit_records_list = []
#     failed_prox_audit_records_list = []; case3_prox_audits_post_auto_merge_table = initial_prox_audits_table.copy()
#     processed_focus_ids = set(); processed_assoc_ids = set() 
    
#     if not pd.api.types.is_datetime64_any_dtype(candidates_table['created_at']):
#         candidates_table['created_at'] = pd.to_datetime(candidates_table['created_at'], errors='coerce')
#     candidates_table['created_at_month'] = candidates_table['created_at'].dt.to_period('M')
#     grouped = candidates_table.groupby('focus_asset_id')
    
#     for focus_asset_id, group in tqdm(grouped, desc="Processing Case 3"):
#         reference_record_df = group[group['associated_asset_id'].isnull()]
#         associated_records_df = group[group['associated_asset_id'].notnull()].copy()
        
#         if len(reference_record_df) != 1: continue
            
#         reference_record = reference_record_df.iloc[0]
#         ref_fcc = reference_record['fcc_asr_number']; ref_faa = reference_record['faa_study_number']
#         ref_source = reference_record['source']; ref_year_month = reference_record['created_at_month']
        
#         remaining_associated_records_df = associated_records_df.copy(); indices_to_remove = set()
        
#         # Filters A, B, C, D, E logic is identical to Case 2
#         if processed_focus_ids:
#             focus_id_match_mask = remaining_associated_records_df['associated_asset_id'].isin(processed_focus_ids)
#             indices_to_remove.update(remaining_associated_records_df[focus_id_match_mask].index)
#         if processed_assoc_ids:
#             assoc_id_match_mask = remaining_associated_records_df['associated_asset_id'].isin(processed_assoc_ids)
#             indices_to_remove.update(remaining_associated_records_df[assoc_id_match_mask].index)

#         source_match_mask = (remaining_associated_records_df['source'] == merge_source_check)
#         indices_to_remove.update(remaining_associated_records_df[source_match_mask].index)
        
#         ref_source_match = reference_record['source'] == merge_source_check
        
#         ref_record_cleaned = reference_record_df.copy()
#         if 'created_at_month' in ref_record_cleaned.columns: ref_record_cleaned.drop(columns=['created_at_month'], inplace=True)
        
#         if ref_source_match:
#             group_associated_cleaned = associated_records_df.copy()
#             if 'created_at_month' in group_associated_cleaned.columns: group_associated_cleaned.drop(columns=['created_at_month'], inplace=True)
#             failed_prox_audit_records_list.append(ref_record_cleaned); failed_prox_audit_records_list.append(group_associated_cleaned)
#             continue 

#         remaining_associated_records_df['created_at_month'] = remaining_associated_records_df['created_at'].dt.to_period('M')
#         date_source_match_mask = (remaining_associated_records_df['source'] == ref_source) & \
#                                  (remaining_associated_records_df['created_at_month'] == ref_year_month)
#         indices_to_remove.update(remaining_associated_records_df[date_source_match_mask].index)

#         removed_associated_records_df = remaining_associated_records_df.loc[list(indices_to_remove)].copy()
#         remaining_associated_records_df = remaining_associated_records_df.loc[~remaining_associated_records_df.index.isin(indices_to_remove)]
#         if 'created_at_month' in removed_associated_records_df.columns: removed_associated_records_df.drop(columns=['created_at_month'], inplace=True)
#         if 'created_at_month' in remaining_associated_records_df.columns: remaining_associated_records_df.drop(columns=['created_at_month'], inplace=True)

#         if not removed_associated_records_df.empty: failed_prox_audit_records_list.append(removed_associated_records_df)

        
#         if len(remaining_associated_records_df) > 0:
#             current_group_to_merge = pd.concat([ref_record_cleaned, remaining_associated_records_df], ignore_index=False)
#             case3_auto_merge_further_filter_list.append(current_group_to_merge)

#             for index, assoc_record in remaining_associated_records_df.iterrows():
#                 final_faa = determine_faa_study_number(ref_fcc, ref_faa, assoc_record['faa_study_number'])

#                 if pd.notnull(final_faa):
#                     merged_record_series = merge_records(reference_record, assoc_record, merging_timestamp, faa_study_number=final_faa)
#                     merged_record_df = pd.DataFrame([merged_record_series], columns=cols)

#                     new_prox_audit_records_list.append(merged_record_df); case3_post_auto_merge_list.append(merged_record_df)
#                     case3_raw_post_auto_merge_list.append(merged_record_df)
#                     assoc_record_df = pd.DataFrame([assoc_record], columns=cols); case3_raw_post_auto_merge_list.append(assoc_record_df)
#                     ref_record_df_cleaned_raw = ref_record_cleaned.copy(); case3_raw_post_auto_merge_list.append(ref_record_df_cleaned_raw)
                    
#                     if pd.notnull(merged_record_series['focus_asset_id']): processed_focus_ids.add(merged_record_series['focus_asset_id'])
#                     if pd.notnull(assoc_record['associated_asset_id']): processed_assoc_ids.add(assoc_record['associated_asset_id'])

#                 else:
#                     assoc_record_df = pd.DataFrame([assoc_record], columns=cols)
#                     failed_prox_audit_records_list.append(ref_record_cleaned); failed_prox_audit_records_list.append(assoc_record_df)
                    
#         else:
#             failed_prox_audit_records_list.append(ref_record_cleaned)
            
#     # Concatenate all results from lists ONCE at the end
#     if new_prox_audit_records_list:
#         new_records_df = pd.concat(new_prox_audit_records_list, ignore_index=True)
#         case3_prox_audits_post_auto_merge_table = pd.concat([case3_prox_audits_post_auto_merge_table, new_records_df], ignore_index=True)
#     if failed_prox_audit_records_list:
#         failed_records_df = pd.concat(failed_prox_audit_records_list, ignore_index=True)
#         case3_prox_audits_post_auto_merge_table = pd.concat([case3_prox_audits_post_auto_merge_table, failed_records_df], ignore_index=True)
    
#     if case3_auto_merge_further_filter_list: case3_auto_merge_further_filter = pd.concat(case3_auto_merge_further_filter_list, ignore_index=False)
#     else: case3_auto_merge_further_filter = pd.DataFrame(columns=cols)
#     if case3_post_auto_merge_list: case3_post_auto_merge_table = pd.concat(case3_post_auto_merge_list, ignore_index=True)
#     else: case3_post_auto_merge_table = pd.DataFrame(columns=cols)
#     if case3_raw_post_auto_merge_list: case3_raw_post_auto_merge_table = pd.concat(case3_raw_post_auto_merge_list, ignore_index=True)
#     else: case3_raw_post_auto_merge_table = pd.DataFrame(columns=cols)

#     end_time = time.time()
#     duration_minutes = (end_time - start_time) / 60
#     print(f"--- Case 3 Processing completed in: {duration_minutes:.2f} minutes ---")
    
#     return (
#         case3_auto_merge_further_filter, 
#         case3_prox_audits_post_auto_merge_table.drop_duplicates(ignore_index=True),
#         case3_post_auto_merge_table.drop_duplicates(ignore_index=True), 
#         case3_raw_post_auto_merge_table.drop_duplicates(ignore_index=True)
#     )

# # CASE 3: Maintenance Conditions for cleanup and preparation of remaining proximity audits, which will be fed for Case 4.
# def apply_case_3_maintenance_logic(
#     prox_audits_table: pd.DataFrame, post_auto_merge_table: pd.DataFrame, 
#     post_merge_table: pd.DataFrame, raw_post_merge_table: pd.DataFrame,
#     running_final_asset_table: pd.DataFrame
# ) -> Tuple[pd.DataFrame, pd.DataFrame]:
#     """Performs final cleanup for Case 3. Updates associated records, and aggregates output for Case 4."""
#     if post_auto_merge_table.empty:
#         empty_df = pd.DataFrame(columns=prox_audits_table.columns)
#         return running_final_asset_table, empty_df
    
#     working_post_merge = post_auto_merge_table.reset_index(drop=True)
#     final_asset_table_list = []
#     associated_records_mask = working_post_merge['associated_asset_id'].notnull()
    
#     if not post_merge_table.empty:
#         post_merge_lookup = post_merge_table.set_index('focus_asset_id')
#         all_cols = working_post_merge.columns.tolist()
#         cols_to_exclude = ["audit_reason", "distance_to_reference", "agldiff_to_reference", "associated_asset_id", "index"]
#         cols_to_update = [col for col in all_cols if col not in cols_to_exclude]
#         for idx, assoc_record in working_post_merge[associated_records_mask].iterrows():
#             assoc_asset_id = assoc_record['associated_asset_id']
#             if assoc_asset_id in post_merge_lookup.index:
#                 matching_merged_record = post_merge_lookup.loc[assoc_asset_id]
#                 if isinstance(matching_merged_record, pd.DataFrame): matching_merged_record = matching_merged_record.iloc[0]
#                 for col in cols_to_update:
#                     if col in matching_merged_record.index:
#                         if col in working_post_merge.columns:
#                             working_post_merge.loc[idx, col] = matching_merged_record[col]

#     if not raw_post_merge_table.empty:
#         raw_assoc_ids = set(raw_post_merge_table['associated_asset_id'].dropna())
#         removal_mask = (working_post_merge['associated_asset_id'].notnull()) & \
#                        (working_post_merge['associated_asset_id'].isin(raw_assoc_ids))
#         working_post_merge = working_post_merge[~removal_mask]

#     valid_groups = working_post_merge['focus_asset_id'].dropna()
#     if not valid_groups.empty:
#         group_sizes = working_post_merge.groupby('focus_asset_id').size()
#         single_record_groups = group_sizes[group_sizes == 1].index
#         final_asset_table_df = working_post_merge[
#             working_post_merge['focus_asset_id'].isin(single_record_groups)
#         ].copy()
#         final_asset_table_list.append(final_asset_table_df)
#         working_post_merge = working_post_merge[
#             ~working_post_merge['focus_asset_id'].isin(single_record_groups)
#         ]

#     sorted_post_merge_table = working_post_merge.sort_values(
#         by=['focus_asset_id', 'associated_asset_id'], 
#         ascending=[True, True],
#         na_position='first'
#     ).reset_index(drop=True)

#     case3_final_assets = pd.concat(final_asset_table_list, ignore_index=True)
#     if case3_final_assets.empty: case3_final_assets = pd.DataFrame(columns=prox_audits_table.columns) 
        
#     aggregated_final_asset_table = pd.concat([running_final_asset_table, case3_final_assets], ignore_index=True)

#     return aggregated_final_asset_table, sorted_post_merge_table

# # CASE 4: Looking for Merging Candidates
# def split_case_4_audits(case3_sorted_post_merge_table: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
#     """CASE 4: Candidates require Ref FCC/FAA NULL, Assoc FCC/FAA NOT NULL, AND a 5-way operational match."""
#     working_table = case3_sorted_post_merge_table.copy(); working_table = clean_asr_in_dataframe(working_table) 
#     candidates_indices = set(); grouped = working_table.groupby('focus_asset_id')

#     for focus_asset_id, group in grouped:
#         reference_record_df = group[group['associated_asset_id'].isnull()]
#         associated_records_df = group[group['associated_asset_id'].notnull()]
#         ref_index = reference_record_df.index[0] if len(reference_record_df) == 1 else None

#         if len(reference_record_df) == 1 and ref_index is not None:
#             reference_record = reference_record_df.iloc[0]
            
#             if pd.isnull(reference_record['fcc_asr_number']) and pd.isnull(reference_record['faa_study_number']):
                
#                 ref_op_name = reference_record['operator_name']; ref_op_site_id = reference_record['operator_site_id']
#                 ref_asset_status = reference_record['asset_status']; ref_type = reference_record['type']
#                 ref_name = reference_record['name']

#                 mask_assoc_not_null = (associated_records_df['fcc_asr_number'].notnull()) & \
#                                       (associated_records_df['faa_study_number'].notnull())
                
#                 mask_field_match = (associated_records_df['operator_name'] == ref_op_name) & \
#                                    (associated_records_df['operator_site_id'] == ref_op_site_id) & \
#                                    (associated_records_df['asset_status'] == ref_asset_status) & \
#                                    (associated_records_df['type'] == ref_type) & \
#                                    (associated_records_df['name'] == ref_name)
                
#                 final_matching_mask = mask_assoc_not_null & mask_field_match
#                 matching_associated_records = associated_records_df[final_matching_mask]
                
#                 if not matching_associated_records.empty:
#                     candidates_indices.add(ref_index); candidates_indices.update(matching_associated_records.index)

#     case4_auto_merge_candidates = working_table.loc[list(candidates_indices)].copy()
#     all_original_indices = set(working_table.index) 
#     final_post_merge_indices = all_original_indices.difference(candidates_indices)
#     case4_prox_audits_post_auto_merge_table = working_table.loc[list(final_post_merge_indices)].copy()

#     cols = working_table.columns
#     if case4_auto_merge_candidates.empty: case4_auto_merge_candidates = pd.DataFrame(columns=cols)
#     if case4_prox_audits_post_auto_merge_table.empty: case4_prox_audits_post_auto_merge_table = pd.DataFrame(columns=cols)

#     return case4_auto_merge_candidates, case4_prox_audits_post_auto_merge_table

# # CASE 4: Further Filter & Merging Process
# def apply_case_4_full_processing(
#     candidates_table: pd.DataFrame,
#     initial_prox_audits_table: pd.DataFrame
# ) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
#     """Executes Case 4 merging logic. Applies standard filters PLUS AGL/Distance logic."""
#     start_time = time.time(); print("Starting Case 4 processing...")
#     merging_timestamp = datetime.now(); cols = candidates_table.columns
#     merge_source_check = f"Auto-Merged {merging_timestamp.strftime('%m/%Y')}"
    
#     # Initialize lists and sets
#     case4_auto_merge_further_filter_list = []; case4_raw_post_auto_merge_list = []
#     case4_post_auto_merge_list = []; new_prox_audit_records_list = []
#     failed_prox_audit_records_list = []; case4_prox_audits_post_auto_merge_table = initial_prox_audits_table.copy()
#     processed_focus_ids = set(); processed_assoc_ids = set()
    
#     if not pd.api.types.is_datetime64_any_dtype(candidates_table['created_at']):
#         candidates_table['created_at'] = pd.to_datetime(candidates_table['created_at'], errors='coerce')
#     candidates_table['created_at_month'] = candidates_table['created_at'].dt.to_period('M')
#     grouped = candidates_table.groupby('focus_asset_id')
    
#     for focus_asset_id, group in tqdm(grouped, desc="Processing Case 4"):
#         reference_record_df = group[group['associated_asset_id'].isnull()]
#         associated_records_df = group[group['associated_asset_id'].notnull()].copy()
        
#         if len(reference_record_df) != 1: continue
            
#         reference_record = reference_record_df.iloc[0]
#         ref_source = reference_record['source']; ref_year_month = reference_record['created_at_month']
#         ref_fcc = reference_record['fcc_asr_number']; ref_faa = reference_record['faa_study_number']
        
#         remaining_associated_records_df = associated_records_df.copy(); indices_to_remove = set()
        

#         if processed_focus_ids:
#             focus_id_match_mask = remaining_associated_records_df['associated_asset_id'].isin(processed_focus_ids)
#             indices_to_remove.update(remaining_associated_records_df[focus_id_match_mask].index)
#         if processed_assoc_ids:
#             assoc_id_match_mask = remaining_associated_records_df['associated_asset_id'].isin(processed_assoc_ids)
#             indices_to_remove.update(remaining_associated_records_df[assoc_id_match_mask].index)

#         source_match_mask = (remaining_associated_records_df['source'] == merge_source_check)
#         indices_to_remove.update(remaining_associated_records_df[source_match_mask].index)
        
#         ref_source_match = reference_record['source'] == merge_source_check
        
#         ref_record_cleaned = reference_record_df.copy()
#         if 'created_at_month' in ref_record_cleaned.columns: ref_record_cleaned.drop(columns=['created_at_month'], inplace=True)
        
#         if ref_source_match:
#             group_associated_cleaned = associated_records_df.copy()
#             if 'created_at_month' in group_associated_cleaned.columns: group_associated_cleaned.drop(columns=['created_at_month'], inplace=True)
#             failed_prox_audit_records_list.append(ref_record_cleaned); failed_prox_audit_records_list.append(group_associated_cleaned)
#             continue 

#         remaining_associated_records_df['created_at_month'] = remaining_associated_records_df['created_at'].dt.to_period('M')
#         date_source_match_mask = (remaining_associated_records_df['source'] == ref_source) & \
#                                  (remaining_associated_records_df['created_at_month'] == ref_year_month)
#         indices_to_remove.update(remaining_associated_records_df[date_source_match_mask].index)
        
#         removed_after_abcde_df = remaining_associated_records_df.loc[list(indices_to_remove)].copy()
        

#         remaining_associated_records_df = remaining_associated_records_df.loc[
#             ~remaining_associated_records_df.index.isin(indices_to_remove)
#         ].copy() 

#         # AGL Percentage Difference (>25% results in removal)
#         indices_to_remove_f = set()
#         if not remaining_associated_records_df.empty:
#             ref_agl = pd.to_numeric(reference_record['agl'], errors='coerce')
#             if pd.notnull(ref_agl):
#                 assoc_agl_series = pd.to_numeric(remaining_associated_records_df['agl'], errors='coerce')
#                 diff = ((ref_agl - assoc_agl_series).abs() / assoc_agl_series) * 100
#                 diff_filled = diff.fillna(0).replace([np.inf, -np.inf], 999) 
#                 agl_diff_mask = diff_filled > 25
#                 indices_to_remove_f.update(remaining_associated_records_df[agl_diff_mask].index)
        
#         removed_after_f_df = remaining_associated_records_df.loc[list(indices_to_remove_f)].copy()
        

#         remaining_associated_records_df = remaining_associated_records_df.loc[
#             ~remaining_associated_records_df.index.isin(indices_to_remove_f)
#         ].copy() 

#         # Least Distance Logic (Tie-breaker)
#         removed_after_g_df = pd.DataFrame(columns=remaining_associated_records_df.columns)
#         if len(remaining_associated_records_df) > 1:
#             remaining_associated_records_df['abs_agldiff'] = pd.to_numeric(
#                 remaining_associated_records_df['agldiff_to_reference'], errors='coerce'
#             ).abs()
            
#             if not remaining_associated_records_df['abs_agldiff'].isnull().all():
#                 closest_record_index = remaining_associated_records_df['abs_agldiff'].idxmin()
#                 closest_record_df = remaining_associated_records_df.loc[[closest_record_index]]
#                 removed_after_g_df = remaining_associated_records_df.loc[
#                     ~remaining_associated_records_df.index.isin([closest_record_index])
#                 ]
#                 remaining_associated_records_df = closest_record_df
#             else:
#                 removed_after_g_df = remaining_associated_records_df.copy()
#                 remaining_associated_records_df = pd.DataFrame(columns=remaining_associated_records_df.columns)
        
#         if 'abs_agldiff' in remaining_associated_records_df.columns:
#             remaining_associated_records_df = remaining_associated_records_df.drop(columns=['abs_agldiff'])
        
#         all_removed_associated_records_df = pd.concat([removed_after_abcde_df, removed_after_f_df, removed_after_g_df], ignore_index=False)
#         if 'created_at_month' in all_removed_associated_records_df.columns: all_removed_associated_records_df.drop(columns=['created_at_month'], inplace=True)
#         if 'created_at_month' in remaining_associated_records_df.columns: remaining_associated_records_df.drop(columns=['created_at_month'], inplace=True)
                
#         if not all_removed_associated_records_df.empty: failed_prox_audit_records_list.append(all_removed_associated_records_df)
        
#         # Final Check and Merge Logic
#         if len(remaining_associated_records_df) == 1:
#             current_group_to_merge = pd.concat([ref_record_cleaned, remaining_associated_records_df], ignore_index=False)
#             case4_auto_merge_further_filter_list.append(current_group_to_merge)
#             assoc_record = remaining_associated_records_df.iloc[0]
            
#             final_faa = determine_faa_study_number(
#                 assoc_record['fcc_asr_number'],    
#                 ref_faa, 
#                 assoc_record['faa_study_number']
#             )

#             if pd.notnull(final_faa):
#                 merged_record_series = merge_records(reference_record, assoc_record, merging_timestamp, faa_study_number=final_faa)
#                 merged_record_df = pd.DataFrame([merged_record_series], columns=cols)
                
#                 new_prox_audit_records_list.append(merged_record_df); case4_post_auto_merge_list.append(merged_record_df)
#                 case4_raw_post_auto_merge_list.append(merged_record_df)
#                 assoc_record_df = pd.DataFrame([assoc_record], columns=cols); case4_raw_post_auto_merge_list.append(assoc_record_df)
#                 ref_record_df_cleaned_raw = ref_record_cleaned.copy(); case4_raw_post_auto_merge_list.append(ref_record_df_cleaned_raw)

#                 if pd.notnull(merged_record_series['focus_asset_id']): processed_focus_ids.add(merged_record_series['focus_asset_id'])
#                 if pd.notnull(assoc_record['associated_asset_id']): processed_assoc_ids.add(assoc_record['associated_asset_id'])
            
#             else:
#                 assoc_record_df = pd.DataFrame([assoc_record], columns=cols)
#                 failed_prox_audit_records_list.append(ref_record_cleaned); failed_prox_audit_records_list.append(assoc_record_df)
#         else:
#             failed_prox_audit_records_list.append(ref_record_cleaned)
            
#     # Concatenate all results from lists ONCE at the end
#     if new_prox_audit_records_list:
#         new_records_df = pd.concat(new_prox_audit_records_list, ignore_index=True)
#         case4_prox_audits_post_auto_merge_table = pd.concat([case4_prox_audits_post_auto_merge_table, new_records_df], ignore_index=True)
#     if failed_prox_audit_records_list:
#         failed_records_df = pd.concat(failed_prox_audit_records_list, ignore_index=True)
#         case4_prox_audits_post_auto_merge_table = pd.concat([case4_prox_audits_post_auto_merge_table, failed_records_df], ignore_index=True)
    
#     if case4_auto_merge_further_filter_list: case4_auto_merge_further_filter = pd.concat(case4_auto_merge_further_filter_list, ignore_index=False)
#     else: case4_auto_merge_further_filter = pd.DataFrame(columns=cols)
#     if case4_post_auto_merge_list: case4_post_auto_merge_table = pd.concat(case4_post_auto_merge_list, ignore_index=True)
#     else: case4_post_auto_merge_table = pd.DataFrame(columns=cols)
#     if case4_raw_post_auto_merge_list: case4_raw_post_auto_merge_table = pd.concat(case4_raw_post_auto_merge_list, ignore_index=True)
#     else: case4_raw_post_auto_merge_table = pd.DataFrame(columns=cols)

#     end_time = time.time()
#     duration_minutes = (end_time - start_time) / 60
#     print(f"--- Case 4 Processing completed in: {duration_minutes:.2f} minutes ---")
    
#     return (
#         case4_auto_merge_further_filter, 
#         case4_prox_audits_post_auto_merge_table.drop_duplicates(ignore_index=True),
#         case4_post_auto_merge_table.drop_duplicates(ignore_index=True), 
#         case4_raw_post_auto_merge_table.drop_duplicates(ignore_index=True)
#     )

# # CASE 4: Maintenance Conditions for cleanup and preparation of remaining proximity audits, which will be fed for Case 5.
# def apply_case_4_maintenance_logic(
#     prox_audits_table: pd.DataFrame, post_auto_merge_table: pd.DataFrame, 
#     post_merge_table: pd.DataFrame, raw_post_merge_table: pd.DataFrame,
#     running_final_asset_table: pd.DataFrame
# ) -> Tuple[pd.DataFrame, pd.DataFrame]:
#     """Performs final cleanup for Case 4. Updates associated records, and aggregates output for Case 5."""
#     if post_auto_merge_table.empty:
#         empty_df = pd.DataFrame(columns=prox_audits_table.columns)
#         return running_final_asset_table, empty_df
        
#     working_post_merge = post_auto_merge_table.reset_index(drop=True)
#     final_asset_table_list = []
#     associated_records_mask = working_post_merge['associated_asset_id'].notnull()
    
#     if not post_merge_table.empty:
#         post_merge_lookup = post_merge_table.set_index('focus_asset_id')
#         all_cols = working_post_merge.columns.tolist()
#         cols_to_exclude = ["audit_reason", "distance_to_reference", "agldiff_to_reference", "associated_asset_id", "index"]
#         cols_to_update = [col for col in all_cols if col not in cols_to_exclude]
#         for idx, assoc_record in working_post_merge[associated_records_mask].iterrows():
#             assoc_asset_id = assoc_record['associated_asset_id']
#             if assoc_asset_id in post_merge_lookup.index:
#                 matching_merged_record = post_merge_lookup.loc[assoc_asset_id]
#                 if isinstance(matching_merged_record, pd.DataFrame): matching_merged_record = matching_merged_record.iloc[0]
#                 for col in cols_to_update:
#                     if col in matching_merged_record.index:
#                         if col in working_post_merge.columns:
#                             working_post_merge.loc[idx, col] = matching_merged_record[col]

#     if not raw_post_merge_table.empty:
#         raw_assoc_ids = set(raw_post_merge_table['associated_asset_id'].dropna())
#         removal_mask = (working_post_merge['associated_asset_id'].notnull()) & \
#                        (working_post_merge['associated_asset_id'].isin(raw_assoc_ids))
#         working_post_merge = working_post_merge[~removal_mask]

#     valid_groups = working_post_merge['focus_asset_id'].dropna()
#     if not valid_groups.empty:
#         group_sizes = working_post_merge.groupby('focus_asset_id').size()
#         single_record_groups = group_sizes[group_sizes == 1].index
#         final_asset_table_df = working_post_merge[
#             working_post_merge['focus_asset_id'].isin(single_record_groups)
#         ].copy()
#         final_asset_table_list.append(final_asset_table_df)
#         working_post_merge = working_post_merge[
#             ~working_post_merge['focus_asset_id'].isin(single_record_groups)
#         ]

#     sorted_post_merge_table = working_post_merge.sort_values(
#         by=['focus_asset_id', 'associated_asset_id'], 
#         ascending=[True, True],
#         na_position='first'
#     ).reset_index(drop=True)

#     case4_final_assets = pd.concat(final_asset_table_list, ignore_index=True)
#     if case4_final_assets.empty: case4_final_assets = pd.DataFrame(columns=prox_audits_table.columns) 
        
#     aggregated_final_asset_table = pd.concat([running_final_asset_table, case4_final_assets], ignore_index=True)

#     return aggregated_final_asset_table, sorted_post_merge_table

# # CASE 5: Looking for Merging Candidates
# def split_case_5_audits(case4_sorted_post_merge_table: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
#     """CASE 5: Candidates require Ref/Assoc records to be both NULL for FCC/FAA AND match on 5 key identifying fields."""
#     working_table = case4_sorted_post_merge_table.copy(); working_table = clean_asr_in_dataframe(working_table)
#     candidates_indices = set(); grouped = working_table.groupby('focus_asset_id')

#     for focus_asset_id, group in grouped:
#         reference_record_df = group[group['associated_asset_id'].isnull()]
#         associated_records_df = group[group['associated_asset_id'].notnull()]
#         ref_index = reference_record_df.index[0] if len(reference_record_df) == 1 else None

#         if len(reference_record_df) == 1 and ref_index is not None:
#             reference_record = reference_record_df.iloc[0]
            
#             ref_is_null = pd.isnull(reference_record['fcc_asr_number']) and \
#                           pd.isnull(reference_record['faa_study_number'])
            
#             if ref_is_null:
#                 ref_op_name = reference_record['operator_name']; ref_op_site_id = reference_record['operator_site_id']
#                 ref_asset_status = reference_record['asset_status']; ref_type = reference_record['type']
#                 ref_name = reference_record['name']

#                 mask_assoc_is_null = (associated_records_df['fcc_asr_number'].isnull()) & \
#                                      (associated_records_df['faa_study_number'].isnull())
                
#                 mask_field_match = (associated_records_df['operator_name'] == ref_op_name) & \
#                                    (associated_records_df['operator_site_id'] == ref_op_site_id) & \
#                                    (associated_records_df['asset_status'] == ref_asset_status) & \
#                                    (associated_records_df['type'] == ref_type) & \
#                                    (associated_records_df['name'] == ref_name)
                
#                 final_matching_mask = mask_assoc_is_null & mask_field_match
#                 matching_associated_records = associated_records_df[final_matching_mask]
                
#                 if not matching_associated_records.empty:
#                     candidates_indices.add(ref_index); candidates_indices.update(matching_associated_records.index)

#     case5_auto_merge_candidates = working_table.loc[list(candidates_indices)].copy()
#     all_original_indices = set(working_table.index) 
#     final_post_merge_indices = all_original_indices.difference(candidates_indices)
#     case5_prox_audits_post_auto_merge_table = working_table.loc[list(final_post_merge_indices)].copy()
    
#     cols = working_table.columns
#     if case5_auto_merge_candidates.empty: case5_auto_merge_candidates = pd.DataFrame(columns=cols)
#     if case5_prox_audits_post_auto_merge_table.empty: case5_prox_audits_post_auto_merge_table = pd.DataFrame(columns=cols)

#     return case5_auto_merge_candidates, case5_prox_audits_post_auto_merge_table

# # CASE 5: Further Filter & Merging Process
# def apply_case_5_full_processing(
#     candidates_table: pd.DataFrame, initial_prox_audits_table: pd.DataFrame
# ) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
#     """Executes Case 5 merging logic. Applies C4 filters (AGL/Distance) but skips scraping."""
#     start_time = time.time(); print("Starting Case 5 processing...")
#     merging_timestamp = datetime.now(); cols = candidates_table.columns
#     merge_source_check = f"Auto-Merged {merging_timestamp.strftime('%m/%Y')}"
    
#     # Initialize lists and sets
#     case5_auto_merge_further_filter_list = []; case5_raw_post_auto_merge_list = []
#     case5_post_auto_merge_list = []; new_prox_audit_records_list = []
#     failed_prox_audit_records_list = []; case5_prox_audits_post_auto_merge_table = initial_prox_audits_table.copy()
#     processed_focus_ids = set(); processed_assoc_ids = set()
    
#     if not pd.api.types.is_datetime64_any_dtype(candidates_table['created_at']):
#         candidates_table['created_at'] = pd.to_datetime(candidates_table['created_at'], errors='coerce')
#     candidates_table['created_at_month'] = candidates_table['created_at'].dt.to_period('M')
#     grouped = candidates_table.groupby('focus_asset_id')
    
#     for focus_asset_id, group in tqdm(grouped, desc="Processing Case 5"):
#         reference_record_df = group[group['associated_asset_id'].isnull()]
#         associated_records_df = group[group['associated_asset_id'].notnull()].copy()
        
#         if len(reference_record_df) != 1: continue
            
#         reference_record = reference_record_df.iloc[0]
#         ref_source = reference_record['source']; ref_year_month = reference_record['created_at_month']
        
#         remaining_associated_records_df = associated_records_df.copy(); indices_to_remove = set()
        

#         if processed_focus_ids:
#             focus_id_match_mask = remaining_associated_records_df['associated_asset_id'].isin(processed_focus_ids)
#             indices_to_remove.update(remaining_associated_records_df[focus_id_match_mask].index)
#         if processed_assoc_ids:
#             assoc_id_match_mask = remaining_associated_records_df['associated_asset_id'].isin(processed_assoc_ids)
#             indices_to_remove.update(remaining_associated_records_df[assoc_id_match_mask].index)

#         source_match_mask = (remaining_associated_records_df['source'] == merge_source_check)
#         indices_to_remove.update(remaining_associated_records_df[source_match_mask].index)
        
#         ref_source_match = reference_record['source'] == merge_source_check
        
#         ref_record_cleaned = reference_record_df.copy()
#         if 'created_at_month' in ref_record_cleaned.columns: ref_record_cleaned.drop(columns=['created_at_month'], inplace=True)
        
#         if ref_source_match:
#             group_associated_cleaned = associated_records_df.copy()
#             if 'created_at_month' in group_associated_cleaned.columns: group_associated_cleaned.drop(columns=['created_at_month'], inplace=True)
#             failed_prox_audit_records_list.append(ref_record_cleaned); failed_prox_audit_records_list.append(group_associated_cleaned)
#             continue 

#         remaining_associated_records_df['created_at_month'] = remaining_associated_records_df['created_at'].dt.to_period('M')
#         date_source_match_mask = (remaining_associated_records_df['source'] == ref_source) & \
#                                  (remaining_associated_records_df['created_at_month'] == ref_year_month)
#         indices_to_remove.update(remaining_associated_records_df[date_source_match_mask].index)
        
#         removed_after_abcde_df = remaining_associated_records_df.loc[list(indices_to_remove)].copy()
        
#         remaining_associated_records_df = remaining_associated_records_df.loc[
#             ~remaining_associated_records_df.index.isin(indices_to_remove)
#         ].copy() 

#         # AGL Percentage Difference (>25% results in removal)
#         indices_to_remove_f = set()
#         if not remaining_associated_records_df.empty:
#             ref_agl = pd.to_numeric(reference_record['agl'], errors='coerce')
#             if pd.notnull(ref_agl):
#                 assoc_agl_series = pd.to_numeric(remaining_associated_records_df['agl'], errors='coerce')
#                 diff = ((ref_agl - assoc_agl_series).abs() / assoc_agl_series) * 100
#                 diff_filled = diff.fillna(0).replace([np.inf, -np.inf], 999) 
#                 agl_diff_mask = diff_filled > 25
#                 indices_to_remove_f.update(remaining_associated_records_df[agl_diff_mask].index)
        
#         removed_after_f_df = remaining_associated_records_df.loc[list(indices_to_remove_f)].copy()

#         remaining_associated_records_df = remaining_associated_records_df.loc[
#             ~remaining_associated_records_df.index.isin(indices_to_remove_f)
#         ].copy() 

#         # Least Distance Logic (Tie-breaker)
#         removed_after_g_df = pd.DataFrame(columns=remaining_associated_records_df.columns)
#         if len(remaining_associated_records_df) > 1:
#             remaining_associated_records_df['abs_agldiff'] = pd.to_numeric(
#                 remaining_associated_records_df['agldiff_to_reference'], errors='coerce'
#             ).abs()
            
#             if not remaining_associated_records_df['abs_agldiff'].isnull().all():
#                 closest_record_index = remaining_associated_records_df['abs_agldiff'].idxmin()
#                 closest_record_df = remaining_associated_records_df.loc[[closest_record_index]]
#                 removed_after_g_df = remaining_associated_records_df.loc[
#                     ~remaining_associated_records_df.index.isin([closest_record_index])
#                 ]
#                 remaining_associated_records_df = closest_record_df
#             else:
#                 removed_after_g_df = remaining_associated_records_df.copy()
#                 remaining_associated_records_df = pd.DataFrame(columns=remaining_associated_records_df.columns)
        
#         if 'abs_agldiff' in remaining_associated_records_df.columns:
#             remaining_associated_records_df = remaining_associated_records_df.drop(columns=['abs_agldiff'])
        
#         all_removed_associated_records_df = pd.concat([removed_after_abcde_df, removed_after_f_df, removed_after_g_df], ignore_index=False)
#         if 'created_at_month' in all_removed_associated_records_df.columns: all_removed_associated_records_df.drop(columns=['created_at_month'], inplace=True)
#         if 'created_at_month' in remaining_associated_records_df.columns: remaining_associated_records_df.drop(columns=['created_at_month'], inplace=True)
                
#         if not all_removed_associated_records_df.empty: failed_prox_audit_records_list.append(all_removed_associated_records_df)
        
#         # Final Check and Merge Logic
#         if len(remaining_associated_records_df) == 1:
#             current_group_to_merge = pd.concat([ref_record_cleaned, remaining_associated_records_df], ignore_index=False)
#             case5_auto_merge_further_filter_list.append(current_group_to_merge)
#             assoc_record = remaining_associated_records_df.iloc[0]
            
#             # MERGE: No web scraping needed. Proceed directly to merge.
#             merged_record_series = merge_records(reference_record, assoc_record, merging_timestamp, faa_study_number=None)
#             merged_record_df = pd.DataFrame([merged_record_series], columns=cols)
            
#             new_prox_audit_records_list.append(merged_record_df); case5_post_auto_merge_list.append(merged_record_df)
#             case5_raw_post_auto_merge_list.append(merged_record_df)
#             assoc_record_df = pd.DataFrame([assoc_record], columns=cols); case5_raw_post_auto_merge_list.append(assoc_record_df)
#             ref_record_df_cleaned_raw = ref_record_cleaned.copy(); case5_raw_post_auto_merge_list.append(ref_record_df_cleaned_raw)

#             if pd.notnull(merged_record_series['focus_asset_id']): processed_focus_ids.add(merged_record_series['focus_asset_id'])
#             if pd.notnull(assoc_record['associated_asset_id']): processed_assoc_ids.add(assoc_record['associated_asset_id'])
            
#         else:
#             failed_prox_audit_records_list.append(ref_record_cleaned)
            
#     # Concatenate all results from lists ONCE at the end
#     if new_prox_audit_records_list:
#         new_records_df = pd.concat(new_prox_audit_records_list, ignore_index=True)
#         case5_prox_audits_post_auto_merge_table = pd.concat([case5_prox_audits_post_auto_merge_table, new_records_df], ignore_index=True)
#     if failed_prox_audit_records_list:
#         failed_records_df = pd.concat(failed_prox_audit_records_list, ignore_index=True)
#         case5_prox_audits_post_auto_merge_table = pd.concat([case5_prox_audits_post_auto_merge_table, failed_records_df], ignore_index=True)
    
#     if case5_auto_merge_further_filter_list: case5_auto_merge_further_filter = pd.concat(case5_auto_merge_further_filter_list, ignore_index=False)
#     else: case5_auto_merge_further_filter = pd.DataFrame(columns=cols)
#     if case5_post_auto_merge_list: case5_post_auto_merge_table = pd.concat(case5_post_auto_merge_list, ignore_index=True)
#     else: case5_post_auto_merge_table = pd.DataFrame(columns=cols)
#     if case5_raw_post_auto_merge_list: case5_raw_post_auto_merge_table = pd.concat(case5_raw_post_auto_merge_list, ignore_index=True)
#     else: case5_raw_post_auto_merge_table = pd.DataFrame(columns=cols)

#     end_time = time.time()
#     duration_minutes = (end_time - start_time) / 60
#     print(f"--- Case 5 Processing completed in: {duration_minutes:.2f} minutes ---")
    
#     return (
#         case5_auto_merge_further_filter, 
#         case5_prox_audits_post_auto_merge_table.drop_duplicates(ignore_index=True),
#         case5_post_auto_merge_table.drop_duplicates(ignore_index=True), 
#         case5_raw_post_auto_merge_table.drop_duplicates(ignore_index=True)
#     )

# # CASE 5: Maintenance Conditions for cleanup and preparation of remaining proximity audits.
# def apply_case_5_maintenance_logic(
#     prox_audits_table: pd.DataFrame, post_auto_merge_table: pd.DataFrame, 
#     post_merge_table: pd.DataFrame, raw_post_merge_table: pd.DataFrame,
#     running_final_asset_table: pd.DataFrame
# ) -> Tuple[pd.DataFrame, pd.DataFrame]:
#     """Final Maintenance Step: Cleans up the remaining records, updates records, and prepares the last output."""
#     if post_auto_merge_table.empty:
#         empty_df = pd.DataFrame(columns=prox_audits_table.columns)
#         return running_final_asset_table, empty_df
        
#     working_post_merge = post_auto_merge_table.reset_index(drop=True)
#     final_asset_table_list = []
#     associated_records_mask = working_post_merge['associated_asset_id'].notnull()
    
#     if not post_merge_table.empty:
#         post_merge_lookup = post_merge_table.set_index('focus_asset_id')
#         all_cols = working_post_merge.columns.tolist()
#         cols_to_exclude = ["audit_reason", "distance_to_reference", "agldiff_to_reference", "associated_asset_id", "index"]
#         cols_to_update = [col for col in all_cols if col not in cols_to_exclude]
#         for idx, assoc_record in working_post_merge[associated_records_mask].iterrows():
#             assoc_asset_id = assoc_record['associated_asset_id']
#             if assoc_asset_id in post_merge_lookup.index:
#                 matching_merged_record = post_merge_lookup.loc[assoc_asset_id]
#                 if isinstance(matching_merged_record, pd.DataFrame): matching_merged_record = matching_merged_record.iloc[0]
#                 for col in cols_to_update:
#                     if col in matching_merged_record.index:
#                         if col in working_post_merge.columns:
#                             working_post_merge.loc[idx, col] = matching_merged_record[col]

#     if not raw_post_merge_table.empty:
#         raw_assoc_ids = set(raw_post_merge_table['associated_asset_id'].dropna())
#         removal_mask = (working_post_merge['associated_asset_id'].notnull()) & \
#                        (working_post_merge['associated_asset_id'].isin(raw_assoc_ids))
#         working_post_merge = working_post_merge[~removal_mask]

#     valid_groups = working_post_merge['focus_asset_id'].dropna()
#     if not valid_groups.empty:
#         group_sizes = working_post_merge.groupby('focus_asset_id').size()
#         single_record_groups = group_sizes[group_sizes == 1].index
#         final_asset_table_df = working_post_merge[
#             working_post_merge['focus_asset_id'].isin(single_record_groups)
#         ].copy()
#         final_asset_table_list.append(final_asset_table_df)
#         working_post_merge = working_post_merge[
#             ~working_post_merge['focus_asset_id'].isin(single_record_groups)
#         ]

#     sorted_post_merge_table = working_post_merge.sort_values(
#         by=['focus_asset_id', 'associated_asset_id'], 
#         ascending=[True, True],
#         na_position='first'
#     ).reset_index(drop=True)

#     case5_final_assets = pd.concat(final_asset_table_list, ignore_index=True)
#     if case5_final_assets.empty: case5_final_assets = pd.DataFrame(columns=prox_audits_table.columns) 
        
#     aggregated_final_asset_table = pd.concat([running_final_asset_table, case5_final_assets], ignore_index=True)

#     final_sorted_table = sorted_post_merge_table
    
#     return aggregated_final_asset_table, final_sorted_table


# # Executing the main functions for Case 1
# case1_auto_merge_candidates, initial_case1_prox_audits_post_auto_merge_table = split_case_1_audits(prox_audits_table)
# case1_auto_merge_further_filter, updated_case1_prox_audits_post_auto_merge_table, case1_post_auto_merge_table, case1_raw_post_auto_merge_table = apply_case_1_full_processing(case1_auto_merge_candidates, initial_case1_prox_audits_post_auto_merge_table)
# case1_aggregated_final_asset_table, final_case1_prox_audits_post_auto_merge_table = apply_case_1_maintenance_logic(prox_audits_table, updated_case1_prox_audits_post_auto_merge_table, case1_post_auto_merge_table, case1_raw_post_auto_merge_table)

# # Executing the main functions for Case 2
# case2_auto_merge_candidates, initial_case2_prox_audits_post_auto_merge_table = split_case_2_audits(final_case1_prox_audits_post_auto_merge_table)
# case2_auto_merge_further_filter, updated_case2_prox_audits_post_auto_merge_table, case2_post_auto_merge_table, case2_raw_post_auto_merge_table = apply_case_2_full_processing(case2_auto_merge_candidates, initial_case2_prox_audits_post_auto_merge_table)
# case2_aggregated_final_asset_table, final_case2_prox_audits_post_auto_merge_table = apply_case_2_maintenance_logic(prox_audits_table, updated_case2_prox_audits_post_auto_merge_table, case2_post_auto_merge_table, case2_raw_post_auto_merge_table, case1_aggregated_final_asset_table)

# # Executing the main functions for Case 3
# case3_auto_merge_candidates, initial_case3_prox_audits_post_auto_merge_table = split_case_3_audits(final_case2_prox_audits_post_auto_merge_table)
# case3_auto_merge_further_filter, updated_case3_prox_audits_post_auto_merge_table, case3_post_auto_merge_table, case3_raw_post_auto_merge_table = apply_case_3_full_processing(case3_auto_merge_candidates, initial_case3_prox_audits_post_auto_merge_table)
# case3_aggregated_final_asset_table, final_case3_prox_audits_post_auto_merge_table = apply_case_3_maintenance_logic(prox_audits_table, updated_case3_prox_audits_post_auto_merge_table, case3_post_auto_merge_table, case3_raw_post_auto_merge_table, case2_aggregated_final_asset_table)

# # Executing the main functions for Case 4
# case4_auto_merge_candidates, initial_case4_prox_audits_post_auto_merge_table = split_case_4_audits(final_case3_prox_audits_post_auto_merge_table)
# case4_auto_merge_further_filter, updated_case4_prox_audits_post_auto_merge_table, case4_post_auto_merge_table, case4_raw_post_auto_merge_table = apply_case_4_full_processing(case4_auto_merge_candidates, initial_case4_prox_audits_post_auto_merge_table)
# case4_aggregated_final_asset_table, final_case4_prox_audits_post_auto_merge_table = apply_case_4_maintenance_logic(prox_audits_table, updated_case4_prox_audits_post_auto_merge_table, case4_post_auto_merge_table, case4_raw_post_auto_merge_table, case3_aggregated_final_asset_table)

# # Executing the main functions for Case 5
# case5_auto_merge_candidates, initial_case5_prox_audits_post_auto_merge_table = split_case_5_audits(final_case4_prox_audits_post_auto_merge_table)
# case5_auto_merge_further_filter, updated_case5_prox_audits_post_auto_merge_table, case5_post_auto_merge_table, case5_raw_post_auto_merge_table = apply_case_5_full_processing(case5_auto_merge_candidates, initial_case5_prox_audits_post_auto_merge_table)
# case5_aggregated_final_asset_table, final_case5_prox_audits_post_auto_merge_table = apply_case_5_maintenance_logic(prox_audits_table, updated_case5_prox_audits_post_auto_merge_table, case5_post_auto_merge_table, case5_raw_post_auto_merge_table, case4_aggregated_final_asset_table)